# Toy Direct Synchrony CST Model

This notebook is a self-contained toy model for exploring how directly controlled spike synchrony affects the cumulative spike train (CST) and synchrony readouts. It is intentionally separate from the full Brian2 motor-neuron simulator.

The model combines a shared/common event process with private independent events. A unit's spike train is the sum of jittered copies of common events plus private spikes. This makes synchrony directly tunable through `common_fraction`, `common_jitter_sd_ms`, and `common_copy_probability`.

Interpretation notes:
- Shared rate modulation alone does not necessarily create excess synchrony after local-rate correction.
- Common events with short jitter create true short-lag synchrony.
- Rate-controlled synchrony asks: are there more coincidences than expected from local rates?
- Absolute synchrony asks: how many near-coincident spike events actually occur?
- This toy model is a controlled readout tool, not a replacement for the full MN simulator.


In [ ]:
from __future__ import annotations

import hashlib
import itertools
import json
import math
import shutil
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.signal import butter, sosfiltfilt
from scipy.stats import gaussian_kde, skew

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


In [ ]:
# EDITABLE CONFIGURATION
# duration_s is the analyzed / displayed duration. ignored_initial_duration_s adds
# a non-analyzed burn-in prefix before it. This helps the gamma-renewal process
# and causal CST smoother settle before summary statistics and figures are made.
# The displayed x-axis keeps absolute simulation time, so the analyzed window starts
# at ignored_initial_duration_s rather than being silently reset to zero.
duration_s = 2.0
ignored_initial_duration_s = 1.0  # set to 0.0 to disable the burn-in prefix
fs_hz = 1000 # 1000
n_units = 30 # 30

# baseline_rate_hz is the target baseline marginal firing rate before adding the
# filtered Gaussian rate-modulating components below.
baseline_rate_hz = 14.5 # match to experimental data
# unit_mean_rate_sd_hz adds heterogeneity across simulated units. The notebook
# computes multiplicative weights on the shared rate trace before the common/private
# decomposition, so each unit's mean analyzed-window rate has this target SD in Hz
# while preserving the population mean rate. 0 keeps all units homogeneous.
unit_mean_rate_sd_hz = 0.0 # 2.0 = reproduces variability in experimental data
# rate_floor_hz is the minimum allowed rate after modulation. It prevents negative
# rates when the summed rate components transiently pull lambda(t) below zero.
rate_floor_hz = 0.05
random_seed = 1000 # 12345

# Each input component modulates lambda(t), not spike-train regularity directly.
# For each component: white noise -> filter in band_hz -> standardize -> scale to
# target_sd_hz -> add to the firing-rate trace.
# A band with low <= 0 is treated as low-pass.
input_components = [
    {'name': 'lf', 'band_hz': (0.0, 5.0), 'target_sd_hz': 0}, # 0.5},
    {'name': 'alpha', 'band_hz': (8.0, 13.0), 'target_sd_hz': 0}, # 0.5},
    {'name': 'beta', 'band_hz': (13.0, 30.0), 'target_sd_hz': 0}, # 1.0},
]

# Optional rectangular pulse input. The pulse is additive in Hz on the rate/input
# trace. Its start is relative to the analyzed window start, so with a 1 s ignored
# prefix and start_rel=1 s, the absolute pulse onset is t=2 s. The duration is
# clipped by the simulated time, so a long duration behaves like a step input.
enable_pulse_input = True
pulse_input_start_rel_analysis_s = 1.0
pulse_input_duration_s = 10.0
pulse_input_amplitude_hz = 0.0
pulse_metric_pre_window_rel_s = (-1.0, -0.2) # relative to pulse onset; e.g. (-0.5, 0) looks at the 0.5 s before the pulse starts
pulse_metric_post_window_rel_s = (0.2, 1.0)

# Direct synchrony controls.
# common_fraction is the fraction of each neuron's marginal rate allocated to
# copied common events. 0 = all spikes are private; 1 = all expected spikes come
# through the common-event process before copy probability and jitter are applied.
common_fraction = 1.0 # 0.0 # 0.25
# common_copy_probability is the probability that each latent common event is
# copied to a given unit. Lower values create smaller synchronous subgroups. The
# generator compensates the latent common-event rate so this does not reduce the
# expected marginal firing rate of each unit. If common_fraction > 0, this must be > 0.
common_copy_probability = 0.05 # 1.0
# common_jitter_sd_ms jitters copied common events independently for each unit.
# 0 ms means perfectly aligned common spikes; larger values smear short-lag synchrony.
common_jitter_sd_ms = 2.0 # 0.0

# Gamma-renewal shape parameters in integrated-rate time.
# k = 1 gives a memoryless inhomogeneous Poisson process.
# k > 1 gives more regular-than-Poisson events.
# k < 1 gives burstier / more irregular events.
# Approximate stationary ISI CV is 1/sqrt(k), but observed CV also depends on
# rate modulation and on the mixture of common and private events.
private_gamma_shape_k = 70.0 # 70.0  # 70.0 ~ match to experimental data
common_gamma_shape_k = 70.0 # 70.0  # 70.0 ~ match to experimental data
# gamma_shape_k_sd adds positive-valued heterogeneity to both shape parameters.
# Private k values are sampled per unit around private_gamma_shape_k; the latent
# common-event process receives one sampled k around common_gamma_shape_k. The
# requested SD is the target SD of the sampling distribution, so the realized
# finite-unit private SD in one simulation may differ slightly. A value of 0 keeps
# both shape parameters fixed exactly at their nominal values.
gamma_shape_k_sd = 0.0 # 20.0 to ~ match experimental data

# To reproduce the previous memoryless Poisson-like toy model:
#     private_gamma_shape_k = 1.0
#     common_gamma_shape_k = 1.0
#     common_fraction still controls copied common events
#     common_fraction = 0 gives fully independent inhomogeneous Poisson neurons

# CST smoothing: causal, finite-support, one-sided half-Hann.
cst_smoothing_kernel = 'causal_half_hann'
cst_smoothing_width_ms = 200.0 # 80.0

# Synchrony analysis.
sync_window_ms = 100.0
sync_step_ms = 10.0
coincidence_lag_ms = 6.0

# Display / optional exploration.
show_input_components = True
summary_xlim_s = None
RUN_VALIDATION_CHECKS = False # True
RUN_GRID_EXPLORATION = False # True # True
# Grid exploration now uses the editable configuration as the fixed baseline.
# Only parameters listed here are varied; all other values stay exactly as above.
GRID_PARAMETER_VALUES = {
    'common_fraction': [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
    # 'common_jitter_sd_ms': [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0],
    'common_copy_probability': [0.05, 0.1, 0.2, 0.4, 0.7, 1.0],
    # Examples:
    # 'common_copy_probability': [0.25, 0.5, 1.0],
    # 'private_gamma_shape_k': [1.0, 10.0, 70.0],
}
GRID_X_PARAMETER = 'common_fraction'
GRID_Y_PARAMETER = 'common_copy_probability' # 'common_jitter_sd_ms'
GRID_BASE_RANDOM_SEED = 1000
# Repetitions / realizations for each grid cell. If GRID_REALIZATION_SEEDS is None,
# seeds are generated as GRID_BASE_RANDOM_SEED + realization_index. If you provide
# a list, those exact seeds are reused for every parameter combination.
GRID_N_REALIZATIONS = 5
GRID_REALIZATION_SEEDS = [1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009] # None  # e.g. [1000, 1001, 1002, 1003]
GRID_METRIC_SPECS = [
    ('mean_sync_rate_controlled', 'rate-controlled synchrony', 'Purples'),
    ('mean_sync_absolute_coincidence_rate_per_s_pair', 'absolute synchrony (1/s/pair)', 'Blues'),
    # ('sd_filtered_cst_spikes_per_s_mu', 'filtered CST SD (sp/s/MU)', 'Oranges'),
    # Other available examples include:
    # ('sd_sync_rate_controlled', 'rate-controlled synchrony SD', 'Purples'),
    # ('cov_sync_rate_controlled', 'rate-controlled synchrony CoV', 'Purples'),
    # ('cov_filtered_cst_spikes_per_s_mu', 'filtered CST CoV', 'Oranges'),
]
# Bottom-row grid curves. Curve colors use each metric's colormap, clipped to the
# upper part of the scale so the lowest hue level remains visible.
GRID_CURVE_X_PARAMETER = 'common_copy_probability' # 'common_jitter_sd_ms' # 'common_fraction'
GRID_CURVE_HUE_PARAMETER = 'common_fraction' # 'common_jitter_sd_ms'
GRID_CURVE_COLORMAP_MIN_FRACTION = 0.5
# Dense grids become unreadable with a number in every cell, so annotations are
# automatically hidden above this number of heatmap cells.
GRID_HEATMAP_ANNOTATION_MAX_CELLS = 80
GRID_SHOW_EXPERIMENTAL_SYNC_REFERENCES = True
GRID_EXPERIMENTAL_BASELINE_SYNC_COLOR = "#0080ff"
GRID_EXPERIMENTAL_RESPONSE_SYNC_COLOR = '#cc2222'
GRID_EXPERIMENTAL_REFERENCE_ALPHA = 0.14
SAVE_FIGURES = True
FIGURE_OUTPUT_DIR = Path('$$$$$$$$$$$$_toy_direct_synchrony_outputs_V3')
# Grid output cache. Loading skips the expensive grid simulation cell but still
# lets the figure cell regenerate plots/tables from the saved grid_df.
GRID_OUTPUT_DIR = FIGURE_OUTPUT_DIR / 'grid_exploration'
GRID_OUTPUT_RUN_NAME = 'toy_sync_grid_current'
GRID_LOAD_EXISTING_OUTPUT = False
GRID_SAVE_OUTPUTS = True
GRID_OVERWRITE_OUTPUT = False
GRID_STRICT_CONFIG_MATCH_ON_LOAD = False # True
GRID_SAVE_FIGURE_FORMATS = ('png') # , 'pdf')
GRID_SHOW_PROGRESS = True


# Trace-level batch exploration. This is separate from the scalar grid above.
# It stores compact traces per parameter set so mean-over-realizations figures can
# be regenerated without rerunning the expensive synchrony analysis.
RUN_TRACE_BATCH = True # False
TRACE_BATCH_PARAMETER_VALUES = {
    # Examples. Only parameters listed here are varied; all others stay as defined above.
    'common_fraction': [1.0], # [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0], # [0.0, 0.25, 0.5, 0.75],
    'common_copy_probability': [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'pulse_input_amplitude_hz': [0.0, 10.0], # , 20.0], # [0.0, 1.0, 5.0, 20.0],
    # 'common_jitter_sd_ms': [0.0, 2.0, 4.0, 6.0],
}
TRACE_BATCH_N_REALIZATIONS = 150
TRACE_BATCH_REALIZATION_SEEDS = None  # e.g. [2000, 2001, 2002]; None uses TRACE_BATCH_BASE_RANDOM_SEED + index
TRACE_BATCH_BASE_RANDOM_SEED = 1000
TRACE_BATCH_OUTPUT_DIR = FIGURE_OUTPUT_DIR / 'trace_batch'
TRACE_BATCH_RUN_NAME = 'toy_sync_trace_batch_current'
TRACE_BATCH_LOAD_EXISTING_OUTPUT = True # False # True
TRACE_BATCH_SAVE_OUTPUTS = True
TRACE_BATCH_OVERWRITE_OUTPUT = False
TRACE_BATCH_STRICT_CONFIG_MATCH_ON_LOAD = False # True
TRACE_BATCH_SHOW_PROGRESS = True
# Figure generation can be skipped when reloading cached trace batches.
# Set PLOT_TRACE_BATCH_FIGURES=False to bypass all figures in the main
# trace-batch figure cell while keeping cached tables/traces available.
# PLOT_TRACE_BATCH_MEAN_TRACE_FIGURES controls the expensive per-parameter-set
# mean trace summaries generated by plot_trace_batch_mean_summary().
PLOT_TRACE_BATCH_FIGURES = True
PLOT_TRACE_BATCH_MEAN_TRACE_FIGURES = False
PLOT_TRACE_BATCH_RELATION_FIGURES = True
TRACE_BATCH_SAVE_FIGURE_FORMATS = ('png', 'pdf')
TRACE_BATCH_INDIVIDUAL_TRACE_ALPHA = 0.1
TRACE_BATCH_MEAN_TRACE_LW = 2.3
TRACE_BATCH_SHARE_MEAN_FIGURE_AXES = True
TRACE_BATCH_MEAN_FIGURE_PARAMETER_SET_IDS = None  # None = all parameter sets; otherwise list of ids
TRACE_BATCH_MAX_MEAN_FIGURES = None  # optional cap when many parameter sets are present
# Extra zoom panel for integrated input/CST. If center is None, it is set to
# ignored_initial_duration_s + 1.0, so with a 1 s ignored prefix the default center is t=2 s.
TRACE_BATCH_INTEGRAL_ZOOM_CENTER_S = None
TRACE_BATCH_INTEGRAL_ZOOM_HALF_WIDTH_S = 0.3
# Pulse-step CST summary figure. Rows are pulse amplitudes and columns are common_fraction by default.
BUILD_TRACE_BATCH_MATCHED_ZERO_BASELINE = True
TRACE_BATCH_MATCHED_ZERO_AMPLITUDE_HZ = 0.0
TRACE_BATCH_MATCHED_ZERO_SAVE_OUTPUTS = True
TRACE_BATCH_MATCHED_ZERO_REQUIRE_ALL_MATCHES = True
BUILD_TRACE_BATCH_RESPONSE_METRICS = True
TRACE_BATCH_RESPONSE_SAVE_OUTPUTS = True
PLOT_TRACE_BATCH_RESPONSE_FIGURES = True
RESPONSE_EPSILON_EXTRA_SPIKES_PER_MN = 0.05 # 3.0
RESPONSE_TARGET_FRACTION_OF_INPUT_RATE = 0.10 # 0.50  # CST-effect threshold = this fraction of the instructed pulse-rate change.
RESPONSE_TARGET_RATE_SOURCE = 'matched_effect_raw'  # 'matched_effect_smooth' or 'matched_effect_raw'
RESPONSE_BASELINE_CORRECTION_WINDOW_REL_S = pulse_metric_pre_window_rel_s
RESPONSE_MAX_SLOPE_SOURCE = 'actual_cst'  # 'matched_effect' or 'actual_cst'
RESPONSE_MAX_SLOPE_WINDOW_S = 0.010
RESPONSE_MAX_SLOPE_SEARCH_HORIZON_S = 0.200
RESPONSE_MAX_SLOPE_STEP_S = 0.005
RESPONSE_CAPTURE_HORIZONS_S = (0.010, 0.050, 0.100, 0.200, 0.500)
RESPONSE_DIAGNOSTIC_ROW_INDEX = None  # None = first nonzero response row
RESPONSE_DIAGNOSTIC_XLIM_REL_S = (-0.05, 0.35) # None  # e.g. (-0.05, 0.35) for a tight post-pulse zoom.
RESPONSE_FIGURE_COLOR_BY = 'common_copy_probability' # 'common_fraction'
RESPONSE_SCATTER_GROUP_COL = 'common_copy_probability' #'common_fraction'
RESPONSE_SCATTER_CAPTURE_HORIZON_S = 0.100  # Which cumulative_capture_* horizon to display in response scatter summaries.
RESPONSE_SCATTER_SHOW_GROUP_MEANS = True
RESPONSE_SCATTER_SHOW_GLOBAL_REGRESSION = True
RESPONSE_SCATTER_SHOW_AGE_GROUP_REGRESSIONS = True
RESPONSE_SCATTER_MEAN_MARKER_SIZE = 95
RESPONSE_SCATTER_MEAN_ERRORBAR_ALPHA = 0.75

# Paper-ready response scatter: baseline rate-normalized synchrony vs max cumulative slope.
# This is independent of PLOT_TRACE_BATCH_RESPONSE_FIGURES so it can be regenerated alone
# from cached trace_batch_response_metrics_df / trace_batch_response_metrics.pkl.
PLOT_PAPER_SYNC_SLOPE_SCATTER = True
PAPER_SYNC_SLOPE_SCATTER_PULSE_VALUES_HZ = None  # None = all nonzero pulse amplitudes; otherwise e.g. [20.0]
PAPER_SYNC_SLOPE_SCATTER_GROUP_COL = 'common_copy_probability'
PAPER_SYNC_SLOPE_SCATTER_X_COL = 'baseline_sync_rate_controlled_mean'
PAPER_SYNC_SLOPE_SCATTER_Y_COL = 'max_early_cumulative_slope_spikes_per_s_per_mn'
PAPER_SYNC_SLOPE_SCATTER_OUTPUT_BASENAME = 'paper_baseline_sync_vs_max_cumulative_slope'
PAPER_SYNC_SLOPE_SCATTER_STYLE = {
    'figsize_per_pulse': (3.2, 2.8),
    'dpi': 300,
    # Synchrony-themed colormap, replacing viridis. Edit these colors freely.
    'group_colors': ["#28218b", "#a647e6", "#ff6e9a"],
    'point_size': 10,
    'point_alpha': 0.55,
    'point_edgecolor': 'none',
    'group_marker_size': 70,
    'group_marker_edgecolor': 'black',
    'group_marker_edgewidth': 1.0,
    'group_errorbar_color': 'black',
    'group_errorbar_lw': 1.0,
    'group_errorbar_capsize': 1.5,
    'group_errorbar_alpha': 0.8,
    'regression_color': 'black',
    'regression_lw': 1.0,
    'regression_ls': '-',
    'regression_alpha': 0.85,
    'show_regression': True,
    'show_group_markers': True,
    'show_colorbar': True,
    'colorbar_label': 'common copy probability',
    'title': None,
    'x_label': 'Baseline rate-normalized synchrony',
    'y_label': 'Max cumulative slope\n(spikes/s/MN)',
    'title_fontsize': 4,
    'axis_label_fontsize': 4,
    'tick_labelsize': 4,
    'annotation_fontsize': 4,
    'colorbar_label_fontsize': 4,
    'colorbar_tick_labelsize': 4,
    'grid_alpha': 0.18,
    'annotation_xy': (0.04, 0.96),
    'annotation_ha': 'left',
    'annotation_va': 'top',
    'annotation_bbox': {'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.72, 'pad': 2.0},
    'shared_xlim': True,
    'shared_ylim': True,
    'xlim': None,
    'ylim': None,
    'pad_fraction': 0.08,
    'save_formats': ('pdf', 'png'),
    'bbox_inches': 'tight',
    'pad_inches': 0.02,
}
PLOT_TRACE_BATCH_PULSE_STEP_CST_SUMMARY = True
TRACE_BATCH_PULSE_STEP_COLUMN_PARAMETER = 'common_copy_probability' #'common_fraction'
TRACE_BATCH_PULSE_STEP_ROW_PARAMETER = 'pulse_input_amplitude_hz'
TRACE_BATCH_PULSE_STEP_INCLUDE_ZERO_PULSE = True
TRACE_BATCH_PULSE_STEP_ZOOM_WINDOW_REL_S = (-0.8, 1.3)
TRACE_BATCH_PULSE_STEP_SHARE_Y = True
TRACE_BATCH_PULSE_STEP_CST_COLOR = '#ff3300'
TRACE_BATCH_PULSE_STEP_INPUT_COLOR = 'black'
TRACE_BATCH_PULSE_STEP_PULSE_COLOR = '#2ca02c'
TRACE_BATCH_PULSE_STEP_SAVE_FIGURES = True

TRACE_BATCH_RELATION_SPECS = [
    {
        'x_metric': 'mean_sync_rate_controlled',
        'y_metric': 'mean_cumulative_cst_minus_input_residual_spikes_per_mu',
        'data_mode': 'realization',  # 'realization' or 'parameter_set_summary'
        'plot_mode': 'both',  # 'scatter', 'curve', or 'both'
        'color_by': None,
        'x_binning': 'auto',  # 'exact', 'quantile', or 'auto'
        'n_quantile_bins': 8,
        'title': 'CST-Input Residuals (y) vs Rate-normalized synchrony (x)',
    },
]

# Optional experimental moment overlays. This table contains participant-condition
# summary features, not raw ISIs, so experimental mean-ISI / ISI-SD overlays below
# are approximate conversions from firing-rate and ISI-CV moments.
EXPERIMENTAL_FEATURE_TABLE_PATH = Path('EXP_DATA_ANALYZED_HDF5/_batch_summaries/experimental_batch_feature_summary.pkl')
LOAD_EXPERIMENTAL_MOMENTS = True
SHOW_EXPERIMENTAL_ISI_OVERLAYS = True
EXPERIMENTAL_CONDITION_COLORS = {
    'HAND_GO': '#ffbb00',
    'HAND_NOGO': '#ff6600',
    'TA_NOGO': '#ff0080',
}
# Choose how experimental moments are grouped in ISI diagnostic overlays:
# 'grand_average' = all participant-condition rows pooled as one target;
# 'condition' = one group/color per condition;
# 'participant' = one group/color per participant.
EXPERIMENTAL_MOMENT_PLOT_LEVEL = 'grand_average'  # 'grand_average', 'condition', 'participant'
EXPERIMENTAL_MOMENT_LEVEL_COLORS = {
    'grand_average': {'grand_average': '#111111'},
    'condition': EXPERIMENTAL_CONDITION_COLORS,
    # Add participant colors manually here if desired; missing participant colors are auto-filled from tab20.
    'participant': {},
}


## Helper Functions

Rates are clipped at `rate_floor_hz`. This keeps the model linear and transparent while preventing invalid negative firing rates. The CST is expressed as `spikes/s/MU`: binned population counts divided by the number of units and multiplied by the sampling rate. Burn-in samples are generated before the analyzed window, then excluded from scalar summaries and plots.


In [ ]:
def _standardize_to_target_sd(x, target_sd):
    x = np.asarray(x, dtype=float)
    x = x - np.nanmean(x)
    sd = float(np.nanstd(x, ddof=0))
    if not np.isfinite(sd) or sd <= 0:
        return np.zeros_like(x, dtype=float)
    return x / sd * float(target_sd)


def _filter_component(noise, fs_hz, band_hz, order=3):
    low_hz, high_hz = map(float, band_hz)
    nyquist = float(fs_hz) / 2.0
    if high_hz <= 0 or high_hz >= nyquist:
        raise ValueError(f'Invalid band_hz={band_hz}; high edge must be between 0 and Nyquist')
    if low_hz <= 0:
        sos = butter(int(order), high_hz, btype='lowpass', fs=float(fs_hz), output='sos')
    else:
        if low_hz >= high_hz:
            raise ValueError(f'Invalid band_hz={band_hz}; low edge must be below high edge')
        sos = butter(int(order), [low_hz, high_hz], btype='bandpass', fs=float(fs_hz), output='sos')
    return sosfiltfilt(sos, np.asarray(noise, dtype=float))


def _rectangular_pulse_trace(t_s, *, enable, start_abs_s, duration_s, amplitude_hz):
    t_s = np.asarray(t_s, dtype=float)
    if duration_s < 0:
        raise ValueError('pulse_input_duration_s must be >= 0')
    if not np.isfinite(start_abs_s):
        raise ValueError('pulse_input_start_abs_s must be finite')
    if not np.isfinite(amplitude_hz):
        raise ValueError('pulse_input_amplitude_hz must be finite')
    trace = np.zeros_like(t_s, dtype=float)
    nominal_end_abs_s = float(start_abs_s) + float(duration_s)
    if bool(enable) and float(duration_s) > 0:
        mask = (t_s >= float(start_abs_s)) & (t_s < nominal_end_abs_s)
        trace[mask] = float(amplitude_hz)
    if t_s.size > 1:
        dt = float(np.median(np.diff(t_s)))
    else:
        dt = 0.0
    simulated_end_s = float(t_s[-1] + dt) if t_s.size else 0.0
    effective_start_s = min(max(float(start_abs_s), 0.0), simulated_end_s)
    effective_end_s = min(max(nominal_end_abs_s, 0.0), simulated_end_s)
    effective_duration_s = max(0.0, effective_end_s - effective_start_s) if bool(enable) else 0.0
    return trace, {
        'pulse_input_enabled': bool(enable),
        'pulse_input_start_abs_s': float(start_abs_s),
        'pulse_input_end_abs_s': float(effective_end_s),
        'pulse_input_nominal_end_abs_s': float(nominal_end_abs_s),
        'pulse_input_duration_effective_s': float(effective_duration_s),
        'pulse_input_amplitude_hz': float(amplitude_hz),
        'pulse_input_area_expected_spikes_per_neuron': float(np.sum(trace) * dt),
    }


def generate_rate_process(
    duration_s,
    fs_hz,
    baseline_rate_hz,
    components,
    rate_floor_hz,
    rng,
    *,
    enable_pulse_input=False,
    pulse_input_start_abs_s=0.0,
    pulse_input_duration_s=0.0,
    pulse_input_amplitude_hz=0.0,
):
    n_samples = int(round(float(duration_s) * float(fs_hz)))
    t_s = np.arange(n_samples, dtype=float) / float(fs_hz)
    component_traces = {}
    total_modulation = np.zeros(n_samples, dtype=float)
    for spec in components:
        name = str(spec['name'])
        target_sd = float(spec.get('target_sd_hz', 0.0))
        noise = rng.normal(0.0, 1.0, n_samples)
        filtered = _filter_component(noise, fs_hz, spec['band_hz'])
        trace = _standardize_to_target_sd(filtered, target_sd)
        component_traces[name] = trace
        total_modulation += trace
    pulse_trace_hz, pulse_metadata = _rectangular_pulse_trace(
        t_s,
        enable=enable_pulse_input,
        start_abs_s=float(pulse_input_start_abs_s),
        duration_s=float(pulse_input_duration_s),
        amplitude_hz=float(pulse_input_amplitude_hz),
    )
    component_traces['pulse_input'] = pulse_trace_hz
    raw_rate_hz = float(baseline_rate_hz) + total_modulation + pulse_trace_hz
    rate_hz = np.maximum(raw_rate_hz, float(rate_floor_hz))
    return {
        'time_s': t_s,
        'component_traces_hz': component_traces,
        'pulse_trace_hz': pulse_trace_hz,
        'pulse_metadata': pulse_metadata,
        'raw_rate_hz': raw_rate_hz,
        'rate_hz': rate_hz,
    }


def generate_inhomogeneous_gamma_renewal_spikes(rate_hz, t_s, shape_k, rng):
    """Generate spikes from an inhomogeneous gamma-renewal process.

    Renewal intervals are drawn in integrated-rate time with mean 1 expected spike.
    shape_k=1 is the inhomogeneous Poisson / exponential-renewal special case.
    """
    rate_hz = np.asarray(rate_hz, dtype=float).reshape(-1)
    t_s = np.asarray(t_s, dtype=float).reshape(-1)
    shape_k = float(shape_k)
    if shape_k <= 0 or not np.isfinite(shape_k):
        raise ValueError('gamma renewal shape_k must be finite and strictly positive')
    if rate_hz.shape != t_s.shape:
        raise ValueError('rate_hz and t_s must have the same shape')
    if rate_hz.size == 0:
        return np.array([], dtype=float)
    if np.any(~np.isfinite(rate_hz)) or np.any(rate_hz < 0):
        raise ValueError('rate_hz must be finite and nonnegative')
    if np.any(~np.isfinite(t_s)) or np.any(np.diff(t_s) <= 0):
        raise ValueError('t_s must be finite and strictly increasing')

    dt = np.empty_like(t_s, dtype=float)
    if t_s.size == 1:
        dt[0] = 0.0
    else:
        dt[0] = t_s[1] - t_s[0]
        dt[1:] = np.diff(t_s)
    cumulative_intensity = np.cumsum(rate_hz * dt)
    total_intensity = float(cumulative_intensity[-1])
    if total_intensity <= 0:
        return np.array([], dtype=float)

    thresholds = []
    threshold = float(rng.gamma(shape=shape_k, scale=1.0 / shape_k))
    while threshold <= total_intensity:
        thresholds.append(threshold)
        threshold += float(rng.gamma(shape=shape_k, scale=1.0 / shape_k))
    if not thresholds:
        return np.array([], dtype=float)

    thresholds = np.asarray(thresholds, dtype=float)
    indices = np.searchsorted(cumulative_intensity, thresholds, side='left')
    spike_times = np.empty(thresholds.size, dtype=float)
    for out_i, (idx, thr) in enumerate(zip(indices, thresholds)):
        idx = int(idx)
        prev_intensity = cumulative_intensity[idx - 1] if idx > 0 else 0.0
        prev_time = t_s[idx - 1] if idx > 0 else max(0.0, t_s[0] - dt[0])
        bin_intensity = cumulative_intensity[idx] - prev_intensity
        frac = np.clip((thr - prev_intensity) / bin_intensity, 0.0, 1.0) if bin_intensity > 0 else 0.0
        spike_times[out_i] = prev_time + frac * dt[idx]
    duration_end = t_s[-1] + (dt[-1] if dt.size else 0.0)
    return spike_times[(spike_times >= 0.0) & (spike_times < duration_end)]



def make_unit_rate_weights(rate_hz, analysis_slice, n_units, target_sd_hz, *, min_mean_rate_hz=1e-6):
    rate_hz = np.asarray(rate_hz, dtype=float)
    n_units = int(n_units)
    target_sd_hz = float(target_sd_hz)
    if n_units <= 0:
        raise ValueError('n_units must be positive')
    if target_sd_hz < 0 or not np.isfinite(target_sd_hz):
        raise ValueError('unit_mean_rate_sd_hz must be finite and >= 0')
    analyzed_rate = rate_hz[analysis_slice]
    mean_rate_hz = float(np.mean(analyzed_rate))
    if mean_rate_hz <= 0 or not np.isfinite(mean_rate_hz):
        raise ValueError('Analyzed mean rate must be finite and > 0 before unit weighting')
    if target_sd_hz == 0 or n_units == 1:
        target_means = np.full(n_units, mean_rate_hz, dtype=float)
        return np.ones(n_units, dtype=float), target_means

    # Deterministic symmetric scores give exactly mean 0 and SD 1 for finite n,
    # avoiding run-to-run changes in the intended heterogeneity pattern.
    scores = np.linspace(-1.0, 1.0, n_units, dtype=float)
    scores = scores - float(np.mean(scores))
    score_sd = float(np.std(scores, ddof=0))
    if score_sd <= 0:
        scores = np.zeros(n_units, dtype=float)
    else:
        scores = scores / score_sd
    target_means = mean_rate_hz + target_sd_hz * scores
    if np.any(target_means <= float(min_mean_rate_hz)):
        min_allowed_sd = (mean_rate_hz - float(min_mean_rate_hz)) / max(abs(float(np.min(scores))), 1e-12)
        raise ValueError(
            'unit_mean_rate_sd_hz is too large for the analyzed mean rate: '
            f'{target_sd_hz:.3g} Hz would create non-positive unit mean rates. '
            f'Use <= {min_allowed_sd:.3g} Hz or increase baseline_rate_hz.'
        )
    weights = target_means / mean_rate_hz
    return weights.astype(float), target_means.astype(float)



def _sample_positive_lognormal_from_mean_sd(mean_value, sd_value, size, rng):
    mean_value = float(mean_value)
    sd_value = float(sd_value)
    if mean_value <= 0 or not np.isfinite(mean_value):
        raise ValueError('gamma shape mean values must be finite and > 0')
    if sd_value < 0 or not np.isfinite(sd_value):
        raise ValueError('gamma_shape_k_sd must be finite and >= 0')
    if sd_value == 0:
        return np.full(size, mean_value, dtype=float)
    sigma2 = math.log1p((sd_value / mean_value) ** 2)
    sigma = math.sqrt(sigma2)
    mu = math.log(mean_value) - 0.5 * sigma2
    return rng.lognormal(mean=mu, sigma=sigma, size=size).astype(float)


def make_gamma_shape_values(private_mean_k, common_mean_k, gamma_shape_k_sd, n_units, rng):
    private_values = _sample_positive_lognormal_from_mean_sd(private_mean_k, gamma_shape_k_sd, int(n_units), rng)
    common_value = float(_sample_positive_lognormal_from_mean_sd(common_mean_k, gamma_shape_k_sd, 1, rng)[0])
    return private_values, common_value


def generate_spike_trains(rate_hz, t_s, n_units, common_fraction, common_jitter_sd_ms, common_copy_probability, private_gamma_shape_k, common_gamma_shape_k, rng, unit_rate_weights=None):
    common_fraction = float(common_fraction)
    if not 0.0 <= common_fraction <= 1.0:
        raise ValueError('common_fraction must be in [0, 1]')
    copy_probability = float(common_copy_probability)
    if not 0.0 <= copy_probability <= 1.0:
        raise ValueError('common_copy_probability must be in [0, 1]')
    if common_fraction > 0.0 and copy_probability <= 0.0:
        raise ValueError('common_copy_probability must be > 0 when common_fraction > 0')
    common_gamma_shape_k = float(common_gamma_shape_k)
    private_gamma_shape_values = np.asarray(private_gamma_shape_k, dtype=float).reshape(-1)
    if private_gamma_shape_values.size == 1:
        private_gamma_shape_values = np.full(int(n_units), float(private_gamma_shape_values[0]), dtype=float)
    if private_gamma_shape_values.size != int(n_units):
        raise ValueError('private_gamma_shape_k must be scalar or one value per unit')
    if np.any(private_gamma_shape_values <= 0) or np.any(~np.isfinite(private_gamma_shape_values)) or common_gamma_shape_k <= 0 or not np.isfinite(common_gamma_shape_k):
        raise ValueError('private_gamma_shape_k and common_gamma_shape_k must be finite and > 0')

    rate_hz = np.asarray(rate_hz, dtype=float)
    t_s = np.asarray(t_s, dtype=float)
    duration_s = float(t_s[-1] + np.median(np.diff(t_s))) if t_s.size > 1 else 0.0
    jitter_s = float(common_jitter_sd_ms) / 1000.0
    if unit_rate_weights is None:
        unit_rate_weights = np.ones(int(n_units), dtype=float)
    unit_rate_weights = np.asarray(unit_rate_weights, dtype=float).reshape(-1)
    if unit_rate_weights.size != int(n_units):
        raise ValueError('unit_rate_weights must have one value per unit')
    if np.any(~np.isfinite(unit_rate_weights)) or np.any(unit_rate_weights <= 0):
        raise ValueError('unit_rate_weights must be finite and strictly positive')
    max_weight = float(np.max(unit_rate_weights))

    if common_fraction > 0.0:
        latent_common_rate_hz = common_fraction * rate_hz * max_weight / copy_probability
    else:
        latent_common_rate_hz = np.zeros_like(rate_hz, dtype=float)
    common_events = generate_inhomogeneous_gamma_renewal_spikes(latent_common_rate_hz, t_s, common_gamma_shape_k, rng)
    spike_trains = []
    for unit_weight, private_k in zip(unit_rate_weights, private_gamma_shape_values):
        unit_rate_hz = rate_hz * float(unit_weight)
        if common_events.size:
            unit_copy_probability = copy_probability * float(unit_weight) / max_weight
            copied = common_events[rng.random(common_events.size) < unit_copy_probability]
            if jitter_s > 0 and copied.size:
                copied = copied + rng.normal(0.0, jitter_s, copied.size)
            copied = copied[(copied >= 0.0) & (copied < duration_s)]
        else:
            copied = np.array([], dtype=float)
        private_rate_hz = (1.0 - common_fraction) * unit_rate_hz
        private = generate_inhomogeneous_gamma_renewal_spikes(private_rate_hz, t_s, float(private_k), rng)
        spikes = np.sort(np.concatenate([copied, private]))
        spike_trains.append(spikes)
    return spike_trains, common_events


def construct_cst(spike_trains, duration_s, fs_hz):
    n_units = len(spike_trains)
    n_samples = int(round(float(duration_s) * float(fs_hz)))
    spike_counts = np.zeros((n_units, n_samples), dtype=np.uint16)
    for unit_i, spikes in enumerate(spike_trains):
        idx = np.floor(np.asarray(spikes, dtype=float) * float(fs_hz)).astype(int)
        idx = idx[(idx >= 0) & (idx < n_samples)]
        if idx.size:
            np.add.at(spike_counts[unit_i], idx, 1)
    binary_spikes = (spike_counts > 0).astype(np.uint8)
    cst_spikes_per_s_mu = spike_counts.sum(axis=0).astype(float) / float(n_units) * float(fs_hz)
    return cst_spikes_per_s_mu, binary_spikes


def build_causal_half_hann_kernel(width_ms, fs_hz):
    width_s = float(width_ms) / 1000.0
    if width_s <= 0:
        raise ValueError('cst_smoothing_width_ms must be strictly positive')
    width_bins = max(1, int(round(width_s * float(fs_hz))))
    lags = np.arange(width_bins + 1, dtype=float)
    kernel = 0.5 * (1.0 + np.cos(np.pi * lags / float(width_bins)))
    kernel /= float(np.sum(kernel))
    return kernel


def causal_half_hann_smooth(signal, width_ms, fs_hz):
    kernel = build_causal_half_hann_kernel(width_ms, fs_hz)
    smoothed = np.convolve(np.asarray(signal, dtype=float), kernel, mode='full')[: len(signal)]
    return smoothed, kernel


In [ ]:
def _coincidence_count_forward(x, y, lag_bins):
    x_idx = np.flatnonzero(np.asarray(x) > 0)
    y_idx = np.flatnonzero(np.asarray(y) > 0)
    if x_idx.size == 0 or y_idx.size == 0:
        return 0.0
    count = 0
    for xi in x_idx:
        insert_pos = np.searchsorted(y_idx, xi)
        matched = False
        if insert_pos < y_idx.size and abs(int(y_idx[insert_pos]) - int(xi)) <= lag_bins:
            matched = True
        elif insert_pos > 0 and abs(int(y_idx[insert_pos - 1]) - int(xi)) <= lag_bins:
            matched = True
        if matched:
            count += 1
    return float(count)


def _coincidence_count_symmetric(x, y, lag_bins):
    return 0.5 * (_coincidence_count_forward(x, y, lag_bins) + _coincidence_count_forward(y, x, lag_bins))


def compute_sliding_synchrony(binary_spikes, fs_hz, sync_window_ms, sync_step_ms, coincidence_lag_ms, *, time_offset_s=0.0):
    binary_spikes = (np.asarray(binary_spikes) > 0).astype(np.uint8)
    n_units, n_bins = binary_spikes.shape
    if n_units < 2:
        raise ValueError('At least two units are required for synchrony')
    win_bins = max(1, int(round(float(sync_window_ms) / 1000.0 * float(fs_hz))))
    step_bins = max(1, int(round(float(sync_step_ms) / 1000.0 * float(fs_hz))))
    lag_bins = max(0, int(round(float(coincidence_lag_ms) / 1000.0 * float(fs_hz))))
    if n_bins < win_bins:
        raise ValueError('Synchrony window is longer than the recording')
    starts = np.arange(0, n_bins - win_bins + 1, step_bins, dtype=int)
    stops = starts + win_bins
    pairs = list(itertools.combinations(range(n_units), 2))
    t_sync = (starts + stops - 1) / 2.0 / float(fs_hz) + float(time_offset_s)
    rate_controlled = np.zeros(starts.size, dtype=float)
    absolute_rate = np.zeros(starts.size, dtype=float)
    window_duration_s = win_bins / float(fs_hz)
    for window_i, (start, stop) in enumerate(zip(starts, stops)):
        excess_values = []
        absolute_values = []
        for ix, iy in pairs:
            x = binary_spikes[ix, start:stop]
            y = binary_spikes[iy, start:stop]
            observed = _coincidence_count_symmetric(x, y, lag_bins)
            rx = float(np.sum(x)) / float(win_bins)
            ry = float(np.sum(y)) / float(win_bins)
            expected = float(win_bins) * rx * ry * float(2 * lag_bins + 1)
            excess_values.append((observed - expected) / expected if expected > 1e-12 else 0.0)
            absolute_values.append(observed / window_duration_s)
        rate_controlled[window_i] = float(np.mean(excess_values))
        absolute_rate[window_i] = float(np.mean(absolute_values))
    return {
        'time_s': t_sync,
        'sync_rate_controlled': rate_controlled,
        'sync_absolute_coincidence_rate': absolute_rate,
        'sync_window_ms': float(sync_window_ms),
        'sync_step_ms': float(sync_step_ms),
        'coincidence_lag_ms': float(coincidence_lag_ms),
        'n_pairs': len(pairs),
    }


def compute_spike_summary(spike_trains, duration_s):
    rates = np.asarray([len(spikes) / float(duration_s) for spikes in spike_trains], dtype=float)
    isi_means = []
    isi_sds = []
    isi_cvs = []
    isi_skews = []
    isi_by_unit = []
    for spikes in spike_trains:
        isi = np.diff(np.asarray(spikes, dtype=float))
        isi = isi[np.isfinite(isi) & (isi > 0)]
        isi_by_unit.append(isi)
        if isi.size >= 2:
            mean_isi = float(np.mean(isi))
            sd_isi = float(np.std(isi, ddof=1))
            isi_means.append(mean_isi)
            isi_sds.append(sd_isi)
            isi_cvs.append(sd_isi / mean_isi if mean_isi > 0 else np.nan)
            isi_skews.append(float(skew(isi, bias=False)) if isi.size >= 3 and sd_isi > 0 else np.nan)
    isi_means = np.asarray(isi_means, dtype=float)
    isi_sds = np.asarray(isi_sds, dtype=float)
    isi_cvs = np.asarray(isi_cvs, dtype=float)
    isi_skews = np.asarray(isi_skews, dtype=float)
    finite_cvs = isi_cvs[np.isfinite(isi_cvs)]
    finite_skews = isi_skews[np.isfinite(isi_skews)]
    return {
        'unit_rates_hz': rates,
        'isi_by_unit_s': isi_by_unit,
        'isi_means_s': isi_means,
        'isi_sds_s': isi_sds,
        'isi_cvs': isi_cvs,
        'isi_skews': isi_skews,
        'mean_firing_rate_hz': float(np.mean(rates)),
        'sd_firing_rate_hz': float(np.std(rates, ddof=1)) if rates.size > 1 else np.nan,
        'isi_cv_mean': float(np.mean(finite_cvs)) if finite_cvs.size else np.nan,
        'isi_cv_sd': float(np.std(finite_cvs, ddof=1)) if finite_cvs.size > 1 else np.nan,
        'isi_skew_mean': float(np.mean(finite_skews)) if finite_skews.size else np.nan,
        'isi_skew_sd': float(np.std(finite_skews, ddof=1)) if finite_skews.size > 1 else np.nan,
    }




def filter_spike_trains_to_window(spike_trains, start_s, stop_s):
    start_s = float(start_s)
    stop_s = float(stop_s)
    return [
        np.asarray(spikes, dtype=float)[
            (np.asarray(spikes, dtype=float) >= start_s) & (np.asarray(spikes, dtype=float) < stop_s)
        ]
        for spikes in spike_trains
    ]

def run_toy_model(config_overrides=None):
    cfg = {
        'duration_s': duration_s,
        'ignored_initial_duration_s': ignored_initial_duration_s,
        'fs_hz': fs_hz,
        'n_units': n_units,
        'baseline_rate_hz': baseline_rate_hz,
        'unit_mean_rate_sd_hz': unit_mean_rate_sd_hz,
        'rate_floor_hz': rate_floor_hz,
        'random_seed': random_seed,
        'input_components': input_components,
        'enable_pulse_input': enable_pulse_input,
        'pulse_input_start_rel_analysis_s': pulse_input_start_rel_analysis_s,
        'pulse_input_duration_s': pulse_input_duration_s,
        'pulse_input_amplitude_hz': pulse_input_amplitude_hz,
        'pulse_metric_pre_window_rel_s': pulse_metric_pre_window_rel_s,
        'pulse_metric_post_window_rel_s': pulse_metric_post_window_rel_s,
        'common_fraction': common_fraction,
        'common_jitter_sd_ms': common_jitter_sd_ms,
        'common_copy_probability': common_copy_probability,
        'private_gamma_shape_k': private_gamma_shape_k,
        'common_gamma_shape_k': common_gamma_shape_k,
        'gamma_shape_k_sd': gamma_shape_k_sd,
        'cst_smoothing_width_ms': cst_smoothing_width_ms,
        'sync_window_ms': sync_window_ms,
        'sync_step_ms': sync_step_ms,
        'coincidence_lag_ms': coincidence_lag_ms,
    }
    if config_overrides:
        cfg.update(config_overrides)
    cfg['duration_s'] = float(cfg['duration_s'])
    cfg['ignored_initial_duration_s'] = float(cfg.get('ignored_initial_duration_s', 0.0))
    if cfg['duration_s'] <= 0:
        raise ValueError('duration_s must be strictly positive')
    if cfg['ignored_initial_duration_s'] < 0:
        raise ValueError('ignored_initial_duration_s must be >= 0')
    cfg['analysis_start_s'] = cfg['ignored_initial_duration_s']
    cfg['analysis_end_s'] = cfg['ignored_initial_duration_s'] + cfg['duration_s']
    cfg['total_duration_s'] = cfg['analysis_end_s']
    cfg['enable_pulse_input'] = bool(cfg.get('enable_pulse_input', False))
    cfg['pulse_input_start_rel_analysis_s'] = float(cfg.get('pulse_input_start_rel_analysis_s', 1.0))
    cfg['pulse_input_duration_s'] = float(cfg.get('pulse_input_duration_s', 0.0))
    cfg['pulse_input_amplitude_hz'] = float(cfg.get('pulse_input_amplitude_hz', 0.0))
    cfg['pulse_metric_pre_window_rel_s'] = tuple(float(x) for x in cfg.get('pulse_metric_pre_window_rel_s', (-0.5, 0.0)))
    cfg['pulse_metric_post_window_rel_s'] = tuple(float(x) for x in cfg.get('pulse_metric_post_window_rel_s', (0.0, 1.0)))
    if len(cfg['pulse_metric_pre_window_rel_s']) != 2 or len(cfg['pulse_metric_post_window_rel_s']) != 2:
        raise ValueError('Pulse metric windows must contain exactly two values')
    cfg['pulse_input_start_abs_s'] = cfg['analysis_start_s'] + cfg['pulse_input_start_rel_analysis_s']

    rng = np.random.default_rng(int(cfg['random_seed']))
    rate = generate_rate_process(
        cfg['total_duration_s'],
        cfg['fs_hz'],
        cfg['baseline_rate_hz'],
        cfg['input_components'],
        cfg['rate_floor_hz'],
        rng,
        enable_pulse_input=cfg['enable_pulse_input'],
        pulse_input_start_abs_s=cfg['pulse_input_start_abs_s'],
        pulse_input_duration_s=cfg['pulse_input_duration_s'],
        pulse_input_amplitude_hz=cfg['pulse_input_amplitude_hz'],
    )
    cfg.update(rate.get('pulse_metadata', {}))
    cfg['pulse_metric_pre_window_abs_s'] = (
        cfg['pulse_input_start_abs_s'] + cfg['pulse_metric_pre_window_rel_s'][0],
        cfg['pulse_input_start_abs_s'] + cfg['pulse_metric_pre_window_rel_s'][1],
    )
    cfg['pulse_metric_post_window_abs_s'] = (
        cfg['pulse_input_start_abs_s'] + cfg['pulse_metric_post_window_rel_s'][0],
        cfg['pulse_input_start_abs_s'] + cfg['pulse_metric_post_window_rel_s'][1],
    )
    analysis_start_idx = int(round(cfg['analysis_start_s'] * float(cfg['fs_hz'])))
    analysis_n_bins = int(round(cfg['duration_s'] * float(cfg['fs_hz'])))
    analysis_stop_idx = min(analysis_start_idx + analysis_n_bins, len(rate['time_s']))
    analysis_slice = slice(analysis_start_idx, analysis_stop_idx)
    if analysis_stop_idx <= analysis_start_idx:
        raise ValueError('Analyzed window is empty after applying ignored_initial_duration_s')

    unit_rate_weights, unit_target_mean_rates_hz = make_unit_rate_weights(
        rate['rate_hz'],
        analysis_slice,
        cfg['n_units'],
        cfg.get('unit_mean_rate_sd_hz', 0.0),
        min_mean_rate_hz=cfg['rate_floor_hz'],
    )
    unit_rate_traces_full_hz = unit_rate_weights[:, None] * rate['rate_hz'][None, :]
    private_gamma_shape_values, common_gamma_shape_value = make_gamma_shape_values(
        cfg['private_gamma_shape_k'],
        cfg['common_gamma_shape_k'],
        cfg.get('gamma_shape_k_sd', 0.0),
        cfg['n_units'],
        rng,
    )
    spikes_full, common_events_full = generate_spike_trains(
        rate['rate_hz'], rate['time_s'], cfg['n_units'], cfg['common_fraction'], cfg['common_jitter_sd_ms'],
        cfg['common_copy_probability'], private_gamma_shape_values, common_gamma_shape_value, rng,
        unit_rate_weights=unit_rate_weights,
    )
    cst_full, binary_spikes_full = construct_cst(spikes_full, cfg['total_duration_s'], cfg['fs_hz'])
    cst_smooth_full, kernel = causal_half_hann_smooth(cst_full, cfg['cst_smoothing_width_ms'], cfg['fs_hz'])

    spikes = filter_spike_trains_to_window(spikes_full, cfg['analysis_start_s'], cfg['analysis_end_s'])
    common_events = common_events_full[(common_events_full >= cfg['analysis_start_s']) & (common_events_full < cfg['analysis_end_s'])]
    binary_spikes = binary_spikes_full[:, analysis_slice]
    sync = compute_sliding_synchrony(
        binary_spikes,
        cfg['fs_hz'],
        cfg['sync_window_ms'],
        cfg['sync_step_ms'],
        cfg['coincidence_lag_ms'],
        time_offset_s=cfg['analysis_start_s'],
    )
    spike_summary = compute_spike_summary(spikes, cfg['duration_s'])
    return {
        'config': cfg,
        'time_s': rate['time_s'][analysis_slice],
        'time_full_s': rate['time_s'],
        'rate_hz': rate['rate_hz'][analysis_slice],
        'rate_full_hz': rate['rate_hz'],
        'raw_rate_hz': rate['raw_rate_hz'][analysis_slice],
        'raw_rate_full_hz': rate['raw_rate_hz'],
        'pulse_trace_hz': rate['pulse_trace_hz'][analysis_slice],
        'pulse_trace_full_hz': rate['pulse_trace_hz'],
        'component_traces_hz': {name: trace[analysis_slice] for name, trace in rate['component_traces_hz'].items()},
        'component_traces_full_hz': rate['component_traces_hz'],
        'unit_rate_weights': unit_rate_weights,
        'unit_target_mean_rates_hz': unit_target_mean_rates_hz,
        'unit_rate_traces_hz': unit_rate_traces_full_hz[:, analysis_slice],
        'private_gamma_shape_values': private_gamma_shape_values,
        'common_gamma_shape_value': common_gamma_shape_value,
        'unit_rate_traces_full_hz': unit_rate_traces_full_hz,
        'spike_trains': spikes,
        'spike_trains_full': spikes_full,
        'common_events_s': common_events,
        'common_events_full_s': common_events_full,
        'binary_spikes': binary_spikes,
        'binary_spikes_full': binary_spikes_full,
        'cst_spikes_per_s_mu': cst_full[analysis_slice],
        'cst_full_spikes_per_s_mu': cst_full,
        'cst_smooth_spikes_per_s_mu': cst_smooth_full[analysis_slice],
        'cst_smooth_full_spikes_per_s_mu': cst_smooth_full,
        'cst_smoothing_kernel': kernel,
        'sync': sync,
        'spike_summary': spike_summary,
    }


EXPERIMENTAL_REQUIRED_COLUMNS = {
    'participant_id': 'DATASET_CONTEXT_participant_id',
    'condition_id': 'DATASET_CONTEXT_condition_id',
    'firing_rate_mean': 'OBS_FIRING_STATISTICS_firing_rate_mean',
    'firing_rate_sd': 'OBS_FIRING_STATISTICS_firing_rate_sd',
    'firing_rate_skew': 'OBS_FIRING_STATISTICS_firing_rate_skew',
    'isi_cv_mean': 'OBS_FIRING_STATISTICS_isi_cv_mean',
    'isi_cv_sd': 'OBS_FIRING_STATISTICS_isi_cv_sd',
    'isi_cv_skew': 'OBS_FIRING_STATISTICS_isi_cv_skew',
    'sync_mean': 'OBS_WINDOW_m3_to_m2_SYNC_mean',
    'sync_baseline_mean': 'OBS_WINDOW_m3_to_m2_SYNC_mean',
    'sync_baseline_max': 'OBS_WINDOW_m3_to_m2_SYNC_max',
    'sync_response_mean': 'OBS_WINDOW_0_to_1_SYNC_mean',
    'sync_response_max': 'OBS_WINDOW_0_to_1_SYNC_max',
}


def load_experimental_firing_isi_moments(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Experimental feature table not found: {path}')
    df = pd.read_pickle(path)
    missing = [col for col in EXPERIMENTAL_REQUIRED_COLUMNS.values() if col not in df.columns]
    if missing:
        raise KeyError(f'Experimental table is missing required columns: {missing}')
    out = pd.DataFrame({name: df[col] for name, col in EXPERIMENTAL_REQUIRED_COLUMNS.items()})
    numeric_cols = [c for c in out.columns if c not in {'participant_id', 'condition_id'}]
    for col in numeric_cols:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    out['approx_mean_isi_ms'] = 1000.0 / out['firing_rate_mean'].replace(0, np.nan)
    out['approx_isi_sd_ms'] = out['isi_cv_mean'] * out['approx_mean_isi_ms']
    return out.replace([np.inf, -np.inf], np.nan)


def extract_toy_feature_moments(result):
    summary = result['spike_summary']
    rates = np.asarray(summary['unit_rates_hz'], dtype=float)
    cvs = np.asarray(summary['isi_cvs'], dtype=float)
    finite_rates = rates[np.isfinite(rates)]
    finite_cvs = cvs[np.isfinite(cvs)]
    return pd.Series({
        'firing_rate_mean': float(np.mean(finite_rates)) if finite_rates.size else np.nan,
        'firing_rate_sd': float(np.std(finite_rates, ddof=1)) if finite_rates.size > 1 else np.nan,
        'firing_rate_skew': float(skew(finite_rates, bias=False)) if finite_rates.size >= 3 and np.std(finite_rates, ddof=0) > 0 else np.nan,
        'isi_cv_mean': float(np.mean(finite_cvs)) if finite_cvs.size else np.nan,
        'isi_cv_sd': float(np.std(finite_cvs, ddof=1)) if finite_cvs.size > 1 else np.nan,
        'isi_cv_skew': float(skew(finite_cvs, bias=False)) if finite_cvs.size >= 3 and np.std(finite_cvs, ddof=0) > 0 else np.nan,
    }, name='toy_value')


def summarize_toy_result(result):
    sync = result['sync']
    spike_summary = result['spike_summary']
    sync_rate_controlled = np.asarray(sync['sync_rate_controlled'], dtype=float)
    sync_absolute = np.asarray(sync['sync_absolute_coincidence_rate'], dtype=float)
    cst_raw = np.asarray(result['cst_spikes_per_s_mu'], dtype=float)
    cst_smooth = np.asarray(result['cst_smooth_spikes_per_s_mu'], dtype=float)
    cfg = result['config']
    t = np.asarray(result['time_s'], dtype=float)
    sync_t = np.asarray(sync['time_s'], dtype=float)
    dt = 1.0 / float(cfg['fs_hz'])
    unit_rate_traces = np.asarray(result.get('unit_rate_traces_hz', result['rate_hz']), dtype=float)
    if unit_rate_traces.ndim == 2:
        mean_rate_hz = np.nanmean(unit_rate_traces, axis=0)
    else:
        mean_rate_hz = np.asarray(result['rate_hz'], dtype=float)
    cumulative_input = np.cumsum(mean_rate_hz) * dt
    cumulative_cst = np.cumsum(cst_raw) * dt
    cumulative_residual = cumulative_cst - cumulative_input

    def _mean_sd_cov(values):
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]
        if values.size == 0:
            return np.nan, np.nan, np.nan
        mean_value = float(np.mean(values))
        sd_value = float(np.std(values, ddof=0))
        cov_value = sd_value / abs(mean_value) if abs(mean_value) > 1e-12 else np.nan
        return mean_value, sd_value, cov_value

    def _moment_stats(values):
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]
        if values.size == 0:
            return {'mean': np.nan, 'sd': np.nan, 'skew': np.nan, 'median': np.nan, 'iqr': np.nan}
        q25, q75 = np.percentile(values, [25, 75])
        return {
            'mean': float(np.mean(values)),
            'sd': float(np.std(values, ddof=0)),
            'skew': float(skew(values, bias=False)) if values.size >= 3 and float(np.std(values, ddof=0)) > 0 else np.nan,
            'median': float(np.median(values)),
            'iqr': float(q75 - q25),
        }

    def _window_values(time_values, trace_values, window_abs_s):
        time_values = np.asarray(time_values, dtype=float)
        trace_values = np.asarray(trace_values, dtype=float)
        start_s, end_s = float(window_abs_s[0]), float(window_abs_s[1])
        mask = (time_values >= start_s) & (time_values < end_s)
        return trace_values[mask]

    def _add_pulse_window_stats(out, metric_name, time_values, trace_values):
        pre = _moment_stats(_window_values(time_values, trace_values, cfg['pulse_metric_pre_window_abs_s']))
        post = _moment_stats(_window_values(time_values, trace_values, cfg['pulse_metric_post_window_abs_s']))
        for stat_name in ['mean', 'sd', 'skew', 'median', 'iqr']:
            out[f'pulse_pre_{metric_name}_{stat_name}'] = pre[stat_name]
            out[f'pulse_post_{metric_name}_{stat_name}'] = post[stat_name]
            out[f'pulse_effect_{metric_name}_{stat_name}'] = post[stat_name] - pre[stat_name] if np.isfinite(post[stat_name]) and np.isfinite(pre[stat_name]) else np.nan

    def _age_stats(values):
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]
        if values.size == 0:
            return {'mean': np.nan, 'sd': np.nan, 'min': np.nan, 'max': np.nan, 'skew': np.nan, 'median': np.nan, 'iqr': np.nan}
        q25, q75 = np.percentile(values, [25, 75])
        return {
            'mean': float(np.mean(values)),
            'sd': float(np.std(values, ddof=0)),
            'min': float(np.min(values)),
            'max': float(np.max(values)),
            'skew': float(skew(values, bias=False)) if values.size >= 3 and float(np.std(values, ddof=0)) > 0 else np.nan,
            'median': float(np.median(values)),
            'iqr': float(q75 - q25),
        }

    def _compute_pool_age_metrics():
        t_step = float(cfg.get('pulse_input_start_abs_s', np.nan))
        full_time = np.asarray(result.get('time_full_s', []), dtype=float)
        full_rates = np.asarray(result.get('unit_rate_traces_full_hz', []), dtype=float)
        spikes_full = result.get('spike_trains_full', None)
        if not np.isfinite(t_step) or full_time.size == 0 or full_rates.ndim != 2 or spikes_full is None:
            out = {'pool_age_metric_source': 'unavailable', 'pool_age_censored_fraction': np.nan}
            for prefix in ['pool_integrated_age', 'pool_raw_age_s']:
                for stat_name in ['mean', 'sd', 'min', 'max', 'skew', 'median', 'iqr']:
                    out[f'{prefix}_{stat_name}'] = np.nan
            return out
        dt_age = 1.0 / float(cfg['fs_hz'])
        step_count = int(np.searchsorted(full_time, t_step, side='left'))
        step_count = max(0, min(step_count, full_time.size))
        cumulative_intensity = np.concatenate(
            [np.zeros((full_rates.shape[0], 1), dtype=float), np.cumsum(full_rates * dt_age, axis=1)],
            axis=1,
        )
        integrated_ages = []
        raw_ages = []
        censored = []
        for unit_i, spikes in enumerate(spikes_full):
            spikes = np.asarray(spikes, dtype=float)
            previous = spikes[spikes < t_step]
            if previous.size:
                last_spike = float(previous[-1])
                censored.append(False)
            else:
                last_spike = 0.0
                censored.append(True)
            last_count = int(np.searchsorted(full_time, last_spike, side='left'))
            last_count = max(0, min(last_count, full_time.size))
            integrated_ages.append(float(cumulative_intensity[unit_i, step_count] - cumulative_intensity[unit_i, last_count]))
            raw_ages.append(float(t_step - last_spike))
        integrated_stats = _age_stats(integrated_ages)
        raw_stats = _age_stats(raw_ages)
        out = {
            'pool_age_metric_source': 'integrated_age',
            'pool_age_censored_fraction': float(np.mean(censored)) if censored else np.nan,
        }
        for stat_name, stat_value in integrated_stats.items():
            out[f'pool_integrated_age_{stat_name}'] = stat_value
        for stat_name, stat_value in raw_stats.items():
            out[f'pool_raw_age_s_{stat_name}'] = stat_value
        return out

    sync_mean, sync_sd, sync_cov = _mean_sd_cov(sync_rate_controlled)
    abs_sync_mean, abs_sync_sd, abs_sync_cov = _mean_sd_cov(sync_absolute)
    cst_mean, cst_sd, cst_cov = _mean_sd_cov(cst_smooth)
    raw_cst_stats = _moment_stats(cst_raw)
    cumulative_residual_stats = _moment_stats(cumulative_residual)
    pool_age_stats = _compute_pool_age_metrics()
    out = {
        'mean_firing_rate_hz': spike_summary['mean_firing_rate_hz'],
        'sd_firing_rate_hz': spike_summary['sd_firing_rate_hz'],
        'target_unit_mean_rate_sd_hz': float(result['config'].get('unit_mean_rate_sd_hz', 0.0)),
        'actual_unit_target_mean_rate_sd_hz': float(np.std(result.get('unit_target_mean_rates_hz', []), ddof=0)) if len(result.get('unit_target_mean_rates_hz', [])) else np.nan,
        'target_gamma_shape_k_sd': float(result['config'].get('gamma_shape_k_sd', 0.0)),
        'actual_private_gamma_shape_k_sd': float(np.std(result.get('private_gamma_shape_values', []), ddof=0)) if len(result.get('private_gamma_shape_values', [])) else np.nan,
        'actual_common_gamma_shape_k': float(result.get('common_gamma_shape_value', np.nan)),
        'isi_cv_mean': spike_summary['isi_cv_mean'],
        'isi_cv_sd': spike_summary['isi_cv_sd'],
        'isi_skew_mean': spike_summary['isi_skew_mean'],
        'isi_skew_sd': spike_summary['isi_skew_sd'],
        'mean_sync_rate_controlled': sync_mean,
        'sd_sync_rate_controlled': sync_sd,
        'cov_sync_rate_controlled': sync_cov,
        'mean_sync_absolute_coincidence_rate_per_s_pair': abs_sync_mean,
        'sd_sync_absolute_coincidence_rate_per_s_pair': abs_sync_sd,
        'cov_sync_absolute_coincidence_rate_per_s_pair': abs_sync_cov,
        'mean_filtered_cst_spikes_per_s_mu': cst_mean,
        'sd_filtered_cst_spikes_per_s_mu': cst_sd,
        'cov_filtered_cst_spikes_per_s_mu': cst_cov,
        'n_common_events': int(len(result['common_events_s'])),
        'pulse_input_enabled': bool(cfg.get('pulse_input_enabled', False)),
        'pulse_input_amplitude_hz': float(cfg.get('pulse_input_amplitude_hz', np.nan)),
        'pulse_input_start_rel_analysis_s': float(cfg.get('pulse_input_start_rel_analysis_s', np.nan)),
        'pulse_input_start_abs_s': float(cfg.get('pulse_input_start_abs_s', np.nan)),
        'pulse_input_end_abs_s': float(cfg.get('pulse_input_end_abs_s', np.nan)),
        'pulse_input_duration_s': float(cfg.get('pulse_input_duration_s', np.nan)),
        'pulse_input_duration_effective_s': float(cfg.get('pulse_input_duration_effective_s', np.nan)),
        'pulse_input_area_expected_spikes_per_neuron': float(cfg.get('pulse_input_area_expected_spikes_per_neuron', np.nan)),
        'pulse_metric_pre_window_start_abs_s': float(cfg['pulse_metric_pre_window_abs_s'][0]),
        'pulse_metric_pre_window_end_abs_s': float(cfg['pulse_metric_pre_window_abs_s'][1]),
        'pulse_metric_post_window_start_abs_s': float(cfg['pulse_metric_post_window_abs_s'][0]),
        'pulse_metric_post_window_end_abs_s': float(cfg['pulse_metric_post_window_abs_s'][1]),
    }
    for stat_name, stat_value in raw_cst_stats.items():
        out[f'{stat_name}_raw_cst_spikes_per_s_mu'] = stat_value
    for stat_name, stat_value in cumulative_residual_stats.items():
        out[f'{stat_name}_cumulative_cst_minus_input_residual_spikes_per_mu'] = stat_value
    out.update(pool_age_stats)
    _add_pulse_window_stats(out, 'raw_cst_spikes_per_s_mu', t, cst_raw)
    _add_pulse_window_stats(out, 'filtered_cst_spikes_per_s_mu', t, cst_smooth)
    _add_pulse_window_stats(out, 'sync_rate_controlled', sync_t, sync_rate_controlled)
    _add_pulse_window_stats(out, 'sync_absolute_coincidence_rate_per_s_pair', sync_t, sync_absolute)
    _add_pulse_window_stats(out, 'cumulative_input_expected_spikes_per_neuron', t, cumulative_input)
    _add_pulse_window_stats(out, 'cumulative_cst_spikes_per_mu', t, cumulative_cst)
    _add_pulse_window_stats(out, 'cumulative_cst_minus_input_residual_spikes_per_mu', t, cumulative_residual)
    return pd.Series(out, name='value')



def _json_safe(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): _json_safe(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return _json_safe(value.tolist())
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value


def make_grid_output_path(output_dir=None, run_name=None):
    output_dir = Path(GRID_OUTPUT_DIR if output_dir is None else output_dir)
    run_name = str(GRID_OUTPUT_RUN_NAME if run_name is None else run_name)
    if not run_name.strip():
        raise ValueError('GRID_OUTPUT_RUN_NAME cannot be empty')
    return output_dir / run_name


def _resolve_grid_realization_seeds_for_snapshot():
    if GRID_REALIZATION_SEEDS is None:
        n_realizations = max(1, int(GRID_N_REALIZATIONS))
        return [int(GRID_BASE_RANDOM_SEED) + realization_index for realization_index in range(n_realizations)]
    seeds = [int(seed) for seed in GRID_REALIZATION_SEEDS]
    if not seeds:
        raise ValueError('GRID_REALIZATION_SEEDS cannot be empty')
    return seeds


def collect_grid_config_snapshot():
    return _json_safe({
        'created_by': 'toy_direct_synchrony_cst_model.ipynb',
        'toy_model_config': {
            'duration_s': duration_s,
            'ignored_initial_duration_s': ignored_initial_duration_s,
            'fs_hz': fs_hz,
            'n_units': n_units,
            'baseline_rate_hz': baseline_rate_hz,
            'unit_mean_rate_sd_hz': unit_mean_rate_sd_hz,
            'rate_floor_hz': rate_floor_hz,
            'random_seed': random_seed,
            'input_components': input_components,
            'enable_pulse_input': enable_pulse_input,
            'pulse_input_start_rel_analysis_s': pulse_input_start_rel_analysis_s,
            'pulse_input_duration_s': pulse_input_duration_s,
            'pulse_input_amplitude_hz': pulse_input_amplitude_hz,
            'pulse_metric_pre_window_rel_s': pulse_metric_pre_window_rel_s,
            'pulse_metric_post_window_rel_s': pulse_metric_post_window_rel_s,
            'common_fraction': common_fraction,
            'common_jitter_sd_ms': common_jitter_sd_ms,
            'common_copy_probability': common_copy_probability,
            'private_gamma_shape_k': private_gamma_shape_k,
            'common_gamma_shape_k': common_gamma_shape_k,
            'gamma_shape_k_sd': gamma_shape_k_sd,
            'cst_smoothing_width_ms': cst_smoothing_width_ms,
            'sync_window_ms': sync_window_ms,
            'sync_step_ms': sync_step_ms,
            'coincidence_lag_ms': coincidence_lag_ms,
        },
        'grid_config': {
            'grid_parameter_values': GRID_PARAMETER_VALUES,
            'grid_x_parameter': GRID_X_PARAMETER,
            'grid_y_parameter': GRID_Y_PARAMETER,
            'grid_base_random_seed': GRID_BASE_RANDOM_SEED,
            'grid_n_realizations': GRID_N_REALIZATIONS,
            'grid_realization_seeds': GRID_REALIZATION_SEEDS,
            'resolved_realization_seeds': _resolve_grid_realization_seeds_for_snapshot(),
        },
        'plot_config': {
            'grid_metric_specs': GRID_METRIC_SPECS,
            'grid_curve_x_parameter': GRID_CURVE_X_PARAMETER,
            'grid_curve_hue_parameter': GRID_CURVE_HUE_PARAMETER,
            'grid_curve_colormap_min_fraction': GRID_CURVE_COLORMAP_MIN_FRACTION,
            'grid_heatmap_annotation_max_cells': GRID_HEATMAP_ANNOTATION_MAX_CELLS,
            'grid_show_experimental_sync_references': GRID_SHOW_EXPERIMENTAL_SYNC_REFERENCES,
            'grid_experimental_baseline_sync_color': GRID_EXPERIMENTAL_BASELINE_SYNC_COLOR,
            'grid_experimental_response_sync_color': GRID_EXPERIMENTAL_RESPONSE_SYNC_COLOR,
            'grid_experimental_reference_alpha': GRID_EXPERIMENTAL_REFERENCE_ALPHA,
            'grid_save_figure_formats': GRID_SAVE_FIGURE_FORMATS,
            'experimental_feature_table_path': EXPERIMENTAL_FEATURE_TABLE_PATH,
            'load_experimental_moments': LOAD_EXPERIMENTAL_MOMENTS,
        },
    })


def grid_config_hash(config_snapshot):
    payload = json.dumps(_json_safe(config_snapshot), sort_keys=True, separators=(',', ':'))
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


def _write_json(path, payload):
    path = Path(path)
    path.write_text(json.dumps(_json_safe(payload), indent=2, sort_keys=True), encoding='utf-8')


def _read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def save_grid_outputs(output_path, *, grid_df=None, grid_summary_df=None, grid_curve_fit_df=None, config_snapshot=None, manifest_updates=None):
    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)
    table_paths = {}
    if config_snapshot is not None:
        _write_json(output_path / 'grid_config.json', {
            'config_hash': grid_config_hash(config_snapshot),
            'config_snapshot': config_snapshot,
        })
    for table_name, table in [
        ('grid_df', grid_df),
        ('grid_summary_df', grid_summary_df),
        ('grid_curve_fit_df', grid_curve_fit_df),
    ]:
        if table is None:
            continue
        csv_path = output_path / f'{table_name}.csv'
        pkl_path = output_path / f'{table_name}.pkl'
        table.to_csv(csv_path, index=False)
        table.to_pickle(pkl_path)
        table_paths[f'{table_name}_csv'] = str(csv_path)
        table_paths[f'{table_name}_pkl'] = str(pkl_path)
    manifest_path = output_path / 'grid_output_manifest.json'
    manifest = {}
    if manifest_path.exists():
        try:
            manifest = _read_json(manifest_path)
        except Exception:
            manifest = {}
    manifest.update({
        'updated_at_utc': datetime.now(timezone.utc).isoformat(),
        'output_path': str(output_path),
        'table_paths': {**manifest.get('table_paths', {}), **table_paths},
    })
    if config_snapshot is not None:
        manifest['config_hash'] = grid_config_hash(config_snapshot)
    if grid_df is not None:
        manifest['grid_df_rows'] = int(len(grid_df))
    if grid_summary_df is not None:
        manifest['grid_summary_df_rows'] = int(len(grid_summary_df))
    if grid_curve_fit_df is not None:
        manifest['grid_curve_fit_df_rows'] = int(len(grid_curve_fit_df))
    if manifest_updates:
        manifest.update(_json_safe(manifest_updates))
    _write_json(manifest_path, manifest)
    return manifest


def load_grid_outputs(output_path):
    output_path = Path(output_path)
    grid_df_path = output_path / 'grid_df.pkl'
    config_path = output_path / 'grid_config.json'
    if not grid_df_path.exists():
        raise FileNotFoundError(f'Cached grid table not found: {grid_df_path}')
    if not config_path.exists():
        raise FileNotFoundError(f'Cached grid config not found: {config_path}')
    config_payload = _read_json(config_path)
    return {
        'output_path': output_path,
        'grid_df': pd.read_pickle(grid_df_path),
        'config_payload': config_payload,
        'config_snapshot': config_payload.get('config_snapshot', {}),
        'config_hash': config_payload.get('config_hash'),
    }


In [ ]:
def plot_toy_summary(result, *, show_components=True, xlim_s=None, experimental_moments_df=None, figsize=(14, 10)):
    cfg = result['config']
    t = np.asarray(result['time_s'], dtype=float)
    sync = result['sync']
    experimental_sync_mean = np.nan
    if experimental_moments_df is not None and 'sync_mean' in experimental_moments_df.columns:
        _sync_values = pd.to_numeric(experimental_moments_df['sync_mean'], errors='coerce').to_numpy(dtype=float)
        _sync_values = _sync_values[np.isfinite(_sync_values)]
        if _sync_values.size:
            experimental_sync_mean = float(np.mean(_sync_values))

    fig, axes = plt.subplots(
        4,
        2,
        figsize=figsize,
        constrained_layout=True,
        gridspec_kw={'width_ratios': [2.1, 1.15]},
    )

    def _set_time_xlim(ax):
        if xlim_s is not None:
            ax.set_xlim(*xlim_s)
        else:
            ax.set_xlim(float(cfg.get('analysis_start_s', t[0])), float(cfg.get('analysis_end_s', t[-1])))

    def _add_pulse_window(ax, *, label=False):
        if not cfg.get('pulse_input_enabled', False):
            return
        start_s = float(cfg.get('pulse_input_start_abs_s', np.nan))
        end_s = float(cfg.get('pulse_input_end_abs_s', np.nan))
        amp_hz = float(cfg.get('pulse_input_amplitude_hz', np.nan))
        if not np.isfinite(start_s) or not np.isfinite(end_s) or end_s <= start_s:
            return
        ax.axvspan(start_s, end_s, color='#2ca02c', alpha=0.08, lw=0, label=f'pulse window ({amp_hz:g} Hz)' if label else None)
        ax.axvline(start_s, color='#2ca02c', lw=1.0, ls='--', alpha=0.8)
        ax.axvline(end_s, color='#2ca02c', lw=1.0, ls=':', alpha=0.7)

    def _distribution_panel(ax, values, *, color, title, xlabel, experimental_reference=None, experimental_label='experimental mean'):
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]
        if values.size == 0:
            ax.text(0.5, 0.5, 'No finite values', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(title)
            ax.set_xlabel(xlabel)
            return
        bins = min(24, max(6, int(np.sqrt(values.size))))
        ax.hist(values, bins=bins, density=True, color=color, alpha=0.30)
        if values.size >= 3 and float(np.std(values, ddof=0)) > 0:
            grid = np.linspace(float(np.min(values)), float(np.max(values)), 240)
            if not np.isclose(grid[0], grid[-1]):
                try:
                    ax.plot(grid, gaussian_kde(values)(grid), color=color, lw=1.8)
                except Exception:
                    pass
        mean_value = float(np.mean(values))
        sd_value = float(np.std(values, ddof=0)) if values.size > 1 else np.nan
        ax.axvline(mean_value, color=color, lw=2.2, label='toy mean')
        if np.isfinite(sd_value) and sd_value > 0:
            ax.axvline(mean_value - sd_value, color=color, lw=1.2, ls='--', alpha=0.75)
            ax.axvline(mean_value + sd_value, color=color, lw=1.2, ls='--', alpha=0.75)
        if experimental_reference is not None and np.isfinite(experimental_reference):
            ax.axvline(float(experimental_reference), color='black', lw=2.2, label=experimental_label)
        ax.text(
            0.98,
            0.96,
            f'mean={mean_value:.3g}\nSD={sd_value:.3g}' + (f'\nexp mean={float(experimental_reference):.3g}' if experimental_reference is not None and np.isfinite(experimental_reference) else ''),
            ha='right',
            va='top',
            transform=ax.transAxes,
            fontsize=8,
            bbox={'facecolor': 'white', 'edgecolor': '0.75', 'alpha': 0.9},
        )
        ax.set_title(title)
        ax.set_xlabel(xlabel)
        ax.set_ylabel('density')
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            unique = dict(zip(labels, handles))
            ax.legend(unique.values(), unique.keys(), fontsize=7, loc='best')

    # Row 1: input rate and its cumulative intensity.
    ax = axes[0, 0]
    unit_rate_traces = result.get('unit_rate_traces_hz')
    if unit_rate_traces is not None and np.asarray(unit_rate_traces).ndim == 2 and np.asarray(unit_rate_traces).shape[0] > 1:
        unit_rate_traces = np.asarray(unit_rate_traces, dtype=float)
        mean_unit_rate_hz = np.nanmean(unit_rate_traces, axis=0)
        if float(cfg.get('unit_mean_rate_sd_hz', 0.0)) > 0:
            ax.fill_between(
                t,
                np.nanmin(unit_rate_traces, axis=0),
                np.nanmax(unit_rate_traces, axis=0),
                color='black',
                alpha=0.10,
                linewidth=0,
                label='unit rate range',
            )
            ax.plot(t, mean_unit_rate_hz, color='black', lw=1.4, label='mean unit rate')
        else:
            ax.plot(t, result['rate_hz'], color='black', lw=1.4, label='final rate')
    else:
        mean_unit_rate_hz = np.asarray(result['rate_hz'], dtype=float)
        ax.plot(t, result['rate_hz'], color='black', lw=1.4, label='final rate')
    ax.plot(t, result['raw_rate_hz'], color='0.55', lw=0.9, alpha=0.8, label='raw rate')
    if show_components:
        for name, trace in result['component_traces_hz'].items():
            ax.plot(t, cfg['baseline_rate_hz'] + trace, lw=0.8, alpha=0.45, label=f'baseline + {name}')
    ax.set_ylabel('rate\n(Hz)')
    ax.legend(loc='upper right', fontsize=8, ncol=2)
    _add_pulse_window(ax, label=True)
    _set_time_xlim(ax)

    dt = 1.0 / float(cfg['fs_hz'])
    axes[0, 1].plot(t, np.cumsum(mean_unit_rate_hz) * dt, color='black', lw=1.5)
    axes[0, 1].set_title('Integrated input')
    axes[0, 1].set_ylabel('expected total spikes\nper neuron')
    _add_pulse_window(axes[0, 1])
    _set_time_xlim(axes[0, 1])

    # Row 2: causal-filtered CST and cumulative CST.
    axes[1, 0].plot(t, result['cst_smooth_spikes_per_s_mu'], color='#ff3300', lw=1.4, label='causal half-Hann CST')
    axes[1, 0].set_ylabel('causal half-Hann CST\n(sp/s/MU)')
    axes[1, 0].legend(loc='upper right', fontsize=8)
    _add_pulse_window(axes[1, 0])
    _set_time_xlim(axes[1, 0])

    axes[1, 1].plot(t, np.cumsum(np.asarray(result['cst_spikes_per_s_mu'], dtype=float)) * dt, color='#ff3300', lw=1.5)
    axes[1, 1].set_title('Integrated CST')
    axes[1, 1].set_ylabel('total spikes\nper neuron')
    _add_pulse_window(axes[1, 1])
    _set_time_xlim(axes[1, 1])

    # Row 3: rate-controlled synchrony and its distribution.
    axes[2, 0].plot(sync['time_s'], sync['sync_rate_controlled'], color='#852680', lw=1.2)
    axes[2, 0].axhline(0, color='black', ls='--', lw=0.8, alpha=0.6)
    axes[2, 0].set_ylabel('rate-normalized\nsynchrony')
    _add_pulse_window(axes[2, 0])
    _set_time_xlim(axes[2, 0])
    _distribution_panel(
        axes[2, 1],
        sync['sync_rate_controlled'],
        color='#852680',
        title='Rate-normalized synchrony distribution',
        xlabel='rate-normalized synchrony',
        experimental_reference=experimental_sync_mean,
        experimental_label='experimental mean',
    )

    # Row 4: absolute synchrony and its distribution.
    axes[3, 0].plot(sync['time_s'], sync['sync_absolute_coincidence_rate'], color='#0050cc', lw=1.2)
    axes[3, 0].set_ylabel('absolute coincidence\nrate / pair (1/s)')
    axes[3, 0].set_xlabel('time (s)')
    _add_pulse_window(axes[3, 0])
    _set_time_xlim(axes[3, 0])
    _distribution_panel(
        axes[3, 1],
        sync['sync_absolute_coincidence_rate'],
        color='#0050cc',
        title='Absolute synchrony distribution',
        xlabel='coincidence rate / pair (1/s)',
    )

    for row in range(2):
        axes[row, 1].set_xlabel('time (s)')
    for ax in axes.ravel():
        ax.grid(alpha=0.18)

    title = (
        f"Toy synchrony CST model | n={cfg['n_units']} | baseline={cfg['baseline_rate_hz']:.1f} Hz | "
        f"ignored={cfg.get('ignored_initial_duration_s', 0.0):.1f}s | "
        f"unit rate SD={cfg.get('unit_mean_rate_sd_hz', 0.0):.1f} Hz | "
        f"common_fraction={cfg['common_fraction']:.2f} | copy_p={cfg['common_copy_probability']:.2f} | "
        f"jitter={cfg['common_jitter_sd_ms']:.1f} ms | private k={cfg['private_gamma_shape_k']:.2f} | "
        f"common k={cfg['common_gamma_shape_k']:.2f} | k SD={cfg.get('gamma_shape_k_sd', 0.0):.2f} | Hann width={cfg['cst_smoothing_width_ms']:.0f} ms | "
        f"sync win={cfg['sync_window_ms']:.0f} ms lag={cfg['coincidence_lag_ms']:.0f} ms"
    )
    fig.suptitle(title, fontsize=10)
    return fig, axes


def _plot_hist_or_kde(ax, values, *, color, label, xlabel):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        ax.text(0.5, 0.5, 'No finite values', ha='center', va='center', transform=ax.transAxes)
        ax.set_xlabel(xlabel)
        return
    ax.hist(values, bins=min(20, max(5, int(np.sqrt(values.size)))), color=color, alpha=0.35, density=True, label=label)
    if values.size >= 3 and float(np.std(values, ddof=0)) > 0:
        grid = np.linspace(float(np.min(values)), float(np.max(values)), 200)
        if not np.isclose(grid[0], grid[-1]):
            try:
                density = gaussian_kde(values)(grid)
                ax.plot(grid, density, color=color, lw=1.5)
            except Exception:
                pass
    ax.set_xlabel(xlabel)
    ax.set_ylabel('density')


def prepare_experimental_moment_plot_data(experimental_moments_df, plot_level, level_colors=None):
    if experimental_moments_df is None or experimental_moments_df.empty:
        return None, [], {}
    plot_level = str(plot_level).strip().lower()
    if plot_level not in {'grand_average', 'condition', 'participant'}:
        raise ValueError("EXPERIMENTAL_MOMENT_PLOT_LEVEL must be 'grand_average', 'condition', or 'participant'")
    df = experimental_moments_df.copy()
    if plot_level == 'grand_average':
        df['plot_group'] = 'grand_average'
    elif plot_level == 'condition':
        df['plot_group'] = df['condition_id'].astype(str)
    else:
        df['plot_group'] = df['participant_id'].astype(str)
    group_labels = list(dict.fromkeys(df['plot_group'].dropna().astype(str).tolist()))
    level_colors = dict(level_colors or {})
    color_spec = level_colors.get(plot_level, {})
    if isinstance(color_spec, str):
        color_spec = {label: color_spec for label in group_labels}
    color_spec = {str(key): value for key, value in dict(color_spec or {}).items()}
    cmap = plt.get_cmap('tab20')
    group_colors = {}
    for idx, label in enumerate(group_labels):
        if label in color_spec:
            group_colors[label] = color_spec[label]
        else:
            group_colors[label] = tuple(float(x) for x in cmap(idx % cmap.N))
    return df, group_labels, group_colors


def _add_mean_sd_reference_lines(ax, values, *, color, mean_label=None, sd_label=None, mean_lw=3.0, sd_lw=1.7, alpha=1.0, zorder=5):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return []
    mean_value = float(np.mean(values))
    sd_value = float(np.std(values, ddof=0)) if values.size > 1 else np.nan
    handles = [
        ax.axvline(mean_value, color=color, lw=mean_lw, ls='-', alpha=alpha, zorder=zorder, label=mean_label)
    ]
    if np.isfinite(sd_value) and sd_value > 0:
        handles.append(ax.axvline(mean_value - sd_value, color=color, lw=sd_lw, ls='--', alpha=alpha * 0.9, zorder=zorder, label=sd_label))
        handles.append(ax.axvline(mean_value + sd_value, color=color, lw=sd_lw, ls='--', alpha=alpha * 0.9, zorder=zorder))
    return handles


def _add_experimental_moment_reference_lines(ax, exp_df, value_col, *, group_col='plot_group', color='black'):
    if exp_df is None or exp_df.empty or value_col not in exp_df.columns:
        return []
    handles = []
    groups = list(exp_df.groupby(group_col, sort=False))
    line_alpha = 0.95 if len(groups) <= 1 else 0.55
    for idx, (_group_label, group) in enumerate(groups):
        values = pd.to_numeric(group[value_col], errors='coerce').to_numpy(dtype=float)
        handles.extend(_add_mean_sd_reference_lines(
            ax,
            values,
            color=color,
            mean_label='experimental mean' if idx == 0 else None,
            sd_label='experimental mean +/- SD' if idx == 0 else None,
            mean_lw=3.0,
            sd_lw=1.7,
            alpha=line_alpha,
            zorder=6,
        ))
    return handles


def plot_isi_diagnostics(result, *, experimental_moments_df=None, experimental_group_colors=None, show_experimental_overlays=True, figsize=(12, 7), max_overlay_units=None):
    cfg = result['config']
    summary = result['spike_summary']
    isi_by_unit_ms = [np.asarray(isi, dtype=float) * 1000.0 for isi in summary['isi_by_unit_s']]
    if max_overlay_units is not None:
        isi_by_unit_ms = isi_by_unit_ms[: int(max_overlay_units)]

    fig, axes = plt.subplots(2, 2, figsize=figsize, constrained_layout=True)
    axes = axes.ravel()

    ax = axes[0]
    n_plotted = 0
    global_finite = np.concatenate([isi[np.isfinite(isi)] for isi in isi_by_unit_ms if np.asarray(isi).size]) if any(np.asarray(isi).size for isi in isi_by_unit_ms) else np.array([], dtype=float)
    if global_finite.size >= 3:
        x_low, x_high = np.nanpercentile(global_finite, [1, 99])
        if np.isclose(x_low, x_high):
            x_low, x_high = float(np.nanmin(global_finite)), float(np.nanmax(global_finite))
        grid = np.linspace(float(x_low), float(x_high), 240) if x_high > x_low else np.array([], dtype=float)
    else:
        grid = np.array([], dtype=float)
    for isi_ms in isi_by_unit_ms:
        isi_ms = isi_ms[np.isfinite(isi_ms)]
        if isi_ms.size < 3 or float(np.std(isi_ms, ddof=0)) <= 0 or grid.size == 0:
            continue
        try:
            ax.plot(grid, gaussian_kde(isi_ms)(grid), color='#852680', alpha=0.18, lw=0.9)
            n_plotted += 1
        except Exception:
            continue
    if n_plotted == 0:
        ax.text(0.5, 0.5, 'Not enough ISIs for KDE overlays', ha='center', va='center', transform=ax.transAxes)
    ax.set_title(f'Simulated per-neuron ISI KDE overlays (n={n_plotted})')
    ax.set_xlabel('ISI (ms)')
    ax.set_ylabel('density')

    mean_isi_color = '#008cff'
    sd_isi_color = '#ff6600'
    skew_color = '#333333'
    _plot_hist_or_kde(axes[1], summary['isi_means_s'] * 1000.0, color=mean_isi_color, label='mean ISI', xlabel='mean ISI per neuron (ms)')
    _plot_hist_or_kde(axes[2], summary['isi_sds_s'] * 1000.0, color=sd_isi_color, label='ISI SD', xlabel='ISI SD per neuron (ms)')
    _plot_hist_or_kde(axes[3], summary['isi_skews'], color=skew_color, label='ISI skew', xlabel='ISI skew per neuron')

    sim_mean_isi_ms = np.asarray(summary['isi_means_s'], dtype=float) * 1000.0
    sim_sd_isi_ms = np.asarray(summary['isi_sds_s'], dtype=float) * 1000.0
    _add_mean_sd_reference_lines(axes[1], sim_mean_isi_ms, color=mean_isi_color, mean_label='toy mean', sd_label='toy mean +/- SD')
    _add_mean_sd_reference_lines(axes[2], sim_sd_isi_ms, color=sd_isi_color, mean_label='toy mean', sd_label='toy mean +/- SD')
    _add_mean_sd_reference_lines(axes[3], summary['isi_skews'], color=skew_color, mean_label='toy mean', sd_label='toy mean +/- SD')

    if show_experimental_overlays and experimental_moments_df is not None:
        _add_experimental_moment_reference_lines(axes[1], experimental_moments_df, 'approx_mean_isi_ms', color='black')
        _add_experimental_moment_reference_lines(axes[2], experimental_moments_df, 'approx_isi_sd_ms', color='black')

    expected_cv = 1.0 / math.sqrt(float(cfg['private_gamma_shape_k']))
    actual_private_k_sd = float(np.std(result.get('private_gamma_shape_values', []), ddof=0)) if len(result.get('private_gamma_shape_values', [])) else np.nan
    annotation = (
        f"private gamma k={cfg['private_gamma_shape_k']:.2f} +/- {cfg.get('gamma_shape_k_sd', 0.0):.2f} -> nominal stationary CV~{expected_cv:.2f}\n"
        f"sampled private k SD={actual_private_k_sd:.2f}; common k={float(result.get('common_gamma_shape_value', np.nan)):.2f}\n"
        f"observed ISI CV={summary['isi_cv_mean']:.2f} +/- {summary['isi_cv_sd']:.2f}\n"
        f"observed ISI skew={summary['isi_skew_mean']:.2f} +/- {summary['isi_skew_sd']:.2f}"
    )
    axes[3].text(0.98, 0.98, annotation, ha='right', va='top', transform=axes[3].transAxes,
                 fontsize=8, bbox={'facecolor': 'white', 'edgecolor': '0.7', 'alpha': 0.9})
    for ax in axes:
        ax.grid(alpha=0.18)
    for ax in axes[1:]:
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            unique = dict(zip(labels, handles))
            ax.legend(unique.values(), unique.keys(), fontsize=7, loc='best')
    fig.suptitle('ISI distribution diagnostics', fontsize=12)
    return fig, axes

def plot_experimental_vs_toy_feature_moments(experimental_moments_df, toy_moments, *, experimental_group_colors=None, group_level='condition', figsize=(12, 7)):
    if experimental_moments_df is None or experimental_moments_df.empty:
        raise ValueError('experimental_moments_df is empty or None')
    experimental_group_colors = dict(experimental_group_colors or {})
    metrics = [
        ('firing_rate_mean', 'Firing-rate mean', 'Hz'),
        ('firing_rate_sd', 'Firing-rate SD', 'Hz'),
        ('firing_rate_skew', 'Firing-rate skew', ''),
        ('isi_cv_mean', 'ISI-CV mean', ''),
        ('isi_cv_sd', 'ISI-CV SD', ''),
        ('isi_cv_skew', 'ISI-CV skew', ''),
    ]
    fig, axes = plt.subplots(2, 3, figsize=figsize, constrained_layout=True)
    axes = axes.ravel()
    rng = np.random.default_rng(2026)
    group_labels = list(dict.fromkeys(experimental_moments_df['plot_group'].astype(str).tolist()))
    x_positions = {label: idx for idx, label in enumerate(group_labels)}
    for ax, (col, title, unit) in zip(axes, metrics):
        for group_label, group in experimental_moments_df.groupby('plot_group', sort=False):
            group_label = str(group_label)
            color = experimental_group_colors.get(group_label, '#666666')
            values = pd.to_numeric(group[col], errors='coerce').to_numpy(dtype=float)
            values = values[np.isfinite(values)]
            if values.size == 0:
                continue
            x0 = x_positions[group_label]
            jitter_width = 0.03 if len(group_labels) == 1 else 0.08
            jitter = rng.uniform(-jitter_width, jitter_width, values.size)
            ax.scatter(np.full(values.size, x0) + jitter, values, color=color, alpha=0.45, s=28, edgecolor='none')
            ax.scatter([x0], [float(np.mean(values))], color=color, marker='D', s=70, edgecolor='black', linewidth=0.6, zorder=4)
        toy_value = float(toy_moments.get(col, np.nan))
        if np.isfinite(toy_value):
            ax.axhline(toy_value, color='black', lw=2.0, label='toy simulation')
        ax.set_xticks(range(len(group_labels)))
        ax.set_xticklabels(group_labels, rotation=35, ha='right', fontsize=7 if len(group_labels) > 8 else 9)
        ylabel = title if not unit else f'{title}\n({unit})'
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(alpha=0.18)
        if np.isfinite(toy_value):
            ax.legend(fontsize=8, loc='best')
    fig.suptitle(f'Experimental {group_level} moments vs toy simulation', fontsize=12)
    return fig, axes


## One Simulation


In [ ]:
toy_result = run_toy_model()
experimental_moments_df = None
experimental_plot_moments_df = None
experimental_group_labels = []
experimental_group_colors = {}
if LOAD_EXPERIMENTAL_MOMENTS:
    experimental_moments_df = load_experimental_firing_isi_moments(EXPERIMENTAL_FEATURE_TABLE_PATH)
    experimental_plot_moments_df, experimental_group_labels, experimental_group_colors = prepare_experimental_moment_plot_data(
        experimental_moments_df,
        EXPERIMENTAL_MOMENT_PLOT_LEVEL,
        EXPERIMENTAL_MOMENT_LEVEL_COLORS,
    )
    display(Markdown(f'**Experimental firing/ISI moment summary by {EXPERIMENTAL_MOMENT_PLOT_LEVEL}**'))
    display(
        experimental_plot_moments_df
        .groupby('plot_group')[[
            'firing_rate_mean', 'firing_rate_sd',
            'isi_cv_mean', 'isi_cv_sd',
            'sync_mean', 'sync_baseline_max', 'sync_response_mean', 'sync_response_max',
            'approx_mean_isi_ms', 'approx_isi_sd_ms',
        ]]
        .agg(['count', 'mean', 'std'])
        .round(4)
    )

display(Markdown('**Single-run scalar summary**'))
display(summarize_toy_result(toy_result).to_frame())
_fig, _axes = plot_toy_summary(
    toy_result,
    show_components=show_input_components,
    xlim_s=summary_xlim_s,
    experimental_moments_df=experimental_plot_moments_df,
)
plt.show()
_isi_fig, _isi_axes = plot_isi_diagnostics(
    toy_result,
    experimental_moments_df=experimental_plot_moments_df,
    experimental_group_colors=experimental_group_colors,
    show_experimental_overlays=SHOW_EXPERIMENTAL_ISI_OVERLAYS,
)
plt.show()
if SAVE_FIGURES:
    FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    _fig.savefig(FIGURE_OUTPUT_DIR / 'toy_direct_synchrony_summary.png', dpi=160, bbox_inches='tight')
    _isi_fig.savefig(FIGURE_OUTPUT_DIR / 'toy_direct_synchrony_isi_diagnostics.png', dpi=160, bbox_inches='tight')


## Validation Checks

These checks are intentionally simple. They are meant to catch major sign, causality, or synchrony-control mistakes rather than prove statistical properties exactly.


In [ ]:
def validation_metric(overrides):
    base = {
        'duration_s': 8.0,
        'fs_hz': 1000.0,
        'n_units': 28,
        'baseline_rate_hz': 18.0,
        'unit_mean_rate_sd_hz': 0.0,
        'input_components': [],
        'sync_window_ms': 160.0,
        'sync_step_ms': 20.0,
        'coincidence_lag_ms': 6.0,
        'cst_smoothing_width_ms': 80.0,
        'common_copy_probability': 1.0,
        'private_gamma_shape_k': 1.0,
        'common_gamma_shape_k': 1.0,
        'gamma_shape_k_sd': 0.0,
    }
    base.update(overrides)
    result = run_toy_model(base)
    summary = summarize_toy_result(result)
    return result, summary


if RUN_VALIDATION_CHECKS:
    _, independent_poisson = validation_metric({
        'random_seed': 1,
        'common_fraction': 0.0,
        'common_jitter_sd_ms': 0.0,
        'private_gamma_shape_k': 1.0,
    })
    _, private_k1 = validation_metric({
        'random_seed': 2,
        'common_fraction': 0.0,
        'private_gamma_shape_k': 1.0,
        'duration_s': 20.0,
        'n_units': 40,
    })
    _, private_k4 = validation_metric({
        'random_seed': 2,
        'common_fraction': 0.0,
        'private_gamma_shape_k': 4.0,
        'duration_s': 20.0,
        'n_units': 40,
    })
    _, low_common = validation_metric({
        'random_seed': 3,
        'common_fraction': 0.0,
        'common_jitter_sd_ms': 0.0,
    })
    _, high_common = validation_metric({
        'random_seed': 3,
        'common_fraction': 0.6,
        'common_jitter_sd_ms': 0.0,
    })
    _, low_jitter = validation_metric({
        'random_seed': 4,
        'common_fraction': 0.5,
        'common_jitter_sd_ms': 0.0,
    })
    _, high_jitter = validation_metric({
        'random_seed': 4,
        'common_fraction': 0.5,
        'common_jitter_sd_ms': 20.0,
    })
    sparse_result, _ = validation_metric({
        'random_seed': 5,
        'duration_s': 2.0,
        'baseline_rate_hz': 2.5,
        'n_units': 16,
        'common_fraction': 0.0,
    })
    try:
        sparse_fig, _ = plot_isi_diagnostics(sparse_result)
        plt.close(sparse_fig)
        isi_plot_ok = True
    except Exception:
        isi_plot_ok = False

    expected_cv_k1 = 1.0
    expected_cv_k4 = 0.5
    validation_df = pd.DataFrame([
        {'check': 'private k=1 and common_fraction=0 gives Poisson-like CV', 'value': independent_poisson['isi_cv_mean'], 'target': expected_cv_k1, 'passed': abs(independent_poisson['isi_cv_mean'] - expected_cv_k1) < 0.35},
        {'check': 'increasing private k reduces ISI CV', 'value': private_k1['isi_cv_mean'] - private_k4['isi_cv_mean'], 'target': np.nan, 'passed': private_k4['isi_cv_mean'] < private_k1['isi_cv_mean']},
        {'check': 'private k=4 stationary CV near 1/sqrt(k)', 'value': private_k4['isi_cv_mean'], 'target': expected_cv_k4, 'passed': abs(private_k4['isi_cv_mean'] - expected_cv_k4) < 0.20},
        {'check': 'increasing common_fraction increases synchrony', 'value': high_common['mean_sync_rate_controlled'] - low_common['mean_sync_rate_controlled'], 'target': np.nan, 'passed': high_common['mean_sync_rate_controlled'] > low_common['mean_sync_rate_controlled']},
        {'check': 'increasing jitter reduces synchrony', 'value': low_jitter['mean_sync_rate_controlled'] - high_jitter['mean_sync_rate_controlled'], 'target': np.nan, 'passed': low_jitter['mean_sync_rate_controlled'] > high_jitter['mean_sync_rate_controlled']},
        {'check': 'ISI diagnostic figure handles sparse spike trains', 'value': float(isi_plot_ok), 'target': 1.0, 'passed': bool(isi_plot_ok)},
    ])
    display(validation_df)
    if not bool(validation_df['passed'].all()):
        warnings.warn('At least one validation check failed. Inspect validation_df before interpreting results.')
else:
    display(Markdown('Validation checks disabled.'))


## Optional Lightweight Grid Exploration

This is deliberately small and in-memory. It is useful for seeing qualitative trends, not for production batch analysis.


In [ ]:
if RUN_GRID_EXPLORATION:
    grid_output_path = make_grid_output_path(GRID_OUTPUT_DIR, GRID_OUTPUT_RUN_NAME)
    current_grid_config_snapshot = collect_grid_config_snapshot()
    current_grid_config_hash = grid_config_hash(current_grid_config_snapshot)

    if GRID_LOAD_EXISTING_OUTPUT:
        loaded_grid_output = load_grid_outputs(grid_output_path)
        loaded_grid_config_snapshot = loaded_grid_output['config_snapshot']
        loaded_grid_config_hash = loaded_grid_output['config_hash']
        if GRID_STRICT_CONFIG_MATCH_ON_LOAD and loaded_grid_config_hash != current_grid_config_hash:
            raise ValueError(
                'Cached grid config does not match the current notebook config. '
                f'Cached hash={loaded_grid_config_hash}; current hash={current_grid_config_hash}. '
                'Set GRID_STRICT_CONFIG_MATCH_ON_LOAD=False to load the table anyway for restyling.'
            )
        grid_df = loaded_grid_output['grid_df']
        grid_config_snapshot = loaded_grid_config_snapshot
        grid_config_hash_value = loaded_grid_config_hash
        display(Markdown(f'Loaded cached grid output from `{grid_output_path}` with {len(grid_df)} rows.'))
        display(grid_df)
    else:
        if GRID_SAVE_OUTPUTS and grid_output_path.exists() and any(grid_output_path.iterdir()):
            if GRID_OVERWRITE_OUTPUT:
                shutil.rmtree(grid_output_path)
            else:
                raise FileExistsError(
                    f'Grid output folder already exists and is not empty: {grid_output_path}. '
                    'Set GRID_OVERWRITE_OUTPUT=True, change GRID_OUTPUT_RUN_NAME, or set GRID_LOAD_EXISTING_OUTPUT=True.'
                )

        grid_parameter_values = {name: list(values) for name, values in GRID_PARAMETER_VALUES.items()}
        if not grid_parameter_values:
            raise ValueError('GRID_PARAMETER_VALUES must contain at least one parameter')
        required_grid_parameters = {GRID_X_PARAMETER, GRID_Y_PARAMETER, GRID_CURVE_X_PARAMETER, GRID_CURVE_HUE_PARAMETER}
        missing_grid_parameters = sorted(name for name in required_grid_parameters if name not in grid_parameter_values)
        if missing_grid_parameters:
            raise ValueError(f'Grid parameter(s) missing from GRID_PARAMETER_VALUES: {missing_grid_parameters}')
        for name, values in grid_parameter_values.items():
            if len(values) == 0:
                raise ValueError(f'GRID_PARAMETER_VALUES[{name!r}] is empty')

        realization_seeds = _resolve_grid_realization_seeds_for_snapshot()
        grid_names = list(grid_parameter_values.keys())
        grid_jobs = []
        for grid_index, combo_values in enumerate(itertools.product(*(grid_parameter_values[name] for name in grid_names))):
            base_overrides = dict(zip(grid_names, combo_values))
            for realization_index, realization_seed in enumerate(realization_seeds):
                grid_jobs.append((int(grid_index), dict(base_overrides), int(realization_index), int(realization_seed)))
        total_grid_jobs = len(grid_jobs)
        rows = []
        if bool(GRID_SHOW_PROGRESS) and tqdm is not None:
            grid_job_iterator = tqdm(
                grid_jobs,
                total=total_grid_jobs,
                desc='Grid simulations + synchrony analysis',
                unit='sim',
                leave=True,
            )
            progress_display = None
        elif bool(GRID_SHOW_PROGRESS):
            progress_display = display(
                Markdown(f'Grid simulations + synchrony analysis: 0/{total_grid_jobs}'),
                display_id=True,
            )
            grid_job_iterator = grid_jobs
        else:
            progress_display = None
            grid_job_iterator = grid_jobs

        for job_counter, (grid_index, base_overrides, realization_index, realization_seed) in enumerate(grid_job_iterator, start=1):
            grid_overrides = dict(base_overrides)
            grid_overrides['random_seed'] = int(realization_seed)
            result = run_toy_model(grid_overrides)
            summary = summarize_toy_result(result).to_dict()
            summary.update(base_overrides)
            summary.update({
                'grid_index': int(grid_index),
                'realization_index': int(realization_index),
                'realization_seed': int(realization_seed),
                'random_seed': int(realization_seed),
            })
            rows.append(summary)
            if progress_display is not None:
                update_every = max(1, total_grid_jobs // 100)
                if job_counter == total_grid_jobs or job_counter % update_every == 0:
                    progress_display.update(Markdown(
                        f'Grid simulations + synchrony analysis: {job_counter}/{total_grid_jobs}'
                    ))
        grid_df = pd.DataFrame(rows)
        grid_config_snapshot = current_grid_config_snapshot
        grid_config_hash_value = current_grid_config_hash
        if GRID_SAVE_OUTPUTS:
            save_grid_outputs(
                grid_output_path,
                grid_df=grid_df,
                config_snapshot=grid_config_snapshot,
                manifest_updates={
                    'stage': 'grid_simulation_saved',
                    'loaded_existing_output': False,
                    'realization_count': int(len(realization_seeds)),
                    'grid_cell_count': int(len(grid_df.groupby(grid_names))),
                    'grid_job_count': int(total_grid_jobs),
                },
            )
            display(Markdown(f'Saved grid simulation output to `{grid_output_path}`.'))
        display(Markdown(f'Grid simulation table: {len(grid_df)} realization rows across {len(grid_df.groupby(grid_names))} grid cells.'))
        display(grid_df)
else:
    display(Markdown('Optional grid exploration disabled. Set `RUN_GRID_EXPLORATION = True` to run it.'))


In [ ]:
if RUN_GRID_EXPLORATION:
    if 'grid_df' not in globals() or grid_df is None or len(grid_df) == 0:
        raise RuntimeError('grid_df is missing or empty. Run the grid simulation cell first.')

    grid_output_path = make_grid_output_path(GRID_OUTPUT_DIR, GRID_OUTPUT_RUN_NAME)
    grid_save_figure_formats = [GRID_SAVE_FIGURE_FORMATS] if isinstance(GRID_SAVE_FIGURE_FORMATS, str) else list(GRID_SAVE_FIGURE_FORMATS)
    if not grid_save_figure_formats:
        raise ValueError('GRID_SAVE_FIGURE_FORMATS cannot be empty')
    if 'grid_config_snapshot' not in globals() or not grid_config_snapshot:
        grid_config_snapshot = collect_grid_config_snapshot()
    grid_config_hash_value = grid_config_hash(grid_config_snapshot)

    heatmap_specs = list(GRID_METRIC_SPECS)
    metric_columns = [spec[0] for spec in heatmap_specs]
    missing_metric_columns = [col for col in metric_columns if col not in grid_df.columns]
    if missing_metric_columns:
        raise KeyError(f'grid_df is missing metric column(s): {missing_metric_columns}')

    grid_names = [name for name in GRID_PARAMETER_VALUES.keys() if name in grid_df.columns]
    if not grid_names:
        raise RuntimeError('Could not infer grid parameter columns from GRID_PARAMETER_VALUES and grid_df.')

    def _sorted_grid_values(values):
        unique = list(pd.unique(values))
        try:
            return sorted(unique, key=float)
        except Exception:
            return sorted(unique, key=str)

    def _format_grid_label(value):
        try:
            return f'{float(value):g}'
        except Exception:
            return str(value)

    def _axis_positions(values):
        sorted_values = _sorted_grid_values(values)
        try:
            positions = np.asarray([float(value) for value in sorted_values], dtype=float)
            use_numeric_axis = True
        except Exception:
            positions = np.arange(len(sorted_values), dtype=float)
            use_numeric_axis = False
        return sorted_values, positions, use_numeric_axis

    def _fit_linear_r2(x, y):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        finite = np.isfinite(x) & np.isfinite(y)
        if int(np.sum(finite)) < 2:
            return np.nan, np.nan, np.nan, np.full_like(y, np.nan, dtype=float)
        xf = x[finite]
        yf = y[finite]
        if float(np.std(xf, ddof=0)) <= 0:
            return np.nan, np.nan, np.nan, np.full_like(y, np.nan, dtype=float)
        slope, intercept = np.polyfit(xf, yf, 1)
        yhat_f = slope * xf + intercept
        ss_res = float(np.sum((yf - yhat_f) ** 2))
        ss_tot = float(np.sum((yf - float(np.mean(yf))) ** 2))
        r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
        return float(slope), float(intercept), float(r2), slope * x + intercept

    def _build_grid_summary_table(raw_df, group_cols, metric_cols):
        rows = []
        for key, sub in raw_df.groupby(group_cols, dropna=False, sort=False):
            if not isinstance(key, tuple):
                key = (key,)
            row = dict(zip(group_cols, key))
            row['n_realizations'] = int(len(sub))
            if 'realization_seed' in sub.columns:
                row['realization_seeds'] = ','.join(str(int(seed)) for seed in sorted(pd.unique(sub['realization_seed'])))
            for metric_col in metric_cols:
                values = pd.to_numeric(sub[metric_col], errors='coerce').to_numpy(dtype=float)
                finite = values[np.isfinite(values)]
                row[f'n_finite_{metric_col}'] = int(finite.size)
                if finite.size:
                    mean_value = float(np.mean(finite))
                    sd_value = float(np.std(finite, ddof=0))
                    cov_value = sd_value / abs(mean_value) if abs(mean_value) > 1e-12 else np.nan
                else:
                    mean_value = np.nan
                    sd_value = np.nan
                    cov_value = np.nan
                row[f'mean_{metric_col}'] = mean_value
                row[f'sd_{metric_col}'] = sd_value
                row[f'cov_{metric_col}'] = cov_value
            rows.append(row)
        return pd.DataFrame(rows)

    def _get_experimental_sync_reference_df():
        if not bool(GRID_SHOW_EXPERIMENTAL_SYNC_REFERENCES):
            return None
        if 'experimental_moments_df' in globals() and experimental_moments_df is not None:
            return experimental_moments_df
        if bool(LOAD_EXPERIMENTAL_MOMENTS):
            try:
                return load_experimental_firing_isi_moments(EXPERIMENTAL_FEATURE_TABLE_PATH)
            except Exception as exc:
                warnings.warn(f'Could not load experimental sync references: {exc}')
        return None

    def _add_reference_band(ax, values, *, color, linestyle='-', label=None):
        values = pd.to_numeric(pd.Series(values), errors='coerce').to_numpy(dtype=float)
        values = values[np.isfinite(values)]
        if values.size == 0:
            return []
        mean_value = float(np.mean(values))
        sd_value = float(np.std(values, ddof=0)) if values.size > 1 else np.nan
        handles = []
        if np.isfinite(sd_value) and sd_value > 0:
            ax.axhspan(
                mean_value - sd_value,
                mean_value + sd_value,
                color=color,
                alpha=float(GRID_EXPERIMENTAL_REFERENCE_ALPHA),
                linewidth=0,
                zorder=0,
            )
        handles.append(ax.axhline(mean_value, color=color, lw=2.0, ls=linestyle, alpha=0.95, label=label, zorder=1))
        return handles

    def _add_experimental_sync_references(ax, exp_df):
        if exp_df is None or exp_df.empty:
            return [], []
        required = ['sync_baseline_mean', 'sync_baseline_max', 'sync_response_mean', 'sync_response_max']
        if not all(col in exp_df.columns for col in required):
            return [], []
        handles = []
        labels = []
        specs = [
            ('sync_baseline_mean', GRID_EXPERIMENTAL_BASELINE_SYNC_COLOR, '-', 'baseline mean exp'),
            ('sync_baseline_max', GRID_EXPERIMENTAL_BASELINE_SYNC_COLOR, '--', 'baseline max exp'),
            ('sync_response_mean', GRID_EXPERIMENTAL_RESPONSE_SYNC_COLOR, '-', '0-1s mean exp'),
            ('sync_response_max', GRID_EXPERIMENTAL_RESPONSE_SYNC_COLOR, '--', '0-1s max exp'),
        ]
        for col, color, linestyle, label in specs:
            new_handles = _add_reference_band(ax, exp_df[col], color=color, linestyle=linestyle, label=label)
            for handle in new_handles:
                handles.append(handle)
                labels.append(label)
        return handles, labels

    grid_summary_df = _build_grid_summary_table(grid_df, grid_names, metric_columns)
    display(Markdown('**Grid summary across realizations**'))
    display(grid_summary_df)

    x_values = _sorted_grid_values(grid_summary_df[GRID_X_PARAMETER])
    y_values = _sorted_grid_values(grid_summary_df[GRID_Y_PARAMETER])
    curve_x_values, curve_x_positions, curve_x_is_numeric = _axis_positions(grid_summary_df[GRID_CURVE_X_PARAMETER])
    curve_hue_values = _sorted_grid_values(grid_summary_df[GRID_CURVE_HUE_PARAMETER])
    n_heatmap_cells = len(x_values) * len(y_values)
    annotate_heatmaps = n_heatmap_cells <= int(GRID_HEATMAP_ANNOTATION_MAX_CELLS)
    exp_sync_df = _get_experimental_sync_reference_df()

    figure_specs = [
        ('mean', 'Mean across realizations'),
        ('sd', 'SD across realizations'),
        ('cov', 'CoV across realizations'),
    ]
    grid_figures = {}
    grid_figure_paths = {}
    grid_curve_fit_rows = []

    for stat_prefix, stat_label in figure_specs:
        value_columns = [f'{stat_prefix}_{metric_col}' for metric_col in metric_columns]
        missing_value_columns = [col for col in value_columns if col not in grid_summary_df.columns]
        if missing_value_columns:
            raise KeyError(f'grid_summary_df is missing value column(s): {missing_value_columns}')

        n_cols = len(heatmap_specs)
        fig, axes = plt.subplots(
            3,
            n_cols,
            figsize=(5.0 * n_cols, 9.2),
            constrained_layout=True,
            gridspec_kw={'height_ratios': [1.0, 1.0, 0.34]},
        )
        axes = np.asarray(axes)
        if axes.ndim == 1:
            axes = axes.reshape(3, 1)

        for metric_i, ((metric_col, label, cmap), value_col) in enumerate(zip(heatmap_specs, value_columns)):
            grouped_grid = (
                grid_summary_df
                .groupby([GRID_X_PARAMETER, GRID_Y_PARAMETER], as_index=False)[value_col]
                .mean()
            )
            ax = axes[0, metric_i]
            pivot = (
                grouped_grid
                .pivot(index=GRID_Y_PARAMETER, columns=GRID_X_PARAMETER, values=value_col)
                .reindex(index=y_values, columns=x_values)
            )
            values = pivot.to_numpy(dtype=float)
            finite_values = values[np.isfinite(values)]
            vmin = float(np.min(finite_values)) if finite_values.size else None
            vmax = float(np.max(finite_values)) if finite_values.size else None
            image = ax.imshow(values, origin='lower', aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)
            cmap_obj = plt.get_cmap(cmap)
            norm = image.norm
            ax.set_xticks(np.arange(len(x_values)))
            ax.set_xticklabels([_format_grid_label(value) for value in x_values], rotation=35, ha='right')
            ax.set_yticks(np.arange(len(y_values)))
            ax.set_yticklabels([_format_grid_label(value) for value in y_values])
            ax.set_xlabel(GRID_X_PARAMETER)
            ax.set_ylabel(GRID_Y_PARAMETER)
            ax.set_title(label)
            if annotate_heatmaps:
                for row_i, _y_value in enumerate(y_values):
                    for col_i, _x_value in enumerate(x_values):
                        value = values[row_i, col_i]
                        if np.isfinite(value):
                            rgba = cmap_obj(norm(value))
                            luminance = 0.2126 * rgba[0] + 0.7152 * rgba[1] + 0.0722 * rgba[2]
                            text_color = 'black' if luminance > 0.55 else 'white'
                            ax.text(col_i, row_i, f'{value:.2g}', ha='center', va='center', fontsize=8, color=text_color)
            fig.colorbar(image, ax=ax, shrink=0.82)

            ax_curve = axes[1, metric_i]
            legend_ax = axes[2, metric_i]
            legend_ax.axis('off')
            grouped_curves = (
                grid_summary_df
                .groupby([GRID_CURVE_X_PARAMETER, GRID_CURVE_HUE_PARAMETER], as_index=False)[value_col]
                .mean()
            )
            curve_cmap = plt.get_cmap(cmap)
            min_frac = float(np.clip(GRID_CURVE_COLORMAP_MIN_FRACTION, 0.0, 1.0))
            n_hues = max(len(curve_hue_values), 1)
            legend_handles = []
            legend_labels = []
            if stat_prefix == 'mean' and metric_col == 'mean_sync_rate_controlled':
                ref_handles, ref_labels = _add_experimental_sync_references(ax_curve, exp_sync_df)
                legend_handles.extend(ref_handles)
                legend_labels.extend(ref_labels)
            for hue_i, hue_value in enumerate(curve_hue_values):
                color_frac = min_frac + (1.0 - min_frac) * (hue_i / max(n_hues - 1, 1))
                color = curve_cmap(color_frac)
                sub = grouped_curves[grouped_curves[GRID_CURVE_HUE_PARAMETER] == hue_value]
                y_by_x = sub.set_index(GRID_CURVE_X_PARAMETER)[value_col]
                y_curve = np.asarray([y_by_x.get(x_value, np.nan) for x_value in curve_x_values], dtype=float)
                finite_mask = np.isfinite(y_curve)
                if not np.any(finite_mask):
                    continue
                slope, intercept, r2, fit_curve = _fit_linear_r2(curve_x_positions, y_curve)
                fit_label = 'nan' if not np.isfinite(r2) else f'{r2:.2f}'
                line_label = f'{GRID_CURVE_HUE_PARAMETER}={_format_grid_label(hue_value)} | R2={fit_label}'
                line, = ax_curve.plot(
                    curve_x_positions[finite_mask],
                    y_curve[finite_mask],
                    marker='o',
                    lw=1.8,
                    color=color,
                    label=line_label,
                )
                legend_handles.append(line)
                legend_labels.append(line_label)
                if np.any(np.isfinite(fit_curve)):
                    ax_curve.plot(
                        curve_x_positions[finite_mask],
                        fit_curve[finite_mask],
                        ls='--',
                        lw=1.0,
                        color=color,
                        alpha=0.55,
                    )
                grid_curve_fit_rows.append({
                    'figure_statistic': stat_prefix,
                    'metric': metric_col,
                    'metric_label': label,
                    'curve_x_parameter': GRID_CURVE_X_PARAMETER,
                    'curve_hue_parameter': GRID_CURVE_HUE_PARAMETER,
                    'curve_hue_value': hue_value,
                    'linear_slope': slope,
                    'linear_intercept': intercept,
                    'linear_r2': r2,
                    'n_points': int(np.sum(finite_mask)),
                })
            ax_curve.set_xlabel(GRID_CURVE_X_PARAMETER)
            ax_curve.set_ylabel(f'{stat_label}\n{label}')
            ax_curve.grid(alpha=0.18)
            if not curve_x_is_numeric:
                ax_curve.set_xticks(curve_x_positions)
                ax_curve.set_xticklabels([_format_grid_label(value) for value in curve_x_values], rotation=35, ha='right')
            if legend_handles:
                legend_ax.legend(
                    legend_handles,
                    legend_labels,
                    fontsize=7,
                    loc='center',
                    ncol=1 if len(legend_handles) <= 8 else 2,
                    frameon=False,
                )

        extra_heatmap_dims = [name for name in grid_names if name not in {GRID_X_PARAMETER, GRID_Y_PARAMETER}]
        extra_curve_dims = [name for name in grid_names if name not in {GRID_CURVE_X_PARAMETER, GRID_CURVE_HUE_PARAMETER}]
        heatmap_suffix = '' if not extra_heatmap_dims else f' | heatmaps averaged over: {", ".join(extra_heatmap_dims)}'
        curve_suffix = '' if not extra_curve_dims else f' | curves averaged over: {", ".join(extra_curve_dims)}'
        annotation_suffix = '' if annotate_heatmaps else f' | heatmap text hidden ({n_heatmap_cells} cells)'
        fig.suptitle(
            f'Toy synchrony grid - {stat_label} | heatmap x={GRID_X_PARAMETER}, y={GRID_Y_PARAMETER}{heatmap_suffix}{annotation_suffix}\n'
            f'Curves: y=metric, x={GRID_CURVE_X_PARAMETER}, color={GRID_CURVE_HUE_PARAMETER}{curve_suffix}',
            fontsize=11,
        )
        grid_figures[stat_prefix] = fig
        if GRID_SAVE_OUTPUTS:
            grid_output_path.mkdir(parents=True, exist_ok=True)
            saved_paths = []
            for figure_format in grid_save_figure_formats:
                figure_format = str(figure_format).lstrip('.').lower()
                figure_path = grid_output_path / f'grid_{stat_prefix}_across_realizations.{figure_format}'
                fig.savefig(figure_path, dpi=160, bbox_inches='tight')
                saved_paths.append(str(figure_path))
            grid_figure_paths[stat_prefix] = saved_paths
        plt.show()

    grid_curve_fit_df = pd.DataFrame(grid_curve_fit_rows)
    display(Markdown('**Grid curve linear-fit R2 values**'))
    display(grid_curve_fit_df)
    if GRID_SAVE_OUTPUTS:
        manifest = save_grid_outputs(
            grid_output_path,
            grid_summary_df=grid_summary_df,
            grid_curve_fit_df=grid_curve_fit_df,
            config_snapshot=grid_config_snapshot,
            manifest_updates={
                'stage': 'grid_figures_saved',
                'loaded_existing_output': bool(GRID_LOAD_EXISTING_OUTPUT),
                'figure_paths': grid_figure_paths,
                'figure_formats': grid_save_figure_formats,
                'grid_metric_columns': metric_columns,
                'grid_statistics': [spec[0] for spec in figure_specs],
            },
        )
        display(Markdown(f'Saved grid summary tables, fit table, figures, and manifest to `{grid_output_path}`.'))
else:
    display(Markdown('Optional grid exploration disabled. Set `RUN_GRID_EXPLORATION = True` to run it.'))


## Trace-Level Batch Exploration

This section runs or reloads compact trace batches over selected parameter values. It differs from the scalar grid above because it preserves time series for every realization, allowing mean-over-realizations trace figures with individual realizations shown underneath. Unspecified parameters inherit the editable configuration at the top of the notebook.

Relationship plots can use all realization rows directly, or summarize each x-value by mean, SD, min, and max across realizations.


In [ ]:

def _resolve_trace_batch_realization_seeds():
    if TRACE_BATCH_REALIZATION_SEEDS is not None:
        seeds = [int(seed) for seed in TRACE_BATCH_REALIZATION_SEEDS]
    else:
        seeds = [int(TRACE_BATCH_BASE_RANDOM_SEED) + idx for idx in range(int(TRACE_BATCH_N_REALIZATIONS))]
    if not seeds:
        raise ValueError('Trace batch requires at least one realization seed')
    return seeds


def make_trace_batch_output_path(output_dir=None, run_name=None):
    output_dir = Path(TRACE_BATCH_OUTPUT_DIR if output_dir is None else output_dir)
    run_name = str(TRACE_BATCH_RUN_NAME if run_name is None else run_name)
    if not run_name:
        raise ValueError('TRACE_BATCH_RUN_NAME cannot be empty')
    return output_dir / run_name


def _current_toy_base_config_snapshot():
    return {
        'duration_s': duration_s,
        'ignored_initial_duration_s': ignored_initial_duration_s,
        'fs_hz': fs_hz,
        'n_units': n_units,
        'baseline_rate_hz': baseline_rate_hz,
        'unit_mean_rate_sd_hz': unit_mean_rate_sd_hz,
        'rate_floor_hz': rate_floor_hz,
        'input_components': input_components,
        'enable_pulse_input': enable_pulse_input,
        'pulse_input_start_rel_analysis_s': pulse_input_start_rel_analysis_s,
        'pulse_input_duration_s': pulse_input_duration_s,
        'pulse_input_amplitude_hz': pulse_input_amplitude_hz,
        'pulse_metric_pre_window_rel_s': pulse_metric_pre_window_rel_s,
        'pulse_metric_post_window_rel_s': pulse_metric_post_window_rel_s,
        'common_fraction': common_fraction,
        'common_jitter_sd_ms': common_jitter_sd_ms,
        'common_copy_probability': common_copy_probability,
        'private_gamma_shape_k': private_gamma_shape_k,
        'common_gamma_shape_k': common_gamma_shape_k,
        'gamma_shape_k_sd': gamma_shape_k_sd,
        'cst_smoothing_kernel': cst_smoothing_kernel,
        'cst_smoothing_width_ms': cst_smoothing_width_ms,
        'sync_window_ms': sync_window_ms,
        'sync_step_ms': sync_step_ms,
        'coincidence_lag_ms': coincidence_lag_ms,
    }


def collect_trace_batch_config_snapshot():
    return _json_safe({
        'toy_base_config': _current_toy_base_config_snapshot(),
        'trace_batch_parameter_values': TRACE_BATCH_PARAMETER_VALUES,
        'trace_batch_realization_seeds': _resolve_trace_batch_realization_seeds(),
        'trace_batch_base_random_seed': TRACE_BATCH_BASE_RANDOM_SEED,
        'trace_batch_n_realizations': TRACE_BATCH_N_REALIZATIONS,
    })


def trace_batch_config_hash(config_snapshot):
    payload = json.dumps(_json_safe(config_snapshot), sort_keys=True, separators=(',', ':'))
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()[:16]


def build_trace_batch_parameter_sets(parameter_values=None):
    parameter_values = TRACE_BATCH_PARAMETER_VALUES if parameter_values is None else parameter_values
    parameter_values = {str(name): list(values) for name, values in dict(parameter_values).items()}
    for name, values in parameter_values.items():
        if len(values) == 0:
            raise ValueError(f'TRACE_BATCH_PARAMETER_VALUES[{name!r}] is empty')
    names = list(parameter_values.keys())
    if not names:
        return pd.DataFrame([{'parameter_set_id': 0}]), []
    rows = []
    for parameter_set_id, combo_values in enumerate(itertools.product(*(parameter_values[name] for name in names))):
        row = {'parameter_set_id': int(parameter_set_id)}
        row.update(dict(zip(names, combo_values)))
        rows.append(row)
    return pd.DataFrame(rows), names


def _extract_compact_trace_result(result):
    cfg = result['config']
    t = np.asarray(result['time_s'], dtype=float)
    sync = result['sync']
    sync_t = np.asarray(sync['time_s'], dtype=float)
    dt = 1.0 / float(cfg['fs_hz'])
    unit_rate_traces = np.asarray(result.get('unit_rate_traces_hz', result['rate_hz']), dtype=float)
    if unit_rate_traces.ndim == 2:
        mean_rate_hz = np.nanmean(unit_rate_traces, axis=0)
    else:
        mean_rate_hz = np.asarray(result['rate_hz'], dtype=float)
    cst_raw = np.asarray(result['cst_spikes_per_s_mu'], dtype=float)
    integrated_input = np.cumsum(mean_rate_hz) * dt
    integrated_cst = np.cumsum(cst_raw) * dt
    cumulative_residual = integrated_cst - integrated_input
    return {
        'time_s': t,
        'sync_time_s': sync_t,
        'rate_hz': np.asarray(result['rate_hz'], dtype=float),
        'raw_rate_hz': np.asarray(result['raw_rate_hz'], dtype=float),
        'pulse_trace_hz': np.asarray(result.get('pulse_trace_hz', np.zeros_like(mean_rate_hz)), dtype=float),
        'mean_unit_rate_hz': mean_rate_hz,
        'cst_spikes_per_s_mu': cst_raw,
        'cst_smooth_spikes_per_s_mu': np.asarray(result['cst_smooth_spikes_per_s_mu'], dtype=float),
        'integrated_input_expected_spikes_per_neuron': integrated_input,
        'integrated_cst_spikes_per_mu': integrated_cst,
        'cumulative_cst_minus_input_residual_spikes_per_mu': cumulative_residual,
        'sync_rate_controlled': np.asarray(sync['sync_rate_controlled'], dtype=float),
        'sync_absolute_coincidence_rate': np.asarray(sync['sync_absolute_coincidence_rate'], dtype=float),
    }


def _interp_to_reference(source_x, source_y, reference_x):
    source_x = np.asarray(source_x, dtype=float)
    source_y = np.asarray(source_y, dtype=float)
    reference_x = np.asarray(reference_x, dtype=float)
    if source_x.shape == reference_x.shape and np.allclose(source_x, reference_x, equal_nan=True):
        return source_y.astype(float, copy=False)
    finite = np.isfinite(source_x) & np.isfinite(source_y)
    if int(np.sum(finite)) < 2:
        return np.full_like(reference_x, np.nan, dtype=float)
    return np.interp(reference_x, source_x[finite], source_y[finite], left=np.nan, right=np.nan)


def _stack_compact_traces(compact_results, realization_rows):
    if not compact_results:
        raise ValueError('No compact trace results to stack')
    ref_time = np.asarray(compact_results[0]['time_s'], dtype=float)
    ref_sync_time = np.asarray(compact_results[0]['sync_time_s'], dtype=float)
    trace_keys = [
        'rate_hz',
        'raw_rate_hz',
        'pulse_trace_hz',
        'mean_unit_rate_hz',
        'cst_spikes_per_s_mu',
        'cst_smooth_spikes_per_s_mu',
        'integrated_input_expected_spikes_per_neuron',
        'integrated_cst_spikes_per_mu',
        'cumulative_cst_minus_input_residual_spikes_per_mu',
    ]
    sync_keys = ['sync_rate_controlled', 'sync_absolute_coincidence_rate']
    scalar_keys = [
        'baseline_rate_hz',
        'pulse_input_enabled',
        'pulse_input_amplitude_hz',
        'pulse_input_start_rel_analysis_s',
        'pulse_input_start_abs_s',
        'pulse_input_end_abs_s',
        'pulse_input_duration_s',
        'pulse_input_duration_effective_s',
        'pulse_input_area_expected_spikes_per_neuron',
        'pulse_metric_pre_window_start_abs_s',
        'pulse_metric_pre_window_end_abs_s',
        'pulse_metric_post_window_start_abs_s',
        'pulse_metric_post_window_end_abs_s',
        'pool_age_censored_fraction',
        'pool_integrated_age_mean',
        'pool_integrated_age_sd',
        'pool_integrated_age_min',
        'pool_integrated_age_max',
        'pool_integrated_age_skew',
        'pool_integrated_age_median',
        'pool_integrated_age_iqr',
        'pool_raw_age_s_mean',
        'pool_raw_age_s_sd',
        'pool_raw_age_s_min',
        'pool_raw_age_s_max',
        'pool_raw_age_s_skew',
        'pool_raw_age_s_median',
        'pool_raw_age_s_iqr',
        'pool_age_metric_source',
    ]
    stacked = {
        'time_s': ref_time,
        'sync_time_s': ref_sync_time,
        'realization_index': np.asarray([row['realization_index'] for row in realization_rows], dtype=int),
        'realization_seed': np.asarray([row['realization_seed'] for row in realization_rows], dtype=int),
    }
    for key in scalar_keys:
        if key in realization_rows[0]:
            stacked[key] = np.asarray([row.get(key, np.nan) for row in realization_rows])
    for key in trace_keys:
        stacked[key] = np.vstack([
            _interp_to_reference(compact['time_s'], compact[key], ref_time) for compact in compact_results
        ])
    for key in sync_keys:
        stacked[key] = np.vstack([
            _interp_to_reference(compact['sync_time_s'], compact[key], ref_sync_time) for compact in compact_results
        ])
    return stacked


def save_trace_batch_parameter_set_npz(output_path, parameter_set_id, stacked_traces, parameter_values):
    ps_dir = Path(output_path) / f'parameter_set_{int(parameter_set_id):04d}'
    ps_dir.mkdir(parents=True, exist_ok=True)
    npz_path = ps_dir / f'trace_batch_parameter_set_{int(parameter_set_id):04d}.npz'
    np.savez_compressed(
        npz_path,
        **stacked_traces,
        parameter_set_id=np.asarray(int(parameter_set_id), dtype=int),
        parameter_values_json=np.asarray(json.dumps(_json_safe(parameter_values), sort_keys=True)),
    )
    return npz_path


def _summarize_parameter_set_rows(realization_df, parameter_columns):
    rows = []
    for parameter_set_id, sub in realization_df.groupby('parameter_set_id', sort=True):
        row = {'parameter_set_id': int(parameter_set_id), 'n_realizations': int(len(sub))}
        for col in parameter_columns:
            if col in sub.columns:
                vals = pd.unique(sub[col])
                row[col] = vals[0] if len(vals) else np.nan
        for col in sub.columns:
            if col in {'parameter_set_id', 'realization_index', 'realization_seed', 'random_seed'} or col in parameter_columns:
                continue
            values = pd.to_numeric(sub[col], errors='coerce')
            if values.notna().sum() == 0:
                continue
            finite = values.to_numpy(dtype=float)
            finite = finite[np.isfinite(finite)]
            if finite.size == 0:
                continue
            row[f'mean_{col}'] = float(np.mean(finite))
            row[f'sd_{col}'] = float(np.std(finite, ddof=0)) if finite.size > 1 else 0.0
            row[f'min_{col}'] = float(np.min(finite))
            row[f'max_{col}'] = float(np.max(finite))
            row[f'n_finite_{col}'] = int(finite.size)
        rows.append(row)
    return pd.DataFrame(rows)


def save_trace_batch_outputs(output_path, *, realization_df=None, parameter_set_df=None, config_snapshot=None, manifest_updates=None):
    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)
    table_paths = {}
    for name, df in [('trace_batch_realization_table', realization_df), ('trace_batch_parameter_set_table', parameter_set_df)]:
        if df is None:
            continue
        pkl_path = output_path / f'{name}.pkl'
        csv_path = output_path / f'{name}.csv'
        df.to_pickle(pkl_path)
        df.to_csv(csv_path, index=False)
        table_paths[name] = {'pkl': str(pkl_path), 'csv': str(csv_path)}
    if config_snapshot is not None:
        config_path = output_path / 'trace_batch_config.json'
        config_hash = trace_batch_config_hash(config_snapshot)
        config_path.write_text(json.dumps({'config_hash': config_hash, 'config': _json_safe(config_snapshot)}, indent=2), encoding='utf-8')
    else:
        config_hash = None
    manifest = {
        'updated_at_utc': datetime.now(timezone.utc).isoformat(),
        'config_hash': config_hash,
        'tables': table_paths,
    }
    if realization_df is not None:
        manifest['realization_rows'] = int(len(realization_df))
    if parameter_set_df is not None:
        manifest['parameter_set_rows'] = int(len(parameter_set_df))
    if manifest_updates:
        manifest.update(_json_safe(manifest_updates))
    manifest_path = output_path / 'trace_batch_manifest.json'
    manifest_path.write_text(json.dumps(_json_safe(manifest), indent=2), encoding='utf-8')
    return manifest


def load_trace_batch_outputs(output_path):
    output_path = Path(output_path)
    config_path = output_path / 'trace_batch_config.json'
    realization_path = output_path / 'trace_batch_realization_table.pkl'
    parameter_set_path = output_path / 'trace_batch_parameter_set_table.pkl'
    if not config_path.exists():
        raise FileNotFoundError(f'Trace batch config not found: {config_path}')
    if not realization_path.exists():
        raise FileNotFoundError(f'Trace batch realization table not found: {realization_path}')
    payload = json.loads(config_path.read_text(encoding='utf-8'))
    return {
        'config_snapshot': payload.get('config', {}),
        'config_hash': payload.get('config_hash'),
        'realization_df': pd.read_pickle(realization_path),
        'parameter_set_df': pd.read_pickle(parameter_set_path) if parameter_set_path.exists() else None,
        'output_path': output_path,
    }


def load_trace_batch_parameter_set_npz(output_path, parameter_set_id):
    npz_path = Path(output_path) / f'parameter_set_{int(parameter_set_id):04d}' / f'trace_batch_parameter_set_{int(parameter_set_id):04d}.npz'
    if not npz_path.exists():
        raise FileNotFoundError(f'Trace batch compact trace file not found: {npz_path}')
    with np.load(npz_path, allow_pickle=False) as data:
        return {key: data[key] for key in data.files}, npz_path


In [ ]:

if RUN_TRACE_BATCH:
    trace_batch_output_path = make_trace_batch_output_path(TRACE_BATCH_OUTPUT_DIR, TRACE_BATCH_RUN_NAME)
    current_trace_batch_config_snapshot = collect_trace_batch_config_snapshot()
    current_trace_batch_config_hash = trace_batch_config_hash(current_trace_batch_config_snapshot)

    if TRACE_BATCH_LOAD_EXISTING_OUTPUT:
        loaded_trace_batch_output = load_trace_batch_outputs(trace_batch_output_path)
        loaded_trace_batch_config_hash = loaded_trace_batch_output['config_hash']
        if TRACE_BATCH_STRICT_CONFIG_MATCH_ON_LOAD and loaded_trace_batch_config_hash != current_trace_batch_config_hash:
            raise ValueError(
                'Cached trace-batch config does not match the current notebook config. '
                f'Cached hash={loaded_trace_batch_config_hash}; current hash={current_trace_batch_config_hash}. '
                'Set TRACE_BATCH_STRICT_CONFIG_MATCH_ON_LOAD=False to load anyway for restyling.'
            )
        trace_batch_realization_df = loaded_trace_batch_output['realization_df']
        trace_batch_parameter_set_df = loaded_trace_batch_output['parameter_set_df']
        trace_batch_config_snapshot = loaded_trace_batch_output['config_snapshot']
        trace_batch_config_hash_value = loaded_trace_batch_config_hash
        display(Markdown(f'Loaded cached trace batch from `{trace_batch_output_path}` with {len(trace_batch_realization_df)} realization rows.'))
        display(trace_batch_realization_df)
    else:
        if TRACE_BATCH_SAVE_OUTPUTS and trace_batch_output_path.exists() and any(trace_batch_output_path.iterdir()):
            if TRACE_BATCH_OVERWRITE_OUTPUT:
                shutil.rmtree(trace_batch_output_path)
            else:
                raise FileExistsError(
                    f'Trace-batch output folder already exists and is not empty: {trace_batch_output_path}. '
                    'Set TRACE_BATCH_OVERWRITE_OUTPUT=True, change TRACE_BATCH_RUN_NAME, or set TRACE_BATCH_LOAD_EXISTING_OUTPUT=True.'
                )
        trace_batch_parameter_sets_df, trace_batch_parameter_names = build_trace_batch_parameter_sets(TRACE_BATCH_PARAMETER_VALUES)
        trace_batch_realization_seeds = _resolve_trace_batch_realization_seeds()
        total_trace_jobs = int(len(trace_batch_parameter_sets_df) * len(trace_batch_realization_seeds))
        rows = []
        trace_npz_paths = []

        if bool(TRACE_BATCH_SHOW_PROGRESS) and tqdm is not None:
            trace_progress = tqdm(total=total_trace_jobs, desc='Trace batch simulations + synchrony analysis', unit='sim', leave=True)
            progress_display = None
        elif bool(TRACE_BATCH_SHOW_PROGRESS):
            trace_progress = None
            progress_display = display(Markdown(f'Trace batch simulations + synchrony analysis: 0/{total_trace_jobs}'), display_id=True)
        else:
            trace_progress = None
            progress_display = None

        completed_jobs = 0
        try:
            for _, ps_row in trace_batch_parameter_sets_df.iterrows():
                parameter_set_id = int(ps_row['parameter_set_id'])
                parameter_values = {name: ps_row[name] for name in trace_batch_parameter_names}
                compact_results = []
                realization_rows = []
                for realization_index, realization_seed in enumerate(trace_batch_realization_seeds):
                    overrides = dict(parameter_values)
                    overrides['random_seed'] = int(realization_seed)
                    result = run_toy_model(overrides)
                    summary = summarize_toy_result(result).to_dict()
                    row = dict(parameter_values)
                    row.update(summary)
                    row.update({
                        'parameter_set_id': int(parameter_set_id),
                        'realization_index': int(realization_index),
                        'realization_seed': int(realization_seed),
                        'random_seed': int(realization_seed),
                    })
                    rows.append(row)
                    realization_rows.append(row)
                    compact_results.append(_extract_compact_trace_result(result))
                    completed_jobs += 1
                    if trace_progress is not None:
                        trace_progress.update(1)
                    elif progress_display is not None:
                        update_every = max(1, total_trace_jobs // 100)
                        if completed_jobs == total_trace_jobs or completed_jobs % update_every == 0:
                            progress_display.update(Markdown(
                                f'Trace batch simulations + synchrony analysis: {completed_jobs}/{total_trace_jobs}'
                            ))
                if TRACE_BATCH_SAVE_OUTPUTS:
                    stacked = _stack_compact_traces(compact_results, realization_rows)
                    npz_path = save_trace_batch_parameter_set_npz(trace_batch_output_path, parameter_set_id, stacked, parameter_values)
                    trace_npz_paths.append(str(npz_path))
        finally:
            if trace_progress is not None:
                trace_progress.close()

        trace_batch_realization_df = pd.DataFrame(rows)
        trace_batch_parameter_set_df = _summarize_parameter_set_rows(trace_batch_realization_df, trace_batch_parameter_names)
        trace_batch_config_snapshot = current_trace_batch_config_snapshot
        trace_batch_config_hash_value = current_trace_batch_config_hash
        if TRACE_BATCH_SAVE_OUTPUTS:
            save_trace_batch_outputs(
                trace_batch_output_path,
                realization_df=trace_batch_realization_df,
                parameter_set_df=trace_batch_parameter_set_df,
                config_snapshot=trace_batch_config_snapshot,
                manifest_updates={
                    'stage': 'trace_batch_simulation_saved',
                    'loaded_existing_output': False,
                    'trace_npz_paths': trace_npz_paths,
                    'parameter_set_count': int(len(trace_batch_parameter_sets_df)),
                    'realization_count': int(len(trace_batch_realization_seeds)),
                },
            )
        display(Markdown(
            f'Trace batch table: {len(trace_batch_realization_df)} realization rows across '
            f'{trace_batch_realization_df["parameter_set_id"].nunique()} parameter sets.'
        ))
        display(trace_batch_realization_df)
else:
    display(Markdown('Trace batch exploration is disabled. Set `RUN_TRACE_BATCH=True` to run or load it.'))


In [ ]:

def _finite_mean_trace(values):
    values = np.asarray(values, dtype=float)
    with np.errstate(invalid='ignore'):
        return np.nanmean(values, axis=0)


def _plot_trace_stack(ax, x, traces, *, color, mean_label, individual_label=None, alpha=0.18, mean_lw=2.3, zero_line=False):
    traces = np.asarray(traces, dtype=float)
    first_label = individual_label
    for row in traces:
        ax.plot(x, row, color=color, alpha=alpha, lw=0.8, label=first_label)
        first_label = None
    mean_trace = _finite_mean_trace(traces)
    ax.plot(x, mean_trace, color=color, lw=mean_lw, label=mean_label, zorder=5)
    if zero_line:
        ax.axhline(0, color='black', ls='--', lw=0.8, alpha=0.55)
    return mean_trace


def _trace_scalar_first(trace_data, key, default=np.nan):
    if key not in trace_data:
        return default
    arr = np.asarray(trace_data[key])
    if arr.size == 0:
        return default
    value = arr.flat[0]
    try:
        return float(value)
    except Exception:
        return value


def _get_pulse_window_from_trace_data(trace_data):
    start_s = _trace_scalar_first(trace_data, 'pulse_input_start_abs_s', np.nan)
    end_s = _trace_scalar_first(trace_data, 'pulse_input_end_abs_s', np.nan)
    amp_hz = _trace_scalar_first(trace_data, 'pulse_input_amplitude_hz', np.nan)
    enabled = bool(_trace_scalar_first(trace_data, 'pulse_input_enabled', False))
    if not enabled or not np.isfinite(start_s) or not np.isfinite(end_s) or end_s <= start_s:
        return None
    return float(start_s), float(end_s), float(amp_hz)


def _add_pulse_window_from_trace_data(ax, trace_data, *, label=False):
    pulse_window = _get_pulse_window_from_trace_data(trace_data)
    if pulse_window is None:
        return
    start_s, end_s, amp_hz = pulse_window
    ax.axvspan(start_s, end_s, color='#2ca02c', alpha=0.08, lw=0, label=f'pulse window ({amp_hz:g} Hz)' if label else None)
    ax.axvline(start_s, color='#2ca02c', lw=1.0, ls='--', alpha=0.8)
    ax.axvline(end_s, color='#2ca02c', lw=1.0, ls=':', alpha=0.7)


def _get_cumulative_residual_trace(trace_data):
    if 'cumulative_cst_minus_input_residual_spikes_per_mu' in trace_data:
        return np.asarray(trace_data['cumulative_cst_minus_input_residual_spikes_per_mu'], dtype=float)
    return (
        np.asarray(trace_data['integrated_cst_spikes_per_mu'], dtype=float)
        - np.asarray(trace_data['integrated_input_expected_spikes_per_neuron'], dtype=float)
    )


def _mean_kde_distribution(values_by_realization, *, n_grid=260):
    arrays = []
    for values in np.asarray(values_by_realization, dtype=float):
        finite = values[np.isfinite(values)]
        if finite.size:
            arrays.append(finite)
    if not arrays:
        return None, None, [], np.nan, np.nan
    all_values = np.concatenate(arrays)
    if all_values.size == 0:
        return None, None, [], np.nan, np.nan
    lo = float(np.min(all_values))
    hi = float(np.max(all_values))
    if np.isclose(lo, hi):
        pad = max(1e-6, abs(lo) * 0.05 + 1e-6)
        lo -= pad
        hi += pad
    grid = np.linspace(lo, hi, int(n_grid))
    densities = []
    for finite in arrays:
        density = np.full_like(grid, np.nan, dtype=float)
        if finite.size >= 3 and float(np.std(finite, ddof=0)) > 0:
            try:
                density = gaussian_kde(finite)(grid)
            except Exception:
                pass
        if not np.all(np.isfinite(density)):
            hist, edges = np.histogram(finite, bins=min(24, max(4, int(np.sqrt(finite.size)))), density=True)
            centers = 0.5 * (edges[:-1] + edges[1:])
            if centers.size >= 2:
                density = np.interp(grid, centers, hist, left=0.0, right=0.0)
            else:
                density = np.zeros_like(grid)
        densities.append(density)
    density_stack = np.vstack(densities)
    mean_density = np.nanmean(density_stack, axis=0)
    area = float(np.trapz(mean_density, grid))
    if np.isfinite(area) and area > 0:
        mean_density = mean_density / area
    denom = float(np.trapz(mean_density, grid))
    if np.isfinite(denom) and denom > 0:
        mean_value = float(np.trapz(grid * mean_density, grid) / denom)
        sd_value = float(np.sqrt(np.trapz((grid - mean_value) ** 2 * mean_density, grid) / denom))
    else:
        mean_value = np.nan
        sd_value = np.nan
    return grid, mean_density, densities, mean_value, sd_value


def _plot_mean_kde_panel(ax, values_by_realization, *, color, title, xlabel, alpha=0.16):
    grid, mean_density, densities, mean_value, sd_value = _mean_kde_distribution(values_by_realization)
    if grid is None:
        ax.text(0.5, 0.5, 'No finite values', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        ax.set_xlabel(xlabel)
        return np.nan, np.nan
    first_label = 'realization KDE'
    for density in densities:
        ax.plot(grid, density, color=color, alpha=alpha, lw=0.8, label=first_label)
        first_label = None
    ax.plot(grid, mean_density, color=color, lw=2.2, label='mean KDE', zorder=5)
    ax.axvline(mean_value, color=color, lw=2.0, label='mean KDE mean')
    if np.isfinite(sd_value) and sd_value > 0:
        ax.axvline(mean_value - sd_value, color=color, ls='--', lw=1.1, alpha=0.8)
        ax.axvline(mean_value + sd_value, color=color, ls='--', lw=1.1, alpha=0.8)
    ax.text(
        0.98,
        0.96,
        f'mean={mean_value:.3g}\nSD={sd_value:.3g}',
        ha='right',
        va='top',
        transform=ax.transAxes,
        fontsize=8,
        bbox={'facecolor': 'white', 'edgecolor': '0.75', 'alpha': 0.9},
    )
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('density')
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        unique = dict(zip(labels, handles))
        ax.legend(unique.values(), unique.keys(), fontsize=7, loc='best')
    return mean_value, sd_value


def _finite_minmax(values):
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return np.nan, np.nan
    return float(np.min(finite)), float(np.max(finite))


def _merge_minmax(current, values):
    lo, hi = _finite_minmax(values)
    if not np.isfinite(lo) or not np.isfinite(hi):
        return current
    if current is None:
        return [lo, hi]
    current[0] = min(current[0], lo)
    current[1] = max(current[1], hi)
    return current


def _pad_minmax(lim, *, pad_fraction=0.05):
    if lim is None:
        return None
    lo, hi = float(lim[0]), float(lim[1])
    if not np.isfinite(lo) or not np.isfinite(hi):
        return None
    if np.isclose(lo, hi):
        pad = max(1e-6, abs(lo) * 0.05 + 1e-6)
    else:
        pad = (hi - lo) * float(pad_fraction)
    return (lo - pad, hi + pad)


def compute_trace_batch_mean_figure_shared_limits(output_path, parameter_set_ids, *, zoom_center_s=None, zoom_half_width_s=None):
    limits = {
        'rate_ylim': None,
        'integrated_input_ylim': None,
        'cst_ylim': None,
        'integrated_cst_ylim': None,
        'sync_rate_controlled_ylim': None,
        'sync_absolute_ylim': None,
        'sync_rate_controlled_dist_xlim': None,
        'sync_absolute_dist_xlim': None,
        'zoom_integrated_input_ylim': None,
        'zoom_integrated_cst_ylim': None,
        'zoom_cumulative_residual_ylim': None,
        'time_xlim': None,
        'sync_time_xlim': None,
    }
    if zoom_center_s is None:
        zoom_center_s = float(ignored_initial_duration_s) + 1.0
    if zoom_half_width_s is None:
        zoom_half_width_s = float(TRACE_BATCH_INTEGRAL_ZOOM_HALF_WIDTH_S)
    zoom_xlim = (float(zoom_center_s) - float(zoom_half_width_s), float(zoom_center_s) + float(zoom_half_width_s))
    limits['integral_zoom_xlim'] = zoom_xlim
    for parameter_set_id in parameter_set_ids:
        trace_data, _ = load_trace_batch_parameter_set_npz(output_path, parameter_set_id)
        t = np.asarray(trace_data['time_s'], dtype=float)
        sync_t = np.asarray(trace_data['sync_time_s'], dtype=float)
        limits['time_xlim'] = _merge_minmax(limits['time_xlim'], t)
        limits['sync_time_xlim'] = _merge_minmax(limits['sync_time_xlim'], sync_t)
        limits['rate_ylim'] = _merge_minmax(limits['rate_ylim'], trace_data['mean_unit_rate_hz'])
        limits['rate_ylim'] = _merge_minmax(limits['rate_ylim'], trace_data['raw_rate_hz'])
        limits['integrated_input_ylim'] = _merge_minmax(limits['integrated_input_ylim'], trace_data['integrated_input_expected_spikes_per_neuron'])
        limits['cst_ylim'] = _merge_minmax(limits['cst_ylim'], trace_data['cst_smooth_spikes_per_s_mu'])
        limits['integrated_cst_ylim'] = _merge_minmax(limits['integrated_cst_ylim'], trace_data['integrated_cst_spikes_per_mu'])
        limits['sync_rate_controlled_ylim'] = _merge_minmax(limits['sync_rate_controlled_ylim'], trace_data['sync_rate_controlled'])
        limits['sync_absolute_ylim'] = _merge_minmax(limits['sync_absolute_ylim'], trace_data['sync_absolute_coincidence_rate'])
        limits['sync_rate_controlled_dist_xlim'] = _merge_minmax(limits['sync_rate_controlled_dist_xlim'], trace_data['sync_rate_controlled'])
        limits['sync_absolute_dist_xlim'] = _merge_minmax(limits['sync_absolute_dist_xlim'], trace_data['sync_absolute_coincidence_rate'])
        zoom_mask = (t >= zoom_xlim[0]) & (t <= zoom_xlim[1])
        if np.any(zoom_mask):
            limits['zoom_integrated_input_ylim'] = _merge_minmax(limits['zoom_integrated_input_ylim'], np.asarray(trace_data['integrated_input_expected_spikes_per_neuron'])[:, zoom_mask])
            limits['zoom_integrated_cst_ylim'] = _merge_minmax(limits['zoom_integrated_cst_ylim'], np.asarray(trace_data['integrated_cst_spikes_per_mu'])[:, zoom_mask])
            limits['zoom_cumulative_residual_ylim'] = _merge_minmax(limits['zoom_cumulative_residual_ylim'], _get_cumulative_residual_trace(trace_data)[:, zoom_mask])
    for key in list(limits.keys()):
        if key.endswith('_ylim') or key.endswith('_xlim'):
            if key in {'integral_zoom_xlim'}:
                continue
            limits[key] = _pad_minmax(limits[key], pad_fraction=0.05)
    return limits


def _apply_limit(ax, limit, setter):
    if limit is not None:
        setter(*limit)


def plot_trace_batch_mean_summary(
    trace_data,
    *,
    parameter_values=None,
    title=None,
    xlim_s=None,
    shared_limits=None,
    zoom_window_center_s=None,
    zoom_window_half_width_s=None,
    figsize=(17, 10),
):
    t = np.asarray(trace_data['time_s'], dtype=float)
    sync_t = np.asarray(trace_data['sync_time_s'], dtype=float)
    if zoom_window_center_s is None:
        zoom_window_center_s = float(ignored_initial_duration_s) + 1.0
    if zoom_window_half_width_s is None:
        zoom_window_half_width_s = float(TRACE_BATCH_INTEGRAL_ZOOM_HALF_WIDTH_S)
    zoom_xlim = (
        float(zoom_window_center_s) - float(zoom_window_half_width_s),
        float(zoom_window_center_s) + float(zoom_window_half_width_s),
    )
    if shared_limits and shared_limits.get('integral_zoom_xlim') is not None:
        zoom_xlim = tuple(shared_limits['integral_zoom_xlim'])

    fig = plt.figure(figsize=figsize, constrained_layout=True)
    gs = fig.add_gridspec(4, 3, width_ratios=[2.1, 1.15, 1.3])
    axes = np.empty((4, 3), dtype=object)
    axes[0, 0] = fig.add_subplot(gs[0, 0])
    axes[0, 1] = fig.add_subplot(gs[0, 1])
    axes[1, 0] = fig.add_subplot(gs[1, 0])
    axes[1, 1] = fig.add_subplot(gs[1, 1])
    axes[2, 0] = fig.add_subplot(gs[2, 0])
    axes[2, 1] = fig.add_subplot(gs[2, 1])
    axes[3, 0] = fig.add_subplot(gs[3, 0])
    axes[3, 1] = fig.add_subplot(gs[3, 1])
    zoom_ax_input = fig.add_subplot(gs[0:2, 2])
    zoom_ax_cst = zoom_ax_input.twinx()
    axes[0, 2] = zoom_ax_input
    axes[1, 2] = zoom_ax_cst
    residual_ax = fig.add_subplot(gs[2:4, 2])
    axes[2, 2] = residual_ax
    axes[3, 2] = None

    alpha = float(TRACE_BATCH_INDIVIDUAL_TRACE_ALPHA)
    mean_lw = float(TRACE_BATCH_MEAN_TRACE_LW)
    shared_limits = dict(shared_limits or {})

    def _set_time_xlim(ax, x, *, sync_axis=False):
        if xlim_s is not None:
            ax.set_xlim(*xlim_s)
        elif shared_limits.get('sync_time_xlim' if sync_axis else 'time_xlim') is not None:
            ax.set_xlim(*shared_limits['sync_time_xlim' if sync_axis else 'time_xlim'])
        elif x.size:
            ax.set_xlim(float(np.nanmin(x)), float(np.nanmax(x)))

    _plot_trace_stack(axes[0, 0], t, trace_data['mean_unit_rate_hz'], color='black', mean_label='mean rate across realizations', individual_label='individual realization', alpha=alpha, mean_lw=mean_lw)
    axes[0, 0].plot(t, _finite_mean_trace(trace_data['raw_rate_hz']), color='0.55', lw=1.0, alpha=0.9, label='mean raw rate')
    if 'pulse_trace_hz' in trace_data:
        baseline_for_pulse = _trace_scalar_first(trace_data, 'baseline_rate_hz', baseline_rate_hz)
        axes[0, 0].plot(t, baseline_for_pulse + _finite_mean_trace(trace_data['pulse_trace_hz']), color='#2ca02c', lw=1.0, alpha=0.75, label='baseline + pulse')
    _add_pulse_window_from_trace_data(axes[0, 0], trace_data, label=True)
    axes[0, 0].set_ylabel('rate\n(Hz)')
    axes[0, 0].legend(loc='upper right', fontsize=8, ncol=2)
    _set_time_xlim(axes[0, 0], t)
    _apply_limit(axes[0, 0], shared_limits.get('rate_ylim'), axes[0, 0].set_ylim)

    _plot_trace_stack(axes[0, 1], t, trace_data['integrated_input_expected_spikes_per_neuron'], color='black', mean_label='mean integral', individual_label='individual realization', alpha=alpha, mean_lw=mean_lw)
    axes[0, 1].set_title('Integrated input')
    axes[0, 1].set_ylabel('expected total spikes\nper neuron')
    _add_pulse_window_from_trace_data(axes[0, 1], trace_data)
    _set_time_xlim(axes[0, 1], t)
    _apply_limit(axes[0, 1], shared_limits.get('integrated_input_ylim'), axes[0, 1].set_ylim)

    _plot_trace_stack(axes[1, 0], t, trace_data['cst_smooth_spikes_per_s_mu'], color='#ff3300', mean_label='mean causal half-Hann CST', individual_label='individual realization', alpha=alpha, mean_lw=mean_lw)
    axes[1, 0].set_ylabel('causal half-Hann CST\n(sp/s/MU)')
    axes[1, 0].legend(loc='upper right', fontsize=8)
    _add_pulse_window_from_trace_data(axes[1, 0], trace_data)
    _set_time_xlim(axes[1, 0], t)
    _apply_limit(axes[1, 0], shared_limits.get('cst_ylim'), axes[1, 0].set_ylim)

    _plot_trace_stack(axes[1, 1], t, trace_data['integrated_cst_spikes_per_mu'], color='#ff3300', mean_label='mean integral', individual_label='individual realization', alpha=alpha, mean_lw=mean_lw)
    axes[1, 1].set_title('Integrated CST')
    axes[1, 1].set_ylabel('total spikes\nper neuron')
    _add_pulse_window_from_trace_data(axes[1, 1], trace_data)
    _set_time_xlim(axes[1, 1], t)
    _apply_limit(axes[1, 1], shared_limits.get('integrated_cst_ylim'), axes[1, 1].set_ylim)

    _plot_trace_stack(axes[2, 0], sync_t, trace_data['sync_rate_controlled'], color='#852680', mean_label='mean rate-normalized sync', individual_label='individual realization', alpha=alpha, mean_lw=mean_lw, zero_line=True)
    axes[2, 0].set_ylabel('rate-normalized\nsynchrony')
    _add_pulse_window_from_trace_data(axes[2, 0], trace_data)
    _set_time_xlim(axes[2, 0], sync_t, sync_axis=True)
    _apply_limit(axes[2, 0], shared_limits.get('sync_rate_controlled_ylim'), axes[2, 0].set_ylim)
    _plot_mean_kde_panel(axes[2, 1], trace_data['sync_rate_controlled'], color='#852680', title='Rate-normalized synchrony distribution', xlabel='rate-normalized synchrony', alpha=alpha)
    _apply_limit(axes[2, 1], shared_limits.get('sync_rate_controlled_dist_xlim'), axes[2, 1].set_xlim)

    _plot_trace_stack(axes[3, 0], sync_t, trace_data['sync_absolute_coincidence_rate'], color='#0050cc', mean_label='mean absolute sync', individual_label='individual realization', alpha=alpha, mean_lw=mean_lw)
    axes[3, 0].set_ylabel('absolute coincidence\nrate / pair (1/s)')
    axes[3, 0].set_xlabel('time (s)')
    _add_pulse_window_from_trace_data(axes[3, 0], trace_data)
    _set_time_xlim(axes[3, 0], sync_t, sync_axis=True)
    _apply_limit(axes[3, 0], shared_limits.get('sync_absolute_ylim'), axes[3, 0].set_ylim)
    _plot_mean_kde_panel(axes[3, 1], trace_data['sync_absolute_coincidence_rate'], color='#0050cc', title='Absolute synchrony distribution', xlabel='coincidence rate / pair (1/s)', alpha=alpha)
    _apply_limit(axes[3, 1], shared_limits.get('sync_absolute_dist_xlim'), axes[3, 1].set_xlim)

    # Two-row zoom panel: integrated input behind, integrated CST above it.
    _plot_trace_stack(zoom_ax_input, t, trace_data['integrated_input_expected_spikes_per_neuron'], color='black', mean_label='input integral', individual_label=None, alpha=max(0.06, alpha * 0.7), mean_lw=mean_lw)
    _plot_trace_stack(zoom_ax_cst, t, trace_data['integrated_cst_spikes_per_mu'], color='#ff3300', mean_label='CST integral', individual_label=None, alpha=max(0.06, alpha * 0.7), mean_lw=mean_lw)
    _add_pulse_window_from_trace_data(zoom_ax_input, trace_data)
    _add_pulse_window_from_trace_data(zoom_ax_cst, trace_data)
    zoom_ax_input.set_title(f'Integrated traces zoom ({zoom_xlim[0]:.2f}-{zoom_xlim[1]:.2f} s)')
    zoom_ax_input.set_xlim(*zoom_xlim)
    zoom_ax_input.set_ylabel('expected spikes\nper neuron', color='black')
    zoom_ax_cst.set_ylabel('CST spikes\nper neuron', color='#ff3300')
    zoom_ax_input.tick_params(axis='y', labelcolor='black')
    zoom_ax_cst.tick_params(axis='y', labelcolor='#ff3300')
    zoom_ax_input.set_xlabel('time (s)')
    zoom_ax_input.set_zorder(1)
    zoom_ax_cst.set_zorder(2)
    zoom_ax_cst.patch.set_visible(False)
    _apply_limit(zoom_ax_input, shared_limits.get('zoom_integrated_input_ylim'), zoom_ax_input.set_ylim)
    _apply_limit(zoom_ax_cst, shared_limits.get('zoom_integrated_cst_ylim'), zoom_ax_cst.set_ylim)
    handles_1, labels_1 = zoom_ax_input.get_legend_handles_labels()
    handles_2, labels_2 = zoom_ax_cst.get_legend_handles_labels()
    zoom_ax_input.legend(handles_1 + handles_2, labels_1 + labels_2, fontsize=8, loc='upper left')

    residual_trace = _get_cumulative_residual_trace(trace_data)
    _plot_trace_stack(
        residual_ax,
        t,
        residual_trace,
        color='#444444',
        mean_label='mean CST - input residual',
        individual_label='individual realization',
        alpha=max(0.06, alpha * 0.7),
        mean_lw=mean_lw,
        zero_line=True,
    )
    _add_pulse_window_from_trace_data(residual_ax, trace_data)
    residual_ax.set_title('Cumulative residual zoom')
    residual_ax.set_xlim(*zoom_xlim)
    residual_ax.set_ylabel('CST - input\nspikes per neuron')
    residual_ax.set_xlabel('time (s)')
    _apply_limit(residual_ax, shared_limits.get('zoom_cumulative_residual_ylim'), residual_ax.set_ylim)
    residual_ax.legend(fontsize=8, loc='best')

    for row in range(2):
        axes[row, 1].set_xlabel('time (s)')
    all_axes = [ax for ax in axes.ravel() if ax is not None]
    for ax in all_axes:
        ax.grid(alpha=0.18)
    if title is None:
        param_text = ''
        if parameter_values:
            param_text = ' | ' + ', '.join(f'{key}={value:g}' if isinstance(value, (int, float, np.integer, np.floating)) else f'{key}={value}' for key, value in parameter_values.items())
        n_reps = int(len(trace_data.get('realization_seed', [])))
        title = f'Trace-batch mean summary | n={n_reps} realizations{param_text}'
    fig.suptitle(title, fontsize=10)
    return fig, axes

def _save_figure_formats(fig, base_path, formats):
    paths = []
    formats = [formats] if isinstance(formats, str) else list(formats)
    for fmt in formats:
        out = Path(str(base_path)).with_suffix(f'.{fmt}')
        out.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, dpi=180, bbox_inches='tight')
        paths.append(str(out))
    return paths


def _resolve_metric_series(df, metric_name):
    if metric_name in df.columns:
        return pd.to_numeric(df[metric_name], errors='coerce')
    mean_col = f'mean_{metric_name}'
    if mean_col in df.columns:
        return pd.to_numeric(df[mean_col], errors='coerce')
    raise KeyError(f'Metric/column not found: {metric_name}')


def _linear_fit_for_label(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    if int(np.sum(finite)) < 2 or float(np.std(x[finite], ddof=0)) <= 0:
        return np.nan, np.nan, np.nan
    slope, intercept = np.polyfit(x[finite], y[finite], 1)
    yhat = slope * x[finite] + intercept
    ss_res = float(np.sum((y[finite] - yhat) ** 2))
    ss_tot = float(np.sum((y[finite] - float(np.mean(y[finite]))) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return float(slope), float(intercept), float(r2)


def _prepare_relationship_curve_table(df, *, x_metric, y_metric, color_by=None, x_binning='auto', n_quantile_bins=8, summary_mode=False):
    work = df.copy()
    work['_x_value'] = _resolve_metric_series(work, x_metric)
    work['_y_value'] = _resolve_metric_series(work, y_metric)
    if summary_mode:
        for prefix, target in [('sd', '_y_sd_value'), ('min', '_y_min_value'), ('max', '_y_max_value')]:
            col = f'{prefix}_{y_metric}'
            work[target] = pd.to_numeric(work[col], errors='coerce') if col in work.columns else np.nan
    work = work[np.isfinite(work['_x_value']) & np.isfinite(work['_y_value'])].copy()
    if work.empty:
        return pd.DataFrame(), None
    unique_x = np.sort(pd.unique(work['_x_value']))
    use_bins = False
    if x_binning == 'quantile':
        use_bins = True
    elif x_binning == 'auto' and len(unique_x) > 30:
        use_bins = True
    if use_bins:
        n_bins = min(int(n_quantile_bins), int(work['_x_value'].nunique()))
        if n_bins < 2:
            use_bins = False
        else:
            work['_x_bin'] = pd.qcut(work['_x_value'], q=n_bins, duplicates='drop')
            group_x = '_x_bin'
    if not use_bins:
        group_x = '_x_value'
    group_cols = [group_x]
    if color_by and color_by in work.columns:
        group_cols = [color_by, group_x]
    rows = []
    for key, sub in work.groupby(group_cols, dropna=False, sort=True):
        if not isinstance(key, tuple):
            key = (key,)
        row = {}
        if color_by and color_by in work.columns:
            row[color_by] = key[0]
            x_key = key[1]
        else:
            x_key = key[0]
        if use_bins:
            row['x_label'] = f'{float(x_key.left):.3g}-{float(x_key.right):.3g}'
            row['x_value'] = float(np.nanmean(sub['_x_value']))
        else:
            row['x_label'] = f'{float(x_key):.3g}'
            row['x_value'] = float(x_key)
        y = sub['_y_value'].to_numpy(dtype=float)
        row['y_mean'] = float(np.nanmean(y))
        if summary_mode and '_y_sd_value' in sub.columns and sub['_y_sd_value'].notna().any():
            row['y_sd'] = float(np.nanmean(sub['_y_sd_value']))
            row['y_min'] = float(np.nanmin(sub['_y_min_value'])) if sub['_y_min_value'].notna().any() else float(np.nanmin(y))
            row['y_max'] = float(np.nanmax(sub['_y_max_value'])) if sub['_y_max_value'].notna().any() else float(np.nanmax(y))
        else:
            row['y_sd'] = float(np.nanstd(y, ddof=0)) if y.size > 1 else 0.0
            row['y_min'] = float(np.nanmin(y))
            row['y_max'] = float(np.nanmax(y))
        row['n'] = int(np.sum(np.isfinite(y)))
        rows.append(row)
    sort_cols = ([color_by] if color_by and color_by in work.columns else []) + ['x_value']
    return pd.DataFrame(rows).sort_values(sort_cols), work


def plot_trace_batch_metric_relationship(realization_df, parameter_set_df=None, *, spec=None, figsize=(8, 5)):
    spec = dict(spec or {})
    x_metric = spec.get('x_metric', 'common_fraction')
    y_metric = spec.get('y_metric', 'mean_sync_rate_controlled')
    data_mode = spec.get('data_mode', 'realization')
    plot_mode = spec.get('plot_mode', 'both')
    color_by = spec.get('color_by', None)
    x_binning = spec.get('x_binning', 'auto')
    n_quantile_bins = int(spec.get('n_quantile_bins', 8))
    title = spec.get('title', f'{y_metric} vs {x_metric}')
    if data_mode not in {'realization', 'parameter_set_summary'}:
        raise ValueError("Relationship data_mode must be 'realization' or 'parameter_set_summary'")
    if plot_mode not in {'scatter', 'curve', 'both'}:
        raise ValueError("Relationship plot_mode must be 'scatter', 'curve', or 'both'")
    if data_mode == 'parameter_set_summary':
        if parameter_set_df is None or len(parameter_set_df) == 0:
            raise ValueError('parameter_set_summary mode requires trace_batch_parameter_set_df')
        source_df = parameter_set_df.copy()
    else:
        source_df = realization_df.copy()
    fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
    plotted_tables = {}

    if plot_mode in {'scatter', 'both'}:
        scatter_df = source_df.copy()
        scatter_df['_x_value'] = _resolve_metric_series(scatter_df, x_metric)
        scatter_df['_y_value'] = _resolve_metric_series(scatter_df, y_metric)
        scatter_df = scatter_df[np.isfinite(scatter_df['_x_value']) & np.isfinite(scatter_df['_y_value'])].copy()
        scatter_label = 'parameter sets' if data_mode == 'parameter_set_summary' else 'realizations'
        if color_by and color_by in scatter_df.columns:
            values = pd.unique(scatter_df[color_by])
            cmap = plt.get_cmap('viridis')
            for idx, value in enumerate(values):
                sub = scatter_df[scatter_df[color_by] == value]
                ax.scatter(sub['_x_value'], sub['_y_value'], s=20, alpha=0.45, color=cmap(idx / max(1, len(values) - 1)), label=f'{color_by}={value}')
        else:
            ax.scatter(scatter_df['_x_value'], scatter_df['_y_value'], s=24, alpha=0.55, color='0.25', label=scatter_label)
        plotted_tables['scatter'] = scatter_df.drop(columns=[col for col in ['_x_value', '_y_value'] if col in scatter_df.columns]).copy()

    curve_df, finite_source_df = _prepare_relationship_curve_table(source_df, x_metric=x_metric, y_metric=y_metric, color_by=color_by, x_binning=x_binning, n_quantile_bins=n_quantile_bins, summary_mode=(data_mode == 'parameter_set_summary'))
    if plot_mode in {'curve', 'both'} and not curve_df.empty:
        if color_by and color_by in curve_df.columns:
            hue_values = pd.unique(curve_df[color_by])
            cmap = plt.get_cmap('plasma')
            for idx, hue_value in enumerate(hue_values):
                sub = curve_df[curve_df[color_by] == hue_value].sort_values('x_value')
                color = cmap(0.25 + 0.75 * idx / max(1, len(hue_values) - 1))
                ax.plot(sub['x_value'], sub['y_mean'], color=color, lw=2.0, label=f'{color_by}={hue_value}')
                ax.fill_between(sub['x_value'], sub['y_mean'] - sub['y_sd'], sub['y_mean'] + sub['y_sd'], color=color, alpha=0.18)
                ax.plot(sub['x_value'], sub['y_min'], color=color, lw=1.0, ls=':', alpha=0.9)
                ax.plot(sub['x_value'], sub['y_max'], color=color, lw=1.0, ls=':', alpha=0.9)
        else:
            sub = curve_df.sort_values('x_value')
            ax.plot(sub['x_value'], sub['y_mean'], color='#852680', lw=2.2, label='mean')
            ax.fill_between(sub['x_value'], sub['y_mean'] - sub['y_sd'], sub['y_mean'] + sub['y_sd'], color='#852680', alpha=0.18, label='mean +/- SD')
            ax.plot(sub['x_value'], sub['y_min'], color='#852680', lw=1.1, ls=':', label='min/max')
            ax.plot(sub['x_value'], sub['y_max'], color='#852680', lw=1.1, ls=':')
        plotted_tables['curve'] = curve_df.copy()

    if finite_source_df is not None and not finite_source_df.empty:
        slope, intercept, r2 = _linear_fit_for_label(finite_source_df['_x_value'], finite_source_df['_y_value'])
        if np.isfinite(r2):
            ax.text(0.02, 0.98, f'linear R2={r2:.3f}', ha='left', va='top', transform=ax.transAxes, fontsize=9, bbox={'facecolor': 'white', 'edgecolor': '0.75', 'alpha': 0.9})
    ax.set_title(title)
    ax.set_xlabel(x_metric)
    ax.set_ylabel(y_metric)
    ax.grid(alpha=0.18)
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        unique = dict(zip(labels, handles))
        ax.legend(unique.values(), unique.keys(), fontsize=8, loc='best')
    return fig, ax, plotted_tables


if RUN_TRACE_BATCH:
    if 'trace_batch_realization_df' not in globals() or trace_batch_realization_df is None or len(trace_batch_realization_df) == 0:
        raise RuntimeError('trace_batch_realization_df is missing or empty. Run the trace-batch simulation/load cell first.')

    trace_batch_output_path = make_trace_batch_output_path(TRACE_BATCH_OUTPUT_DIR, TRACE_BATCH_RUN_NAME)
    trace_batch_figure_paths = []
    trace_batch_relationship_paths = []
    trace_batch_relationship_tables = {}

    if bool(globals().get('PLOT_TRACE_BATCH_FIGURES', True)):
        trace_save_formats = [TRACE_BATCH_SAVE_FIGURE_FORMATS] if isinstance(TRACE_BATCH_SAVE_FIGURE_FORMATS, str) else list(TRACE_BATCH_SAVE_FIGURE_FORMATS)
        if not trace_save_formats:
            raise ValueError('TRACE_BATCH_SAVE_FIGURE_FORMATS cannot be empty')

        if bool(globals().get('PLOT_TRACE_BATCH_MEAN_TRACE_FIGURES', True)):
            selected_parameter_set_ids = TRACE_BATCH_MEAN_FIGURE_PARAMETER_SET_IDS
            available_parameter_set_ids = sorted(int(value) for value in pd.unique(trace_batch_realization_df['parameter_set_id']))
            if selected_parameter_set_ids is None:
                selected_parameter_set_ids = available_parameter_set_ids
            else:
                selected_parameter_set_ids = [int(value) for value in selected_parameter_set_ids]
                missing_ids = sorted(set(selected_parameter_set_ids).difference(available_parameter_set_ids))
                if missing_ids:
                    raise ValueError(f'Requested TRACE_BATCH_MEAN_FIGURE_PARAMETER_SET_IDS not present: {missing_ids}')
            if TRACE_BATCH_MAX_MEAN_FIGURES is not None:
                selected_parameter_set_ids = selected_parameter_set_ids[:int(TRACE_BATCH_MAX_MEAN_FIGURES)]

            trace_batch_shared_limits = None
            trace_batch_zoom_center_s = TRACE_BATCH_INTEGRAL_ZOOM_CENTER_S
            if trace_batch_zoom_center_s is None:
                trace_batch_zoom_center_s = float(ignored_initial_duration_s) + 1.0
            if bool(TRACE_BATCH_SHARE_MEAN_FIGURE_AXES):
                trace_batch_shared_limits = compute_trace_batch_mean_figure_shared_limits(
                    trace_batch_output_path,
                    selected_parameter_set_ids,
                    zoom_center_s=trace_batch_zoom_center_s,
                    zoom_half_width_s=TRACE_BATCH_INTEGRAL_ZOOM_HALF_WIDTH_S,
                )

            for parameter_set_id in selected_parameter_set_ids:
                trace_data, npz_path = load_trace_batch_parameter_set_npz(trace_batch_output_path, parameter_set_id)
                ps_rows = trace_batch_realization_df[trace_batch_realization_df['parameter_set_id'] == parameter_set_id]
                parameter_values = {}
                for name in TRACE_BATCH_PARAMETER_VALUES.keys():
                    if name in ps_rows.columns and len(ps_rows):
                        parameter_values[name] = ps_rows.iloc[0][name]
                fig, axes = plot_trace_batch_mean_summary(
                    trace_data,
                    parameter_values=parameter_values,
                    title=None,
                    xlim_s=summary_xlim_s,
                    shared_limits=trace_batch_shared_limits,
                    zoom_window_center_s=trace_batch_zoom_center_s,
                    zoom_window_half_width_s=TRACE_BATCH_INTEGRAL_ZOOM_HALF_WIDTH_S,
                )
                if TRACE_BATCH_SAVE_OUTPUTS:
                    base = Path(npz_path).with_name('trace_batch_mean_summary')
                    trace_batch_figure_paths.extend(_save_figure_formats(fig, base, trace_save_formats))
                plt.show()
        else:
            display(Markdown('Trace-batch mean trace summary figures skipped (`PLOT_TRACE_BATCH_MEAN_TRACE_FIGURES=False`).'))

        if bool(globals().get('PLOT_TRACE_BATCH_RELATION_FIGURES', True)):
            for relation_index, relation_spec in enumerate(TRACE_BATCH_RELATION_SPECS):
                fig, ax, plotted_tables = plot_trace_batch_metric_relationship(trace_batch_realization_df, trace_batch_parameter_set_df, spec=relation_spec)
                safe_name = f"relationship_{relation_index:02d}_{relation_spec.get('y_metric', 'y')}_vs_{relation_spec.get('x_metric', 'x')}"
                safe_name = ''.join(ch if ch.isalnum() or ch in {'_', '-'} else '_' for ch in safe_name)
                if TRACE_BATCH_SAVE_OUTPUTS:
                    base = trace_batch_output_path / safe_name
                    trace_batch_relationship_paths.extend(_save_figure_formats(fig, base, trace_save_formats))
                    for table_name, table_df in plotted_tables.items():
                        table_path = trace_batch_output_path / f'{safe_name}_{table_name}_table.pkl'
                        table_df.to_pickle(table_path)
                        table_df.to_csv(table_path.with_suffix('.csv'), index=False)
                        trace_batch_relationship_tables[f'{safe_name}_{table_name}'] = str(table_path)
                plt.show()
        else:
            display(Markdown('Trace-batch relationship figures skipped (`PLOT_TRACE_BATCH_RELATION_FIGURES=False`).'))

        if TRACE_BATCH_SAVE_OUTPUTS:
            save_trace_batch_outputs(
                trace_batch_output_path,
                realization_df=trace_batch_realization_df,
                parameter_set_df=trace_batch_parameter_set_df,
                config_snapshot=trace_batch_config_snapshot if 'trace_batch_config_snapshot' in globals() else collect_trace_batch_config_snapshot(),
                manifest_updates={
                    'stage': 'trace_batch_figures_saved',
                    'mean_figure_paths': trace_batch_figure_paths,
                    'relationship_figure_paths': trace_batch_relationship_paths,
                    'relationship_table_paths': trace_batch_relationship_tables,
                    'plot_trace_batch_figures': bool(globals().get('PLOT_TRACE_BATCH_FIGURES', True)),
                    'plot_trace_batch_mean_trace_figures': bool(globals().get('PLOT_TRACE_BATCH_MEAN_TRACE_FIGURES', True)),
                    'plot_trace_batch_relation_figures': bool(globals().get('PLOT_TRACE_BATCH_RELATION_FIGURES', True)),
                },
            )
    else:
        display(Markdown('Trace-batch figure generation skipped (`PLOT_TRACE_BATCH_FIGURES=False`). Cached tables and compact traces remain loaded.'))
else:
    display(Markdown('Trace batch figures are disabled because `RUN_TRACE_BATCH=False`.'))


## Pulse-Step CST Summary

This figure reuses saved trace-batch compact outputs. Each column is one `common_fraction` value by default, each row is one pulse amplitude, and each panel overlays the mean causal-filtered CST across realizations with the mean instructive input trace.


In [ ]:

# def _sorted_unique_numeric(values):
#     values = list(pd.unique(pd.Series(values).dropna()))
#     try:
#         return sorted(values, key=float)
#     except Exception:
#         return sorted(values, key=str)


# def _format_parameter_value_for_title(name, value):
#     try:
#         return f'{name}={float(value):g}'
#     except Exception:
#         return f'{name}={value}'


# def _mean_trace_for_parameter_sets(output_path, parameter_set_ids, *, trace_key, reference_time=None):
#     traces = []
#     reference_time = None if reference_time is None else np.asarray(reference_time, dtype=float)
#     first_trace_data = None
#     for parameter_set_id in parameter_set_ids:
#         trace_data, npz_path = load_trace_batch_parameter_set_npz(output_path, int(parameter_set_id))
#         if first_trace_data is None:
#             first_trace_data = trace_data
#         t = np.asarray(trace_data['time_s'], dtype=float)
#         values = np.asarray(trace_data[trace_key], dtype=float)
#         mean_values = _finite_mean_trace(values)
#         if reference_time is None:
#             reference_time = t
#         else:
#             mean_values = _interp_to_reference(t, mean_values, reference_time)
#         traces.append(mean_values)
#     if not traces:
#         return reference_time, np.asarray([], dtype=float), first_trace_data
#     return reference_time, np.nanmean(np.vstack(traces), axis=0), first_trace_data


# def build_trace_batch_pulse_step_plot_table(realization_df, *, column_parameter, row_parameter, include_zero_pulse=True):
#     required = {'parameter_set_id', column_parameter, row_parameter}
#     missing = sorted(required.difference(realization_df.columns))
#     if missing:
#         raise KeyError(f'trace_batch_realization_df is missing required column(s): {missing}')
#     table = realization_df[list(required)].drop_duplicates().copy()
#     if not include_zero_pulse and row_parameter == 'pulse_input_amplitude_hz':
#         table = table[np.abs(pd.to_numeric(table[row_parameter], errors='coerce')) > 1e-12].copy()
#     table['parameter_set_id'] = table['parameter_set_id'].astype(int)
#     return table.sort_values([row_parameter, column_parameter, 'parameter_set_id']).reset_index(drop=True)


# def plot_trace_batch_pulse_step_cst_grid(
#     realization_df,
#     *,
#     output_path,
#     column_parameter='common_fraction',
#     row_parameter='pulse_input_amplitude_hz',
#     include_zero_pulse=True,
#     zoom_window_rel_s=(-0.5, 1.5),
#     share_y=True,
#     figsize_per_cell=(2.6, 2.2),
# ):
#     plot_table = build_trace_batch_pulse_step_plot_table(
#         realization_df,
#         column_parameter=column_parameter,
#         row_parameter=row_parameter,
#         include_zero_pulse=include_zero_pulse,
#     )
#     if plot_table.empty:
#         raise ValueError('No trace-batch parameter sets available for pulse-step CST plotting')
#     column_values = _sorted_unique_numeric(plot_table[column_parameter])
#     row_values = _sorted_unique_numeric(plot_table[row_parameter])
#     n_rows = len(row_values)
#     n_cols = len(column_values)
#     fig, axes = plt.subplots(
#         n_rows,
#         n_cols,
#         figsize=(max(3.0, figsize_per_cell[0] * n_cols), max(2.5, figsize_per_cell[1] * n_rows)),
#         squeeze=False,
#         sharex=True,
#         sharey=bool(share_y),
#         constrained_layout=True,
#     )
#     global_ylim = None
#     panel_cache = {}
#     for row_value in row_values:
#         for column_value in column_values:
#             ids = plot_table[
#                 (plot_table[row_parameter] == row_value)
#                 & (plot_table[column_parameter] == column_value)
#             ]['parameter_set_id'].to_numpy(dtype=int)
#             if ids.size == 0:
#                 continue
#             t, cst_mean, trace_data = _mean_trace_for_parameter_sets(output_path, ids, trace_key='cst_smooth_spikes_per_s_mu')
#             _, input_mean, _ = _mean_trace_for_parameter_sets(output_path, ids, trace_key='mean_unit_rate_hz', reference_time=t)
#             pulse_window = _get_pulse_window_from_trace_data(trace_data) if trace_data is not None else None
#             if pulse_window is None:
#                 xlim = (float(np.nanmin(t)), float(np.nanmax(t))) if t is not None and len(t) else (0.0, 1.0)
#             else:
#                 pulse_start_s, pulse_end_s, pulse_amp_hz = pulse_window
#                 xlim = (pulse_start_s + float(zoom_window_rel_s[0]), pulse_start_s + float(zoom_window_rel_s[1]))
#             zoom_mask = (t >= xlim[0]) & (t <= xlim[1]) if t is not None and len(t) else np.asarray([], dtype=bool)
#             if np.any(zoom_mask):
#                 global_ylim = _merge_minmax(global_ylim, cst_mean[zoom_mask])
#                 global_ylim = _merge_minmax(global_ylim, input_mean[zoom_mask])
#             panel_cache[(row_value, column_value)] = {
#                 'parameter_set_ids': ids,
#                 'time_s': t,
#                 'cst_mean': cst_mean,
#                 'input_mean': input_mean,
#                 'pulse_window': pulse_window,
#                 'xlim': xlim,
#             }
#     if bool(share_y):
#         global_ylim = _pad_minmax(global_ylim, pad_fraction=0.08)

#     for r, row_value in enumerate(row_values):
#         for c, column_value in enumerate(column_values):
#             ax = axes[r, c]
#             panel = panel_cache.get((row_value, column_value))
#             if panel is None:
#                 ax.text(0.5, 0.5, 'missing', ha='center', va='center', transform=ax.transAxes)
#                 ax.set_axis_off()
#                 continue
#             t = panel['time_s']
#             pulse_window = panel['pulse_window']
#             if pulse_window is not None:
#                 pulse_start_s, pulse_end_s, pulse_amp_hz = pulse_window
#                 ax.axvspan(pulse_start_s, pulse_end_s, color=TRACE_BATCH_PULSE_STEP_PULSE_COLOR, alpha=0.08, lw=0)
#                 ax.axvline(pulse_start_s, color=TRACE_BATCH_PULSE_STEP_PULSE_COLOR, lw=1.0, ls='--', alpha=0.8)
#                 ax.axvline(pulse_end_s, color=TRACE_BATCH_PULSE_STEP_PULSE_COLOR, lw=1.0, ls=':', alpha=0.7)
#             ax.plot(t, panel['cst_mean'], color=TRACE_BATCH_PULSE_STEP_CST_COLOR, lw=2.0, label='mean causal CST', zorder=3)
#             ax.plot(t, panel['input_mean'], color=TRACE_BATCH_PULSE_STEP_INPUT_COLOR, lw=1.5, ls='--', label='mean instructive input', zorder=4)
#             ax.set_xlim(*panel['xlim'])
#             if bool(share_y) and global_ylim is not None:
#                 ax.set_ylim(*global_ylim)
#             ax.grid(alpha=0.18)
#             if r == 0:
#                 ax.set_title(_format_parameter_value_for_title(column_parameter, column_value), fontsize=9)
#             if c == 0:
#                 ax.set_ylabel(_format_parameter_value_for_title(row_parameter, row_value) + '\nsp/s/MU or Hz')
#             if r == n_rows - 1:
#                 ax.set_xlabel('time (s)')
#             if r == 0 and c == 0:
#                 ax.legend(fontsize=8, loc='best')
#             n_reps = len(panel['parameter_set_ids'])
#             ax.text(
#                 0.98,
#                 0.04,
#                 f'n sets={n_reps}',
#                 transform=ax.transAxes,
#                 ha='right',
#                 va='bottom',
#                 fontsize=7,
#                 bbox={'facecolor': 'white', 'edgecolor': '0.75', 'alpha': 0.75},
#             )
#     fig.suptitle(
#         f'Mean causal CST and instructive input around pulse onset | columns={column_parameter}, rows={row_parameter}',
#         fontsize=11,
#     )
#     return fig, axes, plot_table


# if PLOT_TRACE_BATCH_PULSE_STEP_CST_SUMMARY:
#     trace_batch_output_path = make_trace_batch_output_path(TRACE_BATCH_OUTPUT_DIR, TRACE_BATCH_RUN_NAME)
#     if 'trace_batch_realization_df' not in globals() or trace_batch_realization_df is None or len(trace_batch_realization_df) == 0:
#         loaded_trace_batch_output = load_trace_batch_outputs(trace_batch_output_path)
#         trace_batch_realization_df = loaded_trace_batch_output['realization_df']
#         trace_batch_parameter_set_df = loaded_trace_batch_output['parameter_set_df']
#         trace_batch_config_snapshot = loaded_trace_batch_output['config_snapshot']
#     fig, axes, trace_batch_pulse_step_plot_table = plot_trace_batch_pulse_step_cst_grid(
#         trace_batch_realization_df,
#         output_path=trace_batch_output_path,
#         column_parameter=TRACE_BATCH_PULSE_STEP_COLUMN_PARAMETER,
#         row_parameter=TRACE_BATCH_PULSE_STEP_ROW_PARAMETER,
#         include_zero_pulse=TRACE_BATCH_PULSE_STEP_INCLUDE_ZERO_PULSE,
#         zoom_window_rel_s=TRACE_BATCH_PULSE_STEP_ZOOM_WINDOW_REL_S,
#         share_y=TRACE_BATCH_PULSE_STEP_SHARE_Y,
#     )
#     if TRACE_BATCH_PULSE_STEP_SAVE_FIGURES and TRACE_BATCH_SAVE_OUTPUTS:
#         trace_batch_pulse_step_plot_table.to_pickle(trace_batch_output_path / 'trace_batch_pulse_step_cst_plot_table.pkl')
#         trace_batch_pulse_step_plot_table.to_csv(trace_batch_output_path / 'trace_batch_pulse_step_cst_plot_table.csv', index=False)
#         pulse_step_figure_paths = _save_figure_formats(
#             fig,
#             trace_batch_output_path / 'trace_batch_pulse_step_cst_summary',
#             TRACE_BATCH_SAVE_FIGURE_FORMATS,
#         )
#         save_trace_batch_outputs(
#             trace_batch_output_path,
#             realization_df=trace_batch_realization_df,
#             parameter_set_df=trace_batch_parameter_set_df,
#             config_snapshot=trace_batch_config_snapshot if 'trace_batch_config_snapshot' in globals() else collect_trace_batch_config_snapshot(),
#             manifest_updates={
#                 'stage': 'trace_batch_pulse_step_cst_summary_saved',
#                 'pulse_step_cst_figure_paths': pulse_step_figure_paths,
#                 'pulse_step_cst_plot_table': str(trace_batch_output_path / 'trace_batch_pulse_step_cst_plot_table.pkl'),
#             },
#         )
#     plt.show()
# else:
#     display(Markdown('Pulse-step CST summary figure is disabled. Set `PLOT_TRACE_BATCH_PULSE_STEP_CST_SUMMARY=True` to generate it.'))


In [ ]:
def _sorted_unique_numeric(values):
    values = list(pd.unique(pd.Series(values).dropna()))
    try:
        return sorted(values, key=float)
    except Exception:
        return sorted(values, key=str)


def _format_parameter_value_for_title(name, value):
    try:
        return f'{name}={float(value):g}'
    except Exception:
        return f'{name}={value}'


def _format_fixed_parameter_values_for_title(fixed_parameters):
    if not fixed_parameters:
        return ''
    return ', '.join(
        _format_parameter_value_for_title(name, value)
        for name, value in fixed_parameters.items()
    )


def _is_positive_numeric(value, tol=1e-12):
    try:
        return float(value) > tol
    except Exception:
        return False


def _is_zero_numeric(value, tol=1e-12):
    try:
        return abs(float(value)) <= tol
    except Exception:
        return False


def _choose_zero_if_available_else_first(values):
    values = list(values)
    if not values:
        return None
    for value in values:
        if _is_zero_numeric(value):
            return value
    return values[0]


def _parameter_value_mask(values, target_value, tol=1e-12):
    values_series = pd.Series(values)

    numeric_values = pd.to_numeric(values_series, errors='coerce')
    try:
        numeric_target = float(target_value)
        numeric_mask = np.isfinite(numeric_values.to_numpy(dtype=float)) & np.isclose(
            numeric_values.to_numpy(dtype=float),
            numeric_target,
            atol=tol,
            rtol=0.0,
        )
        if np.any(numeric_mask):
            return numeric_mask
    except Exception:
        pass

    return values_series.astype(str).to_numpy() == str(target_value)


def _filter_table_by_parameter_values(table, parameter_values):
    mask = np.ones(len(table), dtype=bool)
    for parameter_name, parameter_value in parameter_values.items():
        if parameter_name not in table.columns:
            raise KeyError(f'Cannot filter by missing parameter column: {parameter_name!r}')
        mask &= _parameter_value_mask(table[parameter_name], parameter_value)
    return table.loc[mask].copy()


def _finite_mean_trace(values):
    values = np.asarray(values, dtype=float)
    with np.errstate(invalid='ignore'):
        return np.nanmean(values, axis=0)


def _merge_minmax(current_minmax, values):
    values = np.asarray(values, dtype=float)
    finite_values = values[np.isfinite(values)]
    if finite_values.size == 0:
        return current_minmax

    new_min = float(np.nanmin(finite_values))
    new_max = float(np.nanmax(finite_values))

    if current_minmax is None:
        return (new_min, new_max)

    return (
        min(float(current_minmax[0]), new_min),
        max(float(current_minmax[1]), new_max),
    )


def _pad_minmax(minmax, pad_fraction=0.08):
    if minmax is None:
        return None

    ymin, ymax = float(minmax[0]), float(minmax[1])
    if not np.isfinite(ymin) or not np.isfinite(ymax):
        return None

    if np.isclose(ymin, ymax):
        pad = max(abs(ymin) * pad_fraction, 1.0)
    else:
        pad = (ymax - ymin) * pad_fraction

    return ymin - pad, ymax + pad


def _interp_to_reference(t, values, reference_time):
    t = np.asarray(t, dtype=float).ravel()
    values = np.asarray(values, dtype=float).ravel()
    reference_time = np.asarray(reference_time, dtype=float).ravel()

    if t.size != values.size:
        raise ValueError(
            f'Cannot interpolate trace because time and value arrays have incompatible shapes: '
            f'time={t.shape}, values={values.shape}'
        )

    valid = np.isfinite(t) & np.isfinite(values)
    if np.sum(valid) < 2:
        return np.full(reference_time.shape, np.nan, dtype=float)

    return np.interp(
        reference_time,
        t[valid],
        values[valid],
        left=np.nan,
        right=np.nan,
    )


def _trace_matrix_for_parameter_sets(output_path, parameter_set_ids, *, trace_key, reference_time=None):
    trace_matrices = []
    reference_time = None if reference_time is None else np.asarray(reference_time, dtype=float)
    first_trace_data = None

    for parameter_set_id in parameter_set_ids:
        trace_data, npz_path = load_trace_batch_parameter_set_npz(output_path, int(parameter_set_id))

        if first_trace_data is None:
            first_trace_data = trace_data

        t = np.asarray(trace_data['time_s'], dtype=float).ravel()
        values = np.asarray(trace_data[trace_key], dtype=float)

        if values.ndim == 1:
            values = values[None, :]
        elif values.ndim != 2:
            raise ValueError(
                f'Expected trace_data[{trace_key!r}] to be 1D or 2D, got shape {values.shape}'
            )

        # Expected case in your data: values.shape == (n_realizations, n_time)
        if values.shape[1] == t.size:
            realization_traces = values
        # Defensive fallback: values.shape == (n_time, n_realizations)
        elif values.shape[0] == t.size:
            realization_traces = values.T
        else:
            raise ValueError(
                f'Cannot align trace_data[{trace_key!r}] with time_s. '
                f'time_s shape={t.shape}, values shape={values.shape}'
            )

        if realization_traces.shape[0] == 0:
            continue

        if reference_time is None:
            reference_time = t
        else:
            realization_traces = np.vstack([
                _interp_to_reference(t, trace, reference_time)
                for trace in realization_traces
            ])

        trace_matrices.append(realization_traces)

    if not trace_matrices:
        return reference_time, np.asarray([], dtype=float), np.asarray([], dtype=float), first_trace_data

    trace_matrix = np.vstack(trace_matrices)

    with np.errstate(invalid='ignore'):
        mean_trace = np.nanmean(trace_matrix, axis=0)

    return reference_time, trace_matrix, mean_trace, first_trace_data


def _mean_trace_for_parameter_sets(output_path, parameter_set_ids, *, trace_key, reference_time=None):
    reference_time, trace_matrix, mean_trace, first_trace_data = _trace_matrix_for_parameter_sets(
        output_path,
        parameter_set_ids,
        trace_key=trace_key,
        reference_time=reference_time,
    )
    return reference_time, mean_trace, first_trace_data


def _select_trace_rows_for_plot(trace_matrix, max_traces=None):
    trace_matrix = np.asarray(trace_matrix, dtype=float)

    if trace_matrix.ndim != 2 or trace_matrix.shape[0] == 0:
        return trace_matrix

    if max_traces is None or trace_matrix.shape[0] <= int(max_traces):
        return trace_matrix

    max_traces = int(max_traces)
    if max_traces <= 0:
        return trace_matrix[:0]

    indices = np.linspace(0, trace_matrix.shape[0] - 1, max_traces)
    indices = np.unique(np.round(indices).astype(int))
    return trace_matrix[indices]


def _representative_trace(trace_matrix, mode='first'):
    trace_matrix = np.asarray(trace_matrix, dtype=float)

    if trace_matrix.ndim == 1:
        return trace_matrix

    if trace_matrix.ndim != 2 or trace_matrix.shape[0] == 0:
        return np.asarray([], dtype=float)

    if mode == 'mean':
        with np.errstate(invalid='ignore'):
            return np.nanmean(trace_matrix, axis=0)

    for trace in trace_matrix:
        if np.any(np.isfinite(trace)):
            return trace

    return np.full(trace_matrix.shape[1], np.nan, dtype=float)


def build_trace_batch_pulse_step_plot_table(
    realization_df,
    *,
    column_parameter,
    row_parameter,
    include_zero_pulse=True,
    extra_parameters=None,
):
    extra_parameters = [] if extra_parameters is None else list(extra_parameters)

    required = {'parameter_set_id', column_parameter, row_parameter}
    required.update(parameter for parameter in extra_parameters if parameter is not None)

    missing = sorted(required.difference(realization_df.columns))
    if missing:
        raise KeyError(f'trace_batch_realization_df is missing required column(s): {missing}')

    table = realization_df[list(required)].drop_duplicates().copy()

    if not include_zero_pulse and row_parameter == 'pulse_input_amplitude_hz':
        table = table[np.abs(pd.to_numeric(table[row_parameter], errors='coerce')) > 1e-12].copy()

    table['parameter_set_id'] = table['parameter_set_id'].astype(int)

    sort_columns = [
        col for col in [row_parameter, column_parameter, *extra_parameters, 'parameter_set_id']
        if col in table.columns
    ]

    return table.sort_values(sort_columns).reset_index(drop=True)


def plot_trace_batch_pulse_step_cst_grid(
    realization_df,
    *,
    output_path,
    column_parameter='common_fraction',
    row_parameter='pulse_input_amplitude_hz',
    include_zero_pulse=True,
    zoom_window_rel_s=(-0.5, 1.5),
    share_y=True,
    figsize_per_cell=(4.0, 4.0),

    # New detail-row options.
    plot_individual_step_zoom=True,
    detail_step_parameter='pulse_input_amplitude_hz',
    detail_step_values=None,
    detail_zoom_xlim=(1.9, 2.2),
    detail_fixed_parameters=None,
    detail_default_fix_row_parameter=True,
    detail_share_y=True,
    detail_ylim=None,
    detail_figsize_per_row=4.0,
    detail_individual_trace_alpha=0.1,
    detail_individual_trace_lw=0.5,
    detail_max_individual_traces=None,
    detail_plot_mean_trace=True,
    detail_input_trace_mode='first',

    # New threshold/highlight options for detail rows.
    detail_highlight_threshold_hz=None,
    detail_highlight_step_value=None,
    detail_highlight_column_value=None,
    detail_highlight_search_xlim=None,
    detail_highlight_trace_lw=1.0,
    detail_highlight_threshold_lw=1.2,
):
    def _values_match_optional(actual_value, requested_value, tol=1e-12):
        if requested_value is None:
            return True

        try:
            return bool(np.isclose(float(actual_value), float(requested_value), atol=tol, rtol=0.0))
        except Exception:
            return str(actual_value) == str(requested_value)

    def _first_and_last_threshold_crossing_trace_indices(trace_matrix, t, threshold_hz, search_xlim):
        trace_matrix = np.asarray(trace_matrix, dtype=float)
        t = np.asarray(t, dtype=float).ravel()

        if trace_matrix.ndim != 2 or trace_matrix.shape[0] == 0 or t.size == 0:
            return [], {}

        if search_xlim is None:
            search_xlim = (float(np.nanmin(t)), float(np.nanmax(t)))

        search_mask = (
            np.isfinite(t)
            & (t >= float(search_xlim[0]))
            & (t <= float(search_xlim[1]))
        )

        if not np.any(search_mask):
            return [], {}

        crossing_times = {}

        for trace_i, trace in enumerate(trace_matrix):
            trace = np.asarray(trace, dtype=float).ravel()

            if trace.size != t.size:
                continue

            crossing_mask = search_mask & np.isfinite(trace) & (trace >= float(threshold_hz))

            if np.any(crossing_mask):
                first_crossing_index = np.flatnonzero(crossing_mask)[0]
                crossing_times[trace_i] = float(t[first_crossing_index])

        if not crossing_times:
            return [], {}

        sorted_indices = sorted(crossing_times, key=lambda idx: crossing_times[idx])

        first_idx = sorted_indices[0]
        last_idx = sorted_indices[-1]

        if first_idx == last_idx:
            return [first_idx], crossing_times

        return [first_idx, last_idx], crossing_times

    detail_fixed_parameters = dict(detail_fixed_parameters or {})

    extra_parameters = []
    if plot_individual_step_zoom:
        extra_parameters.append(detail_step_parameter)
        extra_parameters.extend(detail_fixed_parameters.keys())

    plot_table = build_trace_batch_pulse_step_plot_table(
        realization_df,
        column_parameter=column_parameter,
        row_parameter=row_parameter,
        include_zero_pulse=include_zero_pulse,
        extra_parameters=extra_parameters,
    )

    if plot_table.empty:
        raise ValueError('No trace-batch parameter sets available for pulse-step CST plotting')

    column_values = _sorted_unique_numeric(plot_table[column_parameter])
    row_values = _sorted_unique_numeric(plot_table[row_parameter])

    if plot_individual_step_zoom:
        if detail_step_parameter not in plot_table.columns:
            raise KeyError(
                f'Cannot add individual step-zoom rows because {detail_step_parameter!r} '
                f'is not present in trace_batch_realization_df.'
            )

        if (
            detail_default_fix_row_parameter
            and row_parameter != detail_step_parameter
            and row_parameter not in detail_fixed_parameters
        ):
            row_parameter_values = _sorted_unique_numeric(plot_table[row_parameter])
            default_row_value = _choose_zero_if_available_else_first(row_parameter_values)
            if default_row_value is not None:
                detail_fixed_parameters[row_parameter] = default_row_value

        if detail_step_values is None:
            detail_step_values = [
                value
                for value in _sorted_unique_numeric(plot_table[detail_step_parameter])
                if _is_positive_numeric(value)
            ]
        else:
            detail_step_values = list(detail_step_values)
    else:
        detail_step_values = []

    n_rows = len(row_values)
    n_cols = len(column_values)
    n_detail_rows = len(detail_step_values)
    n_total_rows = n_rows + n_detail_rows

    figure_width = max(3.0, figsize_per_cell[0] * n_cols)
    figure_height = max(
        2.5,
        figsize_per_cell[1] * n_rows + detail_figsize_per_row * n_detail_rows,
    )

    fig, axes = plt.subplots(
        n_total_rows,
        n_cols,
        figsize=(figure_width, figure_height),
        squeeze=False,
        sharex=False,
        sharey=False,
        constrained_layout=True,
    )

    main_global_ylim = None
    main_panel_cache = {}

    for row_value in row_values:
        for column_value in column_values:
            subset = _filter_table_by_parameter_values(
                plot_table,
                {
                    row_parameter: row_value,
                    column_parameter: column_value,
                },
            )

            ids = subset['parameter_set_id'].to_numpy(dtype=int)

            if ids.size == 0:
                continue

            t, cst_mean, trace_data = _mean_trace_for_parameter_sets(
                output_path,
                ids,
                trace_key='cst_smooth_spikes_per_s_mu',
            )

            _, input_mean, _ = _mean_trace_for_parameter_sets(
                output_path,
                ids,
                trace_key='mean_unit_rate_hz',
                reference_time=t,
            )

            pulse_window = _get_pulse_window_from_trace_data(trace_data) if trace_data is not None else None

            if pulse_window is None:
                xlim = (
                    float(np.nanmin(t)),
                    float(np.nanmax(t)),
                ) if t is not None and len(t) else (0.0, 1.0)
            else:
                pulse_start_s, pulse_end_s, pulse_amp_hz = pulse_window
                xlim = (
                    pulse_start_s + float(zoom_window_rel_s[0]),
                    pulse_start_s + float(zoom_window_rel_s[1]),
                )

            zoom_mask = (t >= xlim[0]) & (t <= xlim[1]) if t is not None and len(t) else np.asarray([], dtype=bool)

            if np.any(zoom_mask):
                main_global_ylim = _merge_minmax(main_global_ylim, cst_mean[zoom_mask])
                main_global_ylim = _merge_minmax(main_global_ylim, input_mean[zoom_mask])

            main_panel_cache[(row_value, column_value)] = {
                'parameter_set_ids': ids,
                'time_s': t,
                'cst_mean': cst_mean,
                'input_mean': input_mean,
                'pulse_window': pulse_window,
                'xlim': xlim,
            }

    if bool(share_y):
        main_global_ylim = _pad_minmax(main_global_ylim, pad_fraction=0.08)

    detail_global_ylim = None
    detail_panel_cache = {}

    for step_value in detail_step_values:
        for column_value in column_values:
            filters = {
                detail_step_parameter: step_value,
                column_parameter: column_value,
                **detail_fixed_parameters,
            }

            subset = _filter_table_by_parameter_values(plot_table, filters)
            ids = subset['parameter_set_id'].to_numpy(dtype=int)

            if ids.size == 0:
                continue

            t, cst_traces, cst_mean, trace_data = _trace_matrix_for_parameter_sets(
                output_path,
                ids,
                trace_key='cst_smooth_spikes_per_s_mu',
            )

            _, input_traces, input_mean, _ = _trace_matrix_for_parameter_sets(
                output_path,
                ids,
                trace_key='mean_unit_rate_hz',
                reference_time=t,
            )

            input_trace = _representative_trace(input_traces, mode=detail_input_trace_mode)

            zoom_mask = (
                (t >= detail_zoom_xlim[0]) & (t <= detail_zoom_xlim[1])
                if t is not None and len(t)
                else np.asarray([], dtype=bool)
            )

            if np.any(zoom_mask):
                detail_global_ylim = _merge_minmax(detail_global_ylim, cst_traces[:, zoom_mask])
                detail_global_ylim = _merge_minmax(detail_global_ylim, input_trace[zoom_mask])
                if detail_plot_mean_trace:
                    detail_global_ylim = _merge_minmax(detail_global_ylim, cst_mean[zoom_mask])

            should_highlight_this_panel = (
                detail_highlight_threshold_hz is not None
                and _values_match_optional(step_value, detail_highlight_step_value)
                and _values_match_optional(column_value, detail_highlight_column_value)
            )

            if should_highlight_this_panel:
                highlight_search_xlim = (
                    detail_zoom_xlim
                    if detail_highlight_search_xlim is None
                    else detail_highlight_search_xlim
                )

                highlight_trace_indices, highlight_crossing_times = _first_and_last_threshold_crossing_trace_indices(
                    cst_traces,
                    t,
                    detail_highlight_threshold_hz,
                    highlight_search_xlim,
                )
            else:
                highlight_trace_indices = []
                highlight_crossing_times = {}

            detail_panel_cache[(step_value, column_value)] = {
                'parameter_set_ids': ids,
                'time_s': t,
                'cst_traces': _select_trace_rows_for_plot(cst_traces, detail_max_individual_traces),
                'full_cst_traces': cst_traces,
                'n_cst_traces': int(cst_traces.shape[0]),
                'cst_mean': cst_mean,
                'input_trace': input_trace,
                'xlim': tuple(detail_zoom_xlim),
                'highlight_trace_indices': highlight_trace_indices,
                'highlight_crossing_times': highlight_crossing_times,
                'highlight_threshold_hz': detail_highlight_threshold_hz if should_highlight_this_panel else None,
            }

    if bool(detail_share_y):
        detail_global_ylim = _pad_minmax(detail_global_ylim, pad_fraction=0.08)

    for r, row_value in enumerate(row_values):
        for c, column_value in enumerate(column_values):
            ax = axes[r, c]
            panel = main_panel_cache.get((row_value, column_value))

            if panel is None:
                ax.text(0.5, 0.5, 'missing', ha='center', va='center', transform=ax.transAxes)
                ax.set_axis_off()
                continue

            t = panel['time_s']
            pulse_window = panel['pulse_window']

            if pulse_window is not None:
                pulse_start_s, pulse_end_s, pulse_amp_hz = pulse_window
                ax.axvspan(
                    pulse_start_s,
                    pulse_end_s,
                    color=TRACE_BATCH_PULSE_STEP_PULSE_COLOR,
                    alpha=0.08,
                    lw=0,
                )
                ax.axvline(
                    pulse_start_s,
                    color=TRACE_BATCH_PULSE_STEP_PULSE_COLOR,
                    lw=1.0,
                    ls='--',
                    alpha=0.8,
                )
                ax.axvline(
                    pulse_end_s,
                    color=TRACE_BATCH_PULSE_STEP_PULSE_COLOR,
                    lw=1.0,
                    ls=':',
                    alpha=0.7,
                )

            ax.plot(
                t,
                panel['cst_mean'],
                color=TRACE_BATCH_PULSE_STEP_CST_COLOR,
                lw=2.0,
                label='mean causal CST',
                zorder=3,
            )

            ax.plot(
                t,
                panel['input_mean'],
                color=TRACE_BATCH_PULSE_STEP_INPUT_COLOR,
                lw=1.5,
                ls='--',
                label='mean instructive input',
                zorder=4,
            )

            ax.set_xlim(*panel['xlim'])

            if bool(share_y) and main_global_ylim is not None:
                ax.set_ylim(*main_global_ylim)

            ax.grid(alpha=0.18)

            if r == 0:
                ax.set_title(_format_parameter_value_for_title(column_parameter, column_value), fontsize=9)

            if c == 0:
                ax.set_ylabel(_format_parameter_value_for_title(row_parameter, row_value) + '\nsp/s/MU or Hz')

            if r == n_rows - 1:
                ax.set_xlabel('time (s)')

            if r == 0 and c == 0:
                ax.legend(fontsize=8, loc='best')

            n_sets = len(panel['parameter_set_ids'])
            ax.text(
                0.98,
                0.04,
                f'n sets={n_sets}',
                transform=ax.transAxes,
                ha='right',
                va='bottom',
                fontsize=7,
                bbox={'facecolor': 'white', 'edgecolor': '0.75', 'alpha': 0.75},
            )

    for detail_r, step_value in enumerate(detail_step_values):
        absolute_r = n_rows + detail_r

        for c, column_value in enumerate(column_values):
            ax = axes[absolute_r, c]
            panel = detail_panel_cache.get((step_value, column_value))

            if panel is None:
                ax.text(0.5, 0.5, 'missing', ha='center', va='center', transform=ax.transAxes)
                ax.set_axis_off()
                continue

            t = panel['time_s']

            input_trace = panel['input_trace']
            if input_trace.size:
                ax.plot(
                    t,
                    input_trace,
                    color=TRACE_BATCH_PULSE_STEP_INPUT_COLOR,
                    lw=1.6,
                    ls='--',
                    alpha=0.65,
                    label='instructive input target',
                    zorder=1,
                )

            for trace_i, individual_trace in enumerate(panel['cst_traces']):
                ax.plot(
                    t,
                    individual_trace,
                    color=TRACE_BATCH_PULSE_STEP_CST_COLOR,
                    lw=detail_individual_trace_lw,
                    alpha=detail_individual_trace_alpha,
                    label='individual causal CST' if trace_i == 0 else None,
                    zorder=2,
                )

            highlight_threshold_hz = panel['highlight_threshold_hz']
            highlight_trace_indices = panel['highlight_trace_indices']
            highlight_crossing_times = panel['highlight_crossing_times']

            if highlight_threshold_hz is not None:
                ax.axhline(
                    float(highlight_threshold_hz),
                    color='0.25',
                    lw=detail_highlight_threshold_lw,
                    ls=':',
                    alpha=0.95,
                    label=f'threshold={float(highlight_threshold_hz):g} Hz',
                    zorder=4,
                )

                for highlight_i, trace_idx in enumerate(highlight_trace_indices):
                    highlighted_trace = panel['full_cst_traces'][trace_idx]
                    crossing_time = highlight_crossing_times.get(trace_idx, np.nan)

                    if len(highlight_trace_indices) == 1:
                        highlight_label = f'threshold-reaching trace, t={crossing_time:.3f}s'
                    elif highlight_i == 0:
                        highlight_label = f'first to threshold, t={crossing_time:.3f}s'
                    else:
                        highlight_label = f'last to threshold, t={crossing_time:.3f}s'

                    ax.plot(
                        t,
                        highlighted_trace,
                        color="#830000", # TRACE_BATCH_PULSE_STEP_CST_COLOR,
                        lw=detail_highlight_trace_lw,
                        alpha=0.6,
                        label=highlight_label,
                        zorder=5 + highlight_i,
                    )

            if detail_plot_mean_trace:
                ax.plot(
                    t,
                    panel['cst_mean'],
                    color=TRACE_BATCH_PULSE_STEP_CST_COLOR,
                    lw=2.0,
                    alpha=0.95,
                    label='mean causal CST',
                    zorder=3,
                )

            ax.set_xlim(*panel['xlim'])

            if detail_ylim is not None:
                ax.set_ylim(*detail_ylim)
            elif bool(detail_share_y) and detail_global_ylim is not None:
                ax.set_ylim(*detail_global_ylim)

            ax.grid(alpha=0.18)

            if detail_r == 0:
                ax.set_title(
                    _format_parameter_value_for_title(column_parameter, column_value),
                    fontsize=9,
                )

            if c == 0:
                ax.set_ylabel(
                    _format_parameter_value_for_title(detail_step_parameter, step_value)
                    + '\nindividual traces'
                )

            if absolute_r == n_total_rows - 1:
                ax.set_xlabel('time (s)')

            if detail_r == 0 and c == 0:
                ax.legend(fontsize=8, loc='best')

            n_sets = len(panel['parameter_set_ids'])
            n_traces = panel['n_cst_traces']
            ax.text(
                0.98,
                0.04,
                f'n sets={n_sets}\nn traces={n_traces}',
                transform=ax.transAxes,
                ha='right',
                va='bottom',
                fontsize=7,
                bbox={'facecolor': 'white', 'edgecolor': '0.75', 'alpha': 0.75},
            )

    title = (
        f'Mean causal CST and instructive input around pulse onset | '
        f'columns={column_parameter}, rows={row_parameter}'
    )

    if n_detail_rows > 0:
        fixed_label = _format_fixed_parameter_values_for_title(detail_fixed_parameters)
        title += (
            f'\nBottom rows: individual traces for {detail_step_parameter}>0, '
            f'xlim={tuple(detail_zoom_xlim)}'
        )
        if fixed_label:
            title += f' | fixed: {fixed_label}'

    fig.suptitle(title, fontsize=11)

    return fig, axes, plot_table


if PLOT_TRACE_BATCH_PULSE_STEP_CST_SUMMARY:
    trace_batch_output_path = make_trace_batch_output_path(TRACE_BATCH_OUTPUT_DIR, TRACE_BATCH_RUN_NAME)

    if (
        'trace_batch_realization_df' not in globals()
        or trace_batch_realization_df is None
        or len(trace_batch_realization_df) == 0
    ):
        loaded_trace_batch_output = load_trace_batch_outputs(trace_batch_output_path)
        trace_batch_realization_df = loaded_trace_batch_output['realization_df']
        trace_batch_parameter_set_df = loaded_trace_batch_output['parameter_set_df']
        trace_batch_config_snapshot = loaded_trace_batch_output['config_snapshot']

    fig, axes, trace_batch_pulse_step_plot_table = plot_trace_batch_pulse_step_cst_grid(
        trace_batch_realization_df,
        output_path=trace_batch_output_path,
        column_parameter=TRACE_BATCH_PULSE_STEP_COLUMN_PARAMETER,
        row_parameter=TRACE_BATCH_PULSE_STEP_ROW_PARAMETER,
        include_zero_pulse=TRACE_BATCH_PULSE_STEP_INCLUDE_ZERO_PULSE,
        zoom_window_rel_s=TRACE_BATCH_PULSE_STEP_ZOOM_WINDOW_REL_S,
        share_y=TRACE_BATCH_PULSE_STEP_SHARE_Y,

        detail_ylim=(10, 30),

        detail_highlight_threshold_hz=17, #Mean = 14.5 + 25% of step of 10Hz
        detail_highlight_step_value=None,
        detail_highlight_column_value=None,
        # New bottom-row detail plot options.
        plot_individual_step_zoom=globals().get('TRACE_BATCH_PULSE_STEP_PLOT_INDIVIDUAL_STEP_ZOOM', True),
        detail_step_parameter=globals().get('TRACE_BATCH_PULSE_STEP_DETAIL_STEP_PARAMETER', 'pulse_input_amplitude_hz'),
        detail_step_values=globals().get('TRACE_BATCH_PULSE_STEP_DETAIL_STEP_VALUES', None),
        detail_zoom_xlim=globals().get('TRACE_BATCH_PULSE_STEP_DETAIL_ZOOM_XLIM', (1.8, 2.3)),
        detail_fixed_parameters=globals().get('TRACE_BATCH_PULSE_STEP_DETAIL_FIXED_PARAMETERS', None),
        detail_default_fix_row_parameter=globals().get('TRACE_BATCH_PULSE_STEP_DETAIL_DEFAULT_FIX_ROW_PARAMETER', True),
        detail_share_y=globals().get('TRACE_BATCH_PULSE_STEP_DETAIL_SHARE_Y', True),
        detail_figsize_per_row=globals().get('TRACE_BATCH_PULSE_STEP_DETAIL_FIGSIZE_PER_ROW', 2.0),
        detail_individual_trace_alpha=globals().get('TRACE_BATCH_PULSE_STEP_DETAIL_INDIVIDUAL_TRACE_ALPHA', 0.22),
        detail_individual_trace_lw=globals().get('TRACE_BATCH_PULSE_STEP_DETAIL_INDIVIDUAL_TRACE_LW', 0.7),
        detail_max_individual_traces=globals().get('TRACE_BATCH_PULSE_STEP_DETAIL_MAX_INDIVIDUAL_TRACES', None),
        detail_plot_mean_trace=globals().get('TRACE_BATCH_PULSE_STEP_DETAIL_PLOT_MEAN_TRACE', True),
        detail_input_trace_mode=globals().get('TRACE_BATCH_PULSE_STEP_DETAIL_INPUT_TRACE_MODE', 'first'),
    )

    if TRACE_BATCH_PULSE_STEP_SAVE_FIGURES and TRACE_BATCH_SAVE_OUTPUTS:
        trace_batch_pulse_step_plot_table.to_pickle(
            trace_batch_output_path / 'trace_batch_pulse_step_cst_plot_table.pkl'
        )
        trace_batch_pulse_step_plot_table.to_csv(
            trace_batch_output_path / 'trace_batch_pulse_step_cst_plot_table.csv',
            index=False,
        )

        pulse_step_figure_paths = _save_figure_formats(
            fig,
            trace_batch_output_path / 'trace_batch_pulse_step_cst_summary',
            TRACE_BATCH_SAVE_FIGURE_FORMATS,
        )

        save_trace_batch_outputs(
            trace_batch_output_path,
            realization_df=trace_batch_realization_df,
            parameter_set_df=trace_batch_parameter_set_df,
            config_snapshot=trace_batch_config_snapshot
            if 'trace_batch_config_snapshot' in globals()
            else collect_trace_batch_config_snapshot(),
            manifest_updates={
                'stage': 'trace_batch_pulse_step_cst_summary_saved',
                'pulse_step_cst_figure_paths': pulse_step_figure_paths,
                'pulse_step_cst_plot_table': str(
                    trace_batch_output_path / 'trace_batch_pulse_step_cst_plot_table.pkl'
                ),
            },
        )

    plt.show()

else:
    display(
        Markdown(
            'Pulse-step CST summary figure is disabled. '
            'Set `PLOT_TRACE_BATCH_PULSE_STEP_CST_SUMMARY=True` to generate it.'
        )
    )

## Matched Zero-Pulse Baseline Metrics

This postprocessing section builds a counterfactual baseline table from the saved trace batch. Each row is matched to the same seed and same non-pulse parameters at `pulse_input_amplitude_hz == 0`, then zero-reference baseline and matched post-window effect metrics are added without rerunning simulations.


In [ ]:

MATCHED_ZERO_TRACE_SPECS = {
    'mean_unit_rate_hz': {
        'time_key': 'time_s',
        'output_name': 'mean_unit_rate_hz',
        'actual_post_metric': 'pulse_post_cumulative_input_expected_spikes_per_neuron',
    },
    'cst_spikes_per_s_mu': {
        'time_key': 'time_s',
        'output_name': 'raw_cst_spikes_per_s_mu',
        'actual_post_metric': 'pulse_post_raw_cst_spikes_per_s_mu',
    },
    'cst_smooth_spikes_per_s_mu': {
        'time_key': 'time_s',
        'output_name': 'filtered_cst_spikes_per_s_mu',
        'actual_post_metric': 'pulse_post_filtered_cst_spikes_per_s_mu',
    },
    'sync_rate_controlled': {
        'time_key': 'sync_time_s',
        'output_name': 'sync_rate_controlled',
        'actual_post_metric': 'pulse_post_sync_rate_controlled',
    },
    'sync_absolute_coincidence_rate': {
        'time_key': 'sync_time_s',
        'output_name': 'sync_absolute_coincidence_rate_per_s_pair',
        'actual_post_metric': 'pulse_post_sync_absolute_coincidence_rate_per_s_pair',
    },
    'cumulative_cst_minus_input_residual_spikes_per_mu': {
        'time_key': 'time_s',
        'output_name': 'cumulative_cst_minus_input_residual_spikes_per_mu',
        'actual_post_metric': 'pulse_post_cumulative_cst_minus_input_residual_spikes_per_mu',
    },
}
MATCHED_ZERO_STATS = ['mean', 'sd', 'min', 'max', 'skew', 'median', 'iqr']


def _matched_zero_stats(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return {stat: np.nan for stat in MATCHED_ZERO_STATS}
    q25, q75 = np.percentile(values, [25, 75])
    return {
        'mean': float(np.mean(values)),
        'sd': float(np.std(values, ddof=0)),
        'min': float(np.min(values)),
        'max': float(np.max(values)),
        'skew': float(skew(values, bias=False)) if values.size >= 3 and float(np.std(values, ddof=0)) > 0 else np.nan,
        'median': float(np.median(values)),
        'iqr': float(q75 - q25),
    }


def _windowed_trace_stats(time_s, trace, window_abs_s=None):
    time_s = np.asarray(time_s, dtype=float)
    trace = np.asarray(trace, dtype=float)
    if window_abs_s is not None:
        start_s, end_s = float(window_abs_s[0]), float(window_abs_s[1])
        mask = (time_s >= start_s) & (time_s < end_s)
        trace = trace[mask]
    return _matched_zero_stats(trace)


def _scalar_from_trace_array(trace_data, key, default=np.nan):
    if key not in trace_data:
        return default
    arr = np.asarray(trace_data[key])
    if arr.size == 0:
        return default
    try:
        return float(arr.flat[0])
    except Exception:
        return arr.flat[0]


def _realization_index_for_seed(trace_data, realization_seed):
    seeds = np.asarray(trace_data.get('realization_seed', []), dtype=int)
    matches = np.flatnonzero(seeds == int(realization_seed))
    if matches.size != 1:
        raise ValueError(f'Expected exactly one compact-trace realization for seed {realization_seed}, found {matches.size}')
    return int(matches[0])


def _zero_reference_stats_for_row(output_path, zero_ref_row):
    zero_trace_data, _ = load_trace_batch_parameter_set_npz(output_path, int(zero_ref_row['parameter_set_id']))
    realization_index = _realization_index_for_seed(zero_trace_data, int(zero_ref_row['realization_seed']))
    pre_window = (
        float(zero_ref_row['pulse_metric_pre_window_start_abs_s']),
        float(zero_ref_row['pulse_metric_pre_window_end_abs_s']),
    )
    post_window = (
        float(zero_ref_row['pulse_metric_post_window_start_abs_s']),
        float(zero_ref_row['pulse_metric_post_window_end_abs_s']),
    )
    stats = {}
    for trace_key, spec in MATCHED_ZERO_TRACE_SPECS.items():
        time_values = np.asarray(zero_trace_data[spec['time_key']], dtype=float)
        trace_values = np.asarray(zero_trace_data[trace_key], dtype=float)[realization_index]
        for window_name, window_abs_s in [('pre', pre_window), ('post', post_window), ('full', None)]:
            window_stats = _windowed_trace_stats(time_values, trace_values, window_abs_s)
            for stat_name, stat_value in window_stats.items():
                stats[f'zero_ref_{window_name}_{spec["output_name"]}_{stat_name}'] = stat_value
    return stats


def _infer_matched_zero_group_columns(realization_df, pulse_column='pulse_input_amplitude_hz'):
    excluded = {
        pulse_column,
        'parameter_set_id',
        'realization_index',
        'realization_seed',
        'random_seed',
    }
    candidate_columns = []
    for col in TRACE_BATCH_PARAMETER_VALUES.keys():
        if col not in excluded and col in realization_df.columns:
            candidate_columns.append(col)
    if not candidate_columns:
        for col in ['common_fraction', 'common_jitter_sd_ms', 'common_copy_probability', 'baseline_rate_hz', 'private_gamma_shape_k', 'common_gamma_shape_k']:
            if col not in excluded and col in realization_df.columns:
                candidate_columns.append(col)
    return candidate_columns


def build_trace_batch_matched_zero_baseline_table(
    realization_df,
    *,
    output_path,
    zero_amplitude_hz=0.0,
    require_all_matches=True,
):
    if 'pulse_input_amplitude_hz' not in realization_df.columns:
        raise KeyError('trace_batch_realization_df must contain pulse_input_amplitude_hz')
    if 'realization_seed' not in realization_df.columns:
        raise KeyError('trace_batch_realization_df must contain realization_seed')

    df = realization_df.copy().reset_index(drop=True)
    group_cols = _infer_matched_zero_group_columns(df)
    match_cols = group_cols + ['realization_seed']
    zero_mask = np.isclose(pd.to_numeric(df['pulse_input_amplitude_hz'], errors='coerce'), float(zero_amplitude_hz), atol=1e-12)
    zero_df = df[zero_mask].copy()
    if zero_df.empty:
        raise ValueError(f'No zero-pulse rows found for pulse_input_amplitude_hz={zero_amplitude_hz}')
    duplicate_zero = zero_df.duplicated(match_cols, keep=False)
    if duplicate_zero.any():
        examples = zero_df.loc[duplicate_zero, match_cols + ['parameter_set_id']].head()
        raise ValueError(f'Zero-reference rows are not unique for match columns {match_cols}. Examples:\\n{examples}')

    zero_lookup = {tuple(row[col] for col in match_cols): row for _, row in zero_df.iterrows()}
    zero_stats_cache = {}
    rows = []
    missing_keys = []
    for _, row in df.iterrows():
        key = tuple(row[col] for col in match_cols)
        out = row.to_dict()
        zero_row = zero_lookup.get(key)
        if zero_row is None:
            out.update({
                'zero_ref_match_status': 'missing_zero_reference',
                'zero_ref_parameter_set_id': np.nan,
                'zero_ref_realization_seed': np.nan,
                'zero_ref_pulse_input_amplitude_hz': np.nan,
            })
            missing_keys.append(key)
            rows.append(out)
            continue
        is_self_zero = bool(np.isclose(float(row['pulse_input_amplitude_hz']), float(zero_amplitude_hz), atol=1e-12))
        out.update({
            'zero_ref_match_status': 'self_zero_reference' if is_self_zero else 'matched_zero_reference',
            'zero_ref_parameter_set_id': int(zero_row['parameter_set_id']),
            'zero_ref_realization_seed': int(zero_row['realization_seed']),
            'zero_ref_pulse_input_amplitude_hz': float(zero_row['pulse_input_amplitude_hz']),
        })
        cache_key = (int(zero_row['parameter_set_id']), int(zero_row['realization_seed']))
        if cache_key not in zero_stats_cache:
            zero_stats_cache[cache_key] = _zero_reference_stats_for_row(output_path, zero_row)
        out.update(zero_stats_cache[cache_key])
        for age_col in [col for col in zero_row.index if str(col).startswith('pool_')]:
            out[f'zero_ref_{age_col}'] = zero_row.get(age_col, np.nan)
        for trace_key, spec in MATCHED_ZERO_TRACE_SPECS.items():
            actual_base = spec['actual_post_metric']
            zero_base = f'zero_ref_post_{spec["output_name"]}'
            for stat_name in MATCHED_ZERO_STATS:
                actual_col = f'{actual_base}_{stat_name}'
                zero_col = f'{zero_base}_{stat_name}'
                out_col = f'matched_effect_post_{spec["output_name"]}_{stat_name}'
                if is_self_zero:
                    out[out_col] = 0.0
                elif actual_col in out and zero_col in out:
                    actual_value = out.get(actual_col, np.nan)
                    zero_value = out.get(zero_col, np.nan)
                    out[out_col] = actual_value - zero_value if np.isfinite(actual_value) and np.isfinite(zero_value) else np.nan
                else:
                    out[out_col] = np.nan
        out['baseline_sync_rate_controlled_mean'] = out.get('zero_ref_full_sync_rate_controlled_mean', np.nan)
        out['baseline_sync_absolute_mean'] = out.get('zero_ref_full_sync_absolute_coincidence_rate_per_s_pair_mean', np.nan)
        out['local_pre_sync_rate_controlled_mean'] = out.get('zero_ref_pre_sync_rate_controlled_mean', np.nan)
        out['matched_pulse_effect_filtered_cst_mean'] = out.get('matched_effect_post_filtered_cst_spikes_per_s_mu_mean', np.nan)
        rows.append(out)

    if missing_keys and require_all_matches:
        raise ValueError(f'Missing {len(missing_keys)} zero-reference match(es); first missing key={missing_keys[0]}')
    matched_df = pd.DataFrame(rows)
    matched_df.attrs['matched_zero_group_columns'] = group_cols
    matched_df.attrs['matched_zero_match_columns'] = match_cols
    matched_df.attrs['zero_amplitude_hz'] = float(zero_amplitude_hz)
    return matched_df


if BUILD_TRACE_BATCH_MATCHED_ZERO_BASELINE:
    trace_batch_output_path = make_trace_batch_output_path(TRACE_BATCH_OUTPUT_DIR, TRACE_BATCH_RUN_NAME)
    if 'trace_batch_realization_df' not in globals() or trace_batch_realization_df is None or len(trace_batch_realization_df) == 0:
        loaded_trace_batch_output = load_trace_batch_outputs(trace_batch_output_path)
        trace_batch_realization_df = loaded_trace_batch_output['realization_df']
        trace_batch_parameter_set_df = loaded_trace_batch_output['parameter_set_df']
        trace_batch_config_snapshot = loaded_trace_batch_output['config_snapshot']
    trace_batch_matched_zero_baseline_df = build_trace_batch_matched_zero_baseline_table(
        trace_batch_realization_df,
        output_path=trace_batch_output_path,
        zero_amplitude_hz=TRACE_BATCH_MATCHED_ZERO_AMPLITUDE_HZ,
        require_all_matches=TRACE_BATCH_MATCHED_ZERO_REQUIRE_ALL_MATCHES,
    )
    if TRACE_BATCH_MATCHED_ZERO_SAVE_OUTPUTS:
        matched_pkl = trace_batch_output_path / 'trace_batch_matched_zero_baseline_table.pkl'
        matched_csv = trace_batch_output_path / 'trace_batch_matched_zero_baseline_table.csv'
        trace_batch_matched_zero_baseline_df.to_pickle(matched_pkl)
        trace_batch_matched_zero_baseline_df.to_csv(matched_csv, index=False)
        save_trace_batch_outputs(
            trace_batch_output_path,
            realization_df=trace_batch_realization_df,
            parameter_set_df=trace_batch_parameter_set_df,
            config_snapshot=trace_batch_config_snapshot if 'trace_batch_config_snapshot' in globals() else collect_trace_batch_config_snapshot(),
            manifest_updates={
                'stage': 'trace_batch_matched_zero_baseline_saved',
                'matched_zero_baseline_table_pkl': str(matched_pkl),
                'matched_zero_baseline_table_csv': str(matched_csv),
                'matched_zero_rows': int(len(trace_batch_matched_zero_baseline_df)),
                'matched_zero_status_counts': trace_batch_matched_zero_baseline_df['zero_ref_match_status'].value_counts(dropna=False).to_dict(),
            },
        )
    display(Markdown(
        f'Matched zero-pulse baseline table: {len(trace_batch_matched_zero_baseline_df)} rows; '
        f"statuses: {trace_batch_matched_zero_baseline_df['zero_ref_match_status'].value_counts(dropna=False).to_dict()}"
    ))
    display(trace_batch_matched_zero_baseline_df.head())
else:
    display(Markdown('Matched zero-pulse baseline postprocessing is disabled. Set `BUILD_TRACE_BATCH_MATCHED_ZERO_BASELINE=True` to generate it.'))


## Pulse Response Cumulative Metrics

This section computes matched pulse-response metrics from saved compact trace batches. It uses the same-seed zero-pulse trajectory as the counterfactual baseline and can run on existing saved batches without rerunning simulations. Pool age/readiness plots are generated only for batches that were rerun with age metrics saved.


In [ ]:

RESPONSE_TRACE_KEYS = [
    'response_time_rel_s',
    'input_instruction_rate_hz',
    'sign_corrected_baseline_corrected_cst_effect_spikes_per_s_mu',
    'target_rate_trace_spikes_per_s_mu',
    'cumulative_output_effect_spikes_per_mn',
    'cumulative_input_instruction_spikes_per_mn',
    'cumulative_slope_source_spikes_per_mn',
]


def _response_interp_to_time(source_t, source_y, target_t):
    source_t = np.asarray(source_t, dtype=float)
    source_y = np.asarray(source_y, dtype=float)
    target_t = np.asarray(target_t, dtype=float)
    if source_t.shape == target_t.shape and np.allclose(source_t, target_t, equal_nan=True):
        return source_y.astype(float, copy=False)
    finite = np.isfinite(source_t) & np.isfinite(source_y)
    if int(np.sum(finite)) < 2:
        return np.full_like(target_t, np.nan, dtype=float)
    return np.interp(target_t, source_t[finite], source_y[finite], left=np.nan, right=np.nan)


def _response_get_realization_trace(trace_data, key, realization_seed):
    idx = _realization_index_for_seed(trace_data, int(realization_seed))
    return np.asarray(trace_data[key], dtype=float)[idx]


def _response_fit_line(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    if int(np.sum(finite)) < 2 or float(np.std(x[finite], ddof=0)) <= 0:
        return np.nan, np.nan, np.nan
    slope, intercept = np.polyfit(x[finite], y[finite], 1)
    yhat = slope * x[finite] + intercept
    ss_res = float(np.sum((y[finite] - yhat) ** 2))
    ss_tot = float(np.sum((y[finite] - float(np.mean(y[finite]))) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return float(slope), float(intercept), float(r2)


def _response_value_at(time_rel_s, values, horizon_s):
    time_rel_s = np.asarray(time_rel_s, dtype=float)
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(time_rel_s) & np.isfinite(values)
    if int(np.sum(finite)) < 2:
        return np.nan
    if float(horizon_s) < float(np.nanmin(time_rel_s[finite])) or float(horizon_s) > float(np.nanmax(time_rel_s[finite])):
        return np.nan
    return float(np.interp(float(horizon_s), time_rel_s[finite], values[finite]))


def _first_time_at_or_above(time_rel_s, values, threshold):
    time_rel_s = np.asarray(time_rel_s, dtype=float)
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(time_rel_s) & np.isfinite(values)
    if not np.isfinite(threshold) or int(np.sum(finite)) == 0:
        return np.nan
    hits = np.flatnonzero(finite & (values >= float(threshold)))
    if hits.size == 0:
        return np.nan
    return float(time_rel_s[int(hits[0])])


def _compute_response_for_row(
    row,
    *,
    output_path,
    epsilon,
    target_fraction_of_input_rate,
    target_rate_source,
    baseline_window_rel_s,
    max_slope_source,
    max_slope_window_s,
    max_slope_horizon_s,
    max_slope_step_s,
    capture_horizons_s,
    trace_cache,
):
    actual_id = int(row['parameter_set_id'])
    zero_id = int(row['zero_ref_parameter_set_id'])
    seed = int(row['realization_seed'])
    if actual_id not in trace_cache:
        trace_cache[actual_id], _ = load_trace_batch_parameter_set_npz(output_path, actual_id)
    if zero_id not in trace_cache:
        trace_cache[zero_id], _ = load_trace_batch_parameter_set_npz(output_path, zero_id)
    actual = trace_cache[actual_id]
    zero = trace_cache[zero_id]
    t = np.asarray(actual['time_s'], dtype=float)
    zero_t = np.asarray(zero['time_s'], dtype=float)
    pulse_start_s = float(row['pulse_input_start_abs_s'])
    step_sign = float(np.sign(float(row['pulse_input_amplitude_hz'])))
    if step_sign == 0:
        step_sign = 1.0
    dt = float(np.median(np.diff(t))) if t.size > 1 else np.nan

    actual_cst = _response_get_realization_trace(actual, 'cst_spikes_per_s_mu', seed)
    zero_cst = _response_interp_to_time(zero_t, _response_get_realization_trace(zero, 'cst_spikes_per_s_mu', seed), t)
    actual_cst_smooth = _response_get_realization_trace(actual, 'cst_smooth_spikes_per_s_mu', seed)
    zero_cst_smooth = _response_interp_to_time(zero_t, _response_get_realization_trace(zero, 'cst_smooth_spikes_per_s_mu', seed), t)
    actual_input = _response_get_realization_trace(actual, 'mean_unit_rate_hz', seed)
    zero_input = _response_interp_to_time(zero_t, _response_get_realization_trace(zero, 'mean_unit_rate_hz', seed), t)
    cst_effect = actual_cst - zero_cst
    cst_smooth_effect = actual_cst_smooth - zero_cst_smooth
    input_delta = actual_input - zero_input

    pre_start = pulse_start_s + float(baseline_window_rel_s[0])
    pre_end = pulse_start_s + float(baseline_window_rel_s[1])
    pre_mask = (t >= pre_start) & (t < pre_end)
    pre_values = cst_effect[pre_mask & np.isfinite(cst_effect)]
    baseline_offset = float(np.median(pre_values)) if pre_values.size else 0.0
    smooth_pre_values = cst_smooth_effect[pre_mask & np.isfinite(cst_smooth_effect)]
    smooth_baseline_offset = float(np.median(smooth_pre_values)) if smooth_pre_values.size else 0.0
    corrected_effect = cst_effect - baseline_offset
    corrected_smooth_effect = cst_smooth_effect - smooth_baseline_offset
    sign_corrected_effect = step_sign * corrected_effect
    sign_corrected_smooth_effect = step_sign * corrected_smooth_effect

    post_mask = t >= pulse_start_s
    time_rel = t[post_mask] - pulse_start_s
    effect_post = sign_corrected_effect[post_mask]
    smooth_effect_post = sign_corrected_smooth_effect[post_mask]
    input_post = input_delta[post_mask]
    actual_cst_post = actual_cst[post_mask]
    if not np.isfinite(dt):
        raise ValueError('Could not infer response trace dt')
    cumulative_output = np.cumsum(np.nan_to_num(effect_post, nan=0.0)) * dt
    cumulative_input = np.cumsum(np.abs(np.nan_to_num(input_post, nan=0.0))) * dt

    if cumulative_input.size == 0 or float(np.nanmax(cumulative_input)) < float(epsilon):
        reaction_time = np.nan
        reaction_status = 'epsilon_too_large_for_input'
    else:
        reaction_time = _first_time_at_or_above(time_rel, cumulative_output, float(epsilon))
        reaction_status = 'reached' if np.isfinite(reaction_time) else 'not_reached'

    abs_input_post = np.abs(np.asarray(input_post, dtype=float))
    valid_target_values = abs_input_post[np.isfinite(abs_input_post) & (abs_input_post > 1e-12)]
    target_input_delta_hz = float(np.median(valid_target_values)) if valid_target_values.size else np.nan
    target_fraction = float(target_fraction_of_input_rate)
    target_threshold = target_fraction * target_input_delta_hz if np.isfinite(target_input_delta_hz) else np.nan
    target_source = str(target_rate_source).strip().lower()
    if target_source in {'matched_effect_smooth', 'smooth', 'cst_smooth'}:
        target_source_name = 'matched_effect_smooth'
        target_rate_trace = smooth_effect_post
    elif target_source in {'matched_effect_raw', 'raw', 'cst_raw'}:
        target_source_name = 'matched_effect_raw'
        target_rate_trace = effect_post
    else:
        raise ValueError("target_rate_source must be 'matched_effect_smooth' or 'matched_effect_raw'")
    target_fraction_time = _first_time_at_or_above(time_rel, target_rate_trace, target_threshold)
    if not np.isfinite(target_threshold):
        target_fraction_status = 'target_unavailable'
    elif np.isfinite(target_fraction_time):
        target_fraction_status = 'reached'
    else:
        target_fraction_status = 'not_reached'

    source = str(max_slope_source).strip().lower()
    if source in {'matched_effect', 'matched', 'effect', 'baseline_controlled'}:
        slope_source_name = 'matched_effect'
        slope_source_trace = effect_post
    elif source in {'actual_cst', 'absolute_cst', 'actual', 'step_cst'}:
        slope_source_name = 'actual_cst'
        slope_source_trace = actual_cst_post
    else:
        raise ValueError("max_slope_source must be 'matched_effect' or 'actual_cst'")
    cumulative_slope_source = np.cumsum(np.nan_to_num(slope_source_trace, nan=0.0)) * dt

    slope_rows = []
    max_slope = -np.inf
    selected = None
    start_values = np.arange(0.0, max(0.0, float(max_slope_horizon_s) - float(max_slope_window_s)) + 1e-12, float(max_slope_step_s))
    for start_s in start_values:
        end_s = float(start_s + float(max_slope_window_s))
        mask = (time_rel >= start_s) & (time_rel <= end_s)
        if int(np.sum(mask)) < 2:
            slope, intercept, r2 = np.nan, np.nan, np.nan
            status = 'insufficient_data'
        else:
            slope, intercept, r2 = _response_fit_line(time_rel[mask], cumulative_slope_source[mask])
            status = 'valid' if np.isfinite(slope) else 'invalid_fit'
        slope_row = {
            'response_row_index': int(row.name),
            'parameter_set_id': actual_id,
            'realization_seed': seed,
            'pulse_input_amplitude_hz': float(row['pulse_input_amplitude_hz']),
            'candidate_slope_source': slope_source_name,
            'candidate_window_start_s': float(start_s),
            'candidate_window_end_s': float(end_s),
            'candidate_slope_spikes_per_s_per_mn': slope,
            'candidate_intercept_spikes_per_mn': intercept,
            'candidate_r2': r2,
            'candidate_status': status,
        }
        slope_rows.append(slope_row)
        if np.isfinite(slope) and slope > max_slope:
            max_slope = slope
            selected = slope_row
    if selected is None:
        max_slope_status = 'insufficient_data'
        selected = {
            'candidate_window_start_s': np.nan,
            'candidate_window_end_s': np.nan,
            'candidate_slope_spikes_per_s_per_mn': np.nan,
            'candidate_intercept_spikes_per_mn': np.nan,
            'candidate_r2': np.nan,
        }
    else:
        max_slope_status = 'valid'

    metric_row = row.to_dict()
    metric_row.update({
        'response_row_index': int(row.name),
        'response_match_status': 'matched_zero_reference',
        'matched_no_step_parameter_set_id': zero_id,
        'matched_no_step_realization_seed': seed,
        'response_step_sign': step_sign,
        'response_baseline_cst_effect_median_spikes_per_s_per_mn': baseline_offset,
        'reaction_epsilon_extra_spikes_per_mn': float(epsilon),
        'reaction_time_to_epsilon_s': reaction_time,
        'reaction_time_status': reaction_status,
        'response_target_fraction_of_input_rate': target_fraction,
        'response_target_input_delta_hz': target_input_delta_hz,
        'response_target_rate_source': target_source_name,
        'response_target_threshold_spikes_per_s_per_mn': target_threshold,
        'response_time_to_target_fraction_s': target_fraction_time,
        'response_time_to_target_fraction_status': target_fraction_status,
        'max_early_cumulative_slope_source': slope_source_name,
        'max_early_cumulative_slope_spikes_per_s_per_mn': selected['candidate_slope_spikes_per_s_per_mn'],
        'max_early_cumulative_slope_window_start_s': selected['candidate_window_start_s'],
        'max_early_cumulative_slope_window_end_s': selected['candidate_window_end_s'],
        'max_early_cumulative_slope_r2': selected['candidate_r2'],
        'max_early_cumulative_slope_status': max_slope_status,
        'response_trace_dt_s': dt,
        'response_trace_n_samples': int(time_rel.size),
        'pre_step_cumulative_output_effect_mean_abs_spikes_per_mn': float(np.nanmean(np.abs(np.cumsum(np.nan_to_num(sign_corrected_effect[t < pulse_start_s], nan=0.0)) * dt))) if np.any(t < pulse_start_s) else np.nan,
    })
    for horizon_s in capture_horizons_s:
        label = f'{int(round(float(horizon_s) * 1000)):03d}ms'
        out_value = _response_value_at(time_rel, cumulative_output, horizon_s)
        in_value = _response_value_at(time_rel, cumulative_input, horizon_s)
        metric_row[f'cumulative_output_effect_{label}'] = out_value
        metric_row[f'cumulative_input_instruction_{label}'] = in_value
        metric_row[f'cumulative_capture_{label}'] = out_value / in_value if np.isfinite(out_value) and np.isfinite(in_value) and abs(in_value) > 1e-12 else np.nan

    trace_payload = {
        'response_row_index': int(row.name),
        'parameter_set_id': actual_id,
        'realization_seed': seed,
        'pulse_input_amplitude_hz': float(row['pulse_input_amplitude_hz']),
        'response_time_rel_s': time_rel,
        'input_instruction_rate_hz': input_post,
        'sign_corrected_baseline_corrected_cst_effect_spikes_per_s_mu': effect_post,
        'target_rate_trace_spikes_per_s_mu': target_rate_trace,
        'cumulative_output_effect_spikes_per_mn': cumulative_output,
        'cumulative_input_instruction_spikes_per_mn': cumulative_input,
        'cumulative_slope_source_spikes_per_mn': cumulative_slope_source,
    }
    return metric_row, trace_payload, slope_rows



def build_trace_batch_response_metrics(
    matched_zero_df,
    *,
    output_path,
    epsilon=1.0,
    target_fraction_of_input_rate=0.50,
    target_rate_source='matched_effect_smooth',
    baseline_window_rel_s=(-0.5, 0.0),
    max_slope_source='matched_effect',
    max_slope_window_s=0.050,
    max_slope_horizon_s=0.200,
    max_slope_step_s=0.010,
    capture_horizons_s=(0.050, 0.100, 0.200, 0.500),
):
    nonzero = matched_zero_df[
        (pd.to_numeric(matched_zero_df['pulse_input_amplitude_hz'], errors='coerce').abs() > 1e-12)
        & (matched_zero_df['zero_ref_match_status'] == 'matched_zero_reference')
    ].copy()
    if nonzero.empty:
        raise ValueError('No matched nonzero pulse rows available for response metrics')
    rows = []
    traces = []
    slope_rows = []
    trace_cache = {}
    iterator = nonzero.iterrows()
    if tqdm is not None:
        iterator = tqdm(list(iterator), desc='Response metrics from matched compact traces', unit='sim')
    for original_index, row in iterator:
        row = row.copy()
        row.name = int(original_index)
        metric_row, trace_payload, slope_payload = _compute_response_for_row(
            row,
            output_path=output_path,
            epsilon=epsilon,
            target_fraction_of_input_rate=target_fraction_of_input_rate,
            target_rate_source=target_rate_source,
            baseline_window_rel_s=baseline_window_rel_s,
            max_slope_source=max_slope_source,
            max_slope_window_s=max_slope_window_s,
            max_slope_horizon_s=max_slope_horizon_s,
            max_slope_step_s=max_slope_step_s,
            capture_horizons_s=capture_horizons_s,
            trace_cache=trace_cache,
        )
        rows.append(metric_row)
        traces.append(trace_payload)
        slope_rows.extend(slope_payload)
    metrics_df = pd.DataFrame(rows)
    slopes_df = pd.DataFrame(slope_rows)
    return metrics_df, traces, slopes_df



def _stack_response_trace_arrays(trace_payloads):
    if not trace_payloads:
        return {}
    ref_t = np.asarray(trace_payloads[0]['response_time_rel_s'], dtype=float)
    out = {
        'response_row_index': np.asarray([p['response_row_index'] for p in trace_payloads], dtype=int),
        'parameter_set_id': np.asarray([p['parameter_set_id'] for p in trace_payloads], dtype=int),
        'realization_seed': np.asarray([p['realization_seed'] for p in trace_payloads], dtype=int),
        'pulse_input_amplitude_hz': np.asarray([p['pulse_input_amplitude_hz'] for p in trace_payloads], dtype=float),
        'response_time_rel_s': ref_t,
    }
    for key in RESPONSE_TRACE_KEYS[1:]:
        out[key] = np.vstack([
            _response_interp_to_time(p['response_time_rel_s'], p[key], ref_t)
            for p in trace_payloads
        ])
    return out


def _load_response_trace_for_row(trace_arrays, response_row_index):
    ids = np.asarray(trace_arrays['response_row_index'], dtype=int)
    matches = np.flatnonzero(ids == int(response_row_index))
    if matches.size != 1:
        raise ValueError(f'Expected one response trace for row {response_row_index}, found {matches.size}')
    idx = int(matches[0])
    return {key: (value[idx] if np.asarray(value).ndim > 1 else value) for key, value in trace_arrays.items()}


def plot_baseline_synchrony_vs_pool_age(response_or_matched_df, *, output_path=None):
    age_col = 'zero_ref_pool_integrated_age_mean' if 'zero_ref_pool_integrated_age_mean' in response_or_matched_df.columns else 'pool_integrated_age_mean'
    if age_col not in response_or_matched_df.columns or response_or_matched_df[age_col].notna().sum() == 0:
        warnings.warn('Pool age/readiness columns are not available in this batch. Rerun trace batch after age saving to enable this plot.')
        return None, None
    df = response_or_matched_df.copy()
    zero_df = df[np.isclose(pd.to_numeric(df['pulse_input_amplitude_hz'], errors='coerce'), 0.0, atol=1e-12)].copy()
    if zero_df.empty:
        zero_df = df.drop_duplicates(['zero_ref_parameter_set_id', 'zero_ref_realization_seed']).copy()
    fig, ax = plt.subplots(figsize=(6, 4.5), constrained_layout=True)
    colors = pd.to_numeric(zero_df.get('common_copy_probability', np.nan), errors='coerce')
    sc = ax.scatter(
        zero_df['baseline_sync_rate_controlled_mean'],
        zero_df[age_col],
        c=colors,
        cmap='viridis',
        s=24,
        alpha=0.75,
    )
    ax.set_xlabel('matched zero-pulse baseline rate-normalized synchrony')
    ax.set_ylabel('pool integrated age at pulse time\n(expected spikes)')
    ax.set_title('Baseline synchrony vs pool readiness')
    ax.grid(alpha=0.18)
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label('common_copy_probability')
    if output_path is not None:
        _save_figure_formats(fig, Path(output_path) / 'baseline_synchrony_vs_pool_age', TRACE_BATCH_SAVE_FIGURE_FORMATS)
    return fig, ax


def plot_response_diagnostic(metrics_df, trace_arrays, slopes_df, *, response_row_index=None, xlim_rel_s=None, output_path=None):
    if response_row_index is None:
        response_row_index = int(metrics_df.iloc[0]['response_row_index'])
    row = metrics_df.loc[metrics_df['response_row_index'] == int(response_row_index)].iloc[0]
    trace = _load_response_trace_for_row(trace_arrays, response_row_index)
    t = np.asarray(trace['response_time_rel_s'], dtype=float)
    fig, axes = plt.subplots(4, 1, figsize=(10, 10.5), sharex=True, constrained_layout=True)
    subtitle = (
        f"param_set={int(row['parameter_set_id'])}, iter={int(row.get('realization_index', -1))}, "
        # f"seed={int(row['realization_seed'])}, common_fraction={row.get('common_fraction', np.nan):.3g}, "
        f"seed={int(row['common_copy_probability'])}, common_copy_probability={row.get('common_copy_probability', np.nan):.3g}, "
        f"pulse={row['pulse_input_amplitude_hz']:.3g} Hz"
    )
    axes[0].plot(t, trace['input_instruction_rate_hz'], color='black', lw=1.5)
    axes[0].axhline(0, color='0.4', ls='--', lw=0.8)
    axes[0].set_ylabel('input diff\n(Hz)')
    axes[0].set_title(f'Matched pulse response diagnostic\\n{subtitle}', fontsize=10)

    axes[1].plot(t, trace['sign_corrected_baseline_corrected_cst_effect_spikes_per_s_mu'], color='#ff3300', lw=1.0, alpha=0.55, label='matched raw CST effect')
    target_trace = trace.get('target_rate_trace_spikes_per_s_mu', trace['sign_corrected_baseline_corrected_cst_effect_spikes_per_s_mu'])
    axes[1].plot(t, target_trace, color='#0072b2', lw=1.7, label=f"target RT source: {row.get('response_target_rate_source', 'matched_effect')}")
    target_threshold = row.get('response_target_threshold_spikes_per_s_per_mn', np.nan)
    target_fraction = row.get('response_target_fraction_of_input_rate', np.nan)
    if np.isfinite(target_threshold):
        axes[1].axhline(
            target_threshold,
            color='#0072b2',
            ls='--',
            lw=1.1,
            label=f'{100 * target_fraction:.0f}% input-rate target',
        )
    if np.isfinite(row.get('response_time_to_target_fraction_s', np.nan)):
        axes[1].axvline(row['response_time_to_target_fraction_s'], color='#0072b2', lw=1.0, alpha=0.9)
    axes[1].axhline(0, color='0.4', ls='--', lw=0.8)
    axes[1].set_ylabel('sign-corrected\nCST effect (sp/s/MN)')
    axes[1].legend(fontsize=8, loc='best')

    axes[2].plot(t, trace['cumulative_output_effect_spikes_per_mn'], color='#ff3300', lw=1.8, label='cumulative matched output effect')
    axes[2].plot(t, trace['cumulative_input_instruction_spikes_per_mn'], color='black', ls='--', lw=1.4, label='cumulative instructed input')
    slope_source = str(row.get('max_early_cumulative_slope_source', 'matched_effect'))
    slope_trace = trace.get('cumulative_slope_source_spikes_per_mn', trace['cumulative_output_effect_spikes_per_mn'])
    if slope_source != 'matched_effect':
        axes[2].plot(t, slope_trace, color='0.45', lw=1.2, alpha=0.7, label=f'slope source: {slope_source}')
    eps = float(row['reaction_epsilon_extra_spikes_per_mn'])
    axes[2].axhline(eps, color='#2ca02c', ls='--', lw=1.0, label=f'epsilon={eps:g}')
    if np.isfinite(row['reaction_time_to_epsilon_s']):
        axes[2].axvline(row['reaction_time_to_epsilon_s'], color='#2ca02c', lw=1.0)
    for horizon_s in RESPONSE_CAPTURE_HORIZONS_S:
        axes[2].axvline(float(horizon_s), color='0.65', lw=0.8, alpha=0.6)
    if np.isfinite(row['max_early_cumulative_slope_window_start_s']):
        start_s = float(row['max_early_cumulative_slope_window_start_s'])
        end_s = float(row['max_early_cumulative_slope_window_end_s'])
        axes[2].axvspan(start_s, end_s, color='#ffbb00', alpha=0.18, lw=0)
        slope = float(row['max_early_cumulative_slope_spikes_per_s_per_mn'])
        fit_mask = (t >= start_s) & (t <= end_s)
        if np.any(fit_mask):
            y0 = _response_value_at(t, slope_trace, start_s)
            axes[2].plot(t[fit_mask], y0 + slope * (t[fit_mask] - start_s), color='#ffbb00', lw=2.0, label='selected max-slope fit')
    axes[2].set_ylabel('cumulative\nspikes/MN')
    axes[2].legend(fontsize=8, loc='best')

    sub_slopes = slopes_df[slopes_df['response_row_index'] == int(response_row_index)].copy()
    axes[3].plot(
        sub_slopes['candidate_window_start_s'],
        sub_slopes['candidate_slope_spikes_per_s_per_mn'],
        color='#852680',
        marker='o',
        ms=3,
        lw=1.2,
    )
    axes[3].set_ylabel('candidate slope\n(sp/s/MN)')
    axes[3].set_xlabel('time after pulse onset (s)')
    axes[3].set_title(f'max-slope candidates from {slope_source}', fontsize=9)
    annotation = f""
    # annotation = (
    #     f"Debug example: parameter_set={int(row['parameter_set_id'])}, iter={int(row.get('realization_index', -1))}, seed={int(row['realization_seed'])}\\n"
    #     f"common_copy_probability={row.get('common_copy_probability', np.nan):.3g}; pulse={row['pulse_input_amplitude_hz']:.3g} Hz\\n"
    #     f"baseline sync={row.get('baseline_sync_rate_controlled_mean', np.nan):.3g}; "
    #     f"absolute sync={row.get('baseline_sync_absolute_mean', np.nan):.3g} /s/pair\\n"
    #     f"RT eps={row['reaction_time_to_epsilon_s']:.3g}s ({row['reaction_time_status']})\\n"
    #     f"RT {100 * row.get('response_target_fraction_of_input_rate', np.nan):.0f}% input target="
    #     f"{row.get('response_time_to_target_fraction_s', np.nan):.3g}s "
    #     f"({row.get('response_time_to_target_fraction_status', '')}; {row.get('response_target_rate_source', '')})\\n"
    #     f"max slope={row['max_early_cumulative_slope_spikes_per_s_per_mn']:.3g} sp/s/MN [{slope_source}]\\n"
    #     f"capture 200ms={row.get('cumulative_capture_200ms', np.nan):.3g}"
    # )
    if 'zero_ref_pool_integrated_age_mean' in row.index and np.isfinite(row.get('zero_ref_pool_integrated_age_mean', np.nan)):
        annotation += f"\\npool age={row['zero_ref_pool_integrated_age_mean']:.3g}"
    axes[2].text(
        0.98,
        0.04,
        annotation,
        transform=axes[2].transAxes,
        ha='right',
        va='bottom',
        fontsize=8,
        bbox={'facecolor': 'white', 'edgecolor': '0.75', 'alpha': 0.9},
    )
    for ax in axes:
        ax.grid(alpha=0.18)
    if xlim_rel_s is not None:
        axes[-1].set_xlim(float(xlim_rel_s[0]), float(xlim_rel_s[1]))
    if output_path is not None:
        _save_figure_formats(fig, Path(output_path) / 'trace_batch_response_diagnostic', TRACE_BATCH_SAVE_FIGURE_FORMATS)
    return fig, axes



def _response_capture_column(horizon_s):
    label = f"{int(round(float(horizon_s) * 1000)):03d}ms"
    return f'cumulative_capture_{label}', label


def _response_pretty_label(column_name):
    capture_col, capture_label = _response_capture_column(RESPONSE_SCATTER_CAPTURE_HORIZON_S)
    labels = {
        'baseline_sync_rate_controlled_mean': 'baseline rate-normalized sync',
        'reaction_time_to_epsilon_s': 'reaction time to epsilon (s)',
        'max_early_cumulative_slope_spikes_per_s_per_mn': 'max early cumulative slope (sp/s/MN)',
        capture_col: f'cumulative capture at {capture_label.replace("ms", " ms")}',
        'zero_ref_pool_integrated_age_mean': 'pool integrated age (expected spikes)',
    }
    return labels.get(column_name, str(column_name).replace('_', ' '))


def _response_common_fraction_colors(values, cmap_name='viridis'):
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return plt.get_cmap(cmap_name), Normalize(vmin=0.0, vmax=1.0)
    vmin = float(np.min(finite))
    vmax = float(np.max(finite))
    if np.isclose(vmin, vmax):
        pad = max(0.05, abs(vmin) * 0.05)
        vmin -= pad
        vmax += pad
    return plt.get_cmap(cmap_name), Normalize(vmin=vmin, vmax=vmax)


def _response_linear_fit(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    if int(np.sum(finite)) < 3 or float(np.std(x[finite], ddof=0)) <= 0:
        return np.nan, np.nan, np.nan
    slope, intercept = np.polyfit(x[finite], y[finite], 1)
    y_hat = slope * x[finite] + intercept
    ss_res = float(np.sum((y[finite] - y_hat) ** 2))
    ss_tot = float(np.sum((y[finite] - float(np.mean(y[finite]))) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return float(slope), float(intercept), float(r2)


def _add_regression_line(ax, x, y, *, color='black', linestyle=':', linewidth=1.6, alpha=0.9, label=None, zorder=4):
    slope, intercept, r2 = _response_linear_fit(x, y)
    if not np.isfinite(slope):
        return np.nan
    x = np.asarray(x, dtype=float)
    finite_x = x[np.isfinite(x)]
    if finite_x.size < 2:
        return np.nan
    x_line = np.linspace(float(np.min(finite_x)), float(np.max(finite_x)), 80)
    ax.plot(
        x_line,
        slope * x_line + intercept,
        color=color,
        linestyle=linestyle,
        linewidth=linewidth,
        alpha=alpha,
        label=label,
        zorder=zorder,
    )
    return r2


def _add_group_mean_markers(ax, df, x_col, y_col, *, group_col, cmap, norm):
    if group_col not in df.columns:
        return
    for group_value, group_df in df.groupby(group_col, dropna=True):
        x = pd.to_numeric(group_df[x_col], errors='coerce')
        y = pd.to_numeric(group_df[y_col], errors='coerce')
        finite = np.isfinite(x) & np.isfinite(y)
        if int(np.sum(finite)) == 0:
            continue
        x = x[finite]
        y = y[finite]
        x_mean = float(np.mean(x))
        y_mean = float(np.mean(y))
        x_sd = float(np.std(x, ddof=0)) if len(x) > 1 else 0.0
        y_sd = float(np.std(y, ddof=0)) if len(y) > 1 else 0.0
        try:
            color_value = float(group_value)
            face_color = cmap(norm(color_value))
        except Exception:
            face_color = 'white'
        ax.errorbar(
            x_mean,
            y_mean,
            xerr=x_sd,
            yerr=y_sd,
            fmt='none',
            ecolor='black',
            elinewidth=1.1,
            capsize=2.5,
            alpha=float(RESPONSE_SCATTER_MEAN_ERRORBAR_ALPHA),
            zorder=5,
        )
        ax.scatter(
            [x_mean],
            [y_mean],
            s=float(RESPONSE_SCATTER_MEAN_MARKER_SIZE),
            marker='o',
            facecolor=face_color,
            edgecolor='black',
            linewidth=1.0,
            zorder=6,
        )


def _add_age_group_regression_lines(ax, df, x_col, y_col, *, group_col, cmap, norm):
    if group_col not in df.columns:
        return []
    labels = []
    for group_value, group_df in df.groupby(group_col, dropna=True):
        x = pd.to_numeric(group_df[x_col], errors='coerce')
        y = pd.to_numeric(group_df[y_col], errors='coerce')
        try:
            color_value = float(group_value)
            color = cmap(norm(color_value))
            label_value = f'{color_value:g}'
        except Exception:
            color = '0.35'
            label_value = str(group_value)
        r2 = _add_regression_line(
            ax,
            x,
            y,
            color=color,
            linestyle='-',
            linewidth=1.8,
            alpha=0.95,
            zorder=5,
        )
        if np.isfinite(r2):
            labels.append(f'{group_col}={label_value}: R2={r2:.2f}')
    return labels


def plot_response_global_scatters(metrics_df, *, output_path=None):
    capture_col, capture_label = _response_capture_column(RESPONSE_SCATTER_CAPTURE_HORIZON_S)
    specs = [
        ('baseline_sync_rate_controlled_mean', 'reaction_time_to_epsilon_s', 'baseline sync vs reaction time'),
        ('baseline_sync_rate_controlled_mean', 'max_early_cumulative_slope_spikes_per_s_per_mn', 'baseline sync vs max early slope'),
        ('baseline_sync_rate_controlled_mean', capture_col, f'baseline sync vs capture {capture_label.replace("ms", " ms")}'),
        ('baseline_sync_rate_controlled_mean', 'zero_ref_pool_integrated_age_mean', 'baseline sync vs pool age'),
        ('zero_ref_pool_integrated_age_mean', 'reaction_time_to_epsilon_s', 'pool age vs reaction time'),
        ('zero_ref_pool_integrated_age_mean', 'max_early_cumulative_slope_spikes_per_s_per_mn', 'pool age vs max early slope'),
    ]
    df = metrics_df.copy()
    if 'pulse_input_amplitude_hz' not in df.columns:
        raise ValueError('metrics_df must contain pulse_input_amplitude_hz')
    if capture_col not in df.columns:
        available_capture_cols = sorted(c for c in df.columns if str(c).startswith('cumulative_capture_'))
        raise ValueError(
            f'Requested RESPONSE_SCATTER_CAPTURE_HORIZON_S={RESPONSE_SCATTER_CAPTURE_HORIZON_S:g}s, '
            f'but column {capture_col!r} is missing. Available capture columns: {available_capture_cols}'
        )
    df = df[pd.to_numeric(df['pulse_input_amplitude_hz'], errors='coerce').fillna(0.0) != 0.0].copy()
    available = [
        (x, y, title)
        for x, y, title in specs
        if x in df.columns and y in df.columns and df[x].notna().sum() > 0 and df[y].notna().sum() > 0
    ]
    if not available or df.empty:
        warnings.warn('No response scatter specs have finite x/y values.')
        return None, None

    pulse_values = sorted(float(v) for v in pd.unique(pd.to_numeric(df['pulse_input_amplitude_hz'], errors='coerce')) if np.isfinite(v) and float(v) != 0.0)
    n_rows = len(pulse_values)
    n_cols = len(available)
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4.35 * n_cols, 3.65 * n_rows),
        squeeze=False,
        constrained_layout=True,
    )

    color_values = pd.to_numeric(df.get(RESPONSE_FIGURE_COLOR_BY, np.nan), errors='coerce')
    cmap, norm = _response_common_fraction_colors(color_values)
    scalar_mappable = ScalarMappable(norm=norm, cmap=cmap)
    scalar_mappable.set_array([])

    x_limits = {}
    y_limits = {}
    for x_col, y_col, _ in available:
        x = pd.to_numeric(df[x_col], errors='coerce')
        y = pd.to_numeric(df[y_col], errors='coerce')
        if np.isfinite(x).any():
            lo, hi = float(np.nanmin(x)), float(np.nanmax(x))
            pad = max(1e-9, (hi - lo) * 0.06)
            x_limits[x_col] = (lo - pad, hi + pad)
        if np.isfinite(y).any():
            lo, hi = float(np.nanmin(y)), float(np.nanmax(y))
            pad = max(1e-9, (hi - lo) * 0.06)
            y_limits[y_col] = (lo - pad, hi + pad)

    for row_idx, pulse_value in enumerate(pulse_values):
        pulse_df = df[np.isclose(pd.to_numeric(df['pulse_input_amplitude_hz'], errors='coerce'), pulse_value)].copy()
        for col_idx, (x_col, y_col, title) in enumerate(available):
            ax = axes[row_idx, col_idx]
            x = pd.to_numeric(pulse_df[x_col], errors='coerce')
            y = pd.to_numeric(pulse_df[y_col], errors='coerce')
            c = pd.to_numeric(pulse_df[RESPONSE_FIGURE_COLOR_BY], errors='coerce') if RESPONSE_FIGURE_COLOR_BY in pulse_df.columns else pd.Series(np.nan, index=pulse_df.index)
            finite = np.isfinite(x) & np.isfinite(y)
            ax.scatter(
                x[finite],
                y[finite],
                c=c[finite],
                cmap=cmap,
                norm=norm,
                s=18,
                alpha=0.58,
                edgecolors='none',
                zorder=2,
            )

            global_r2 = np.nan
            if bool(RESPONSE_SCATTER_SHOW_GLOBAL_REGRESSION):
                global_r2 = _add_regression_line(
                    ax,
                    x[finite],
                    y[finite],
                    color='black',
                    linestyle=':',
                    linewidth=1.8,
                    alpha=0.95,
                    zorder=4,
                )

            is_age_x = str(x_col).startswith('zero_ref_pool_') or 'pool_integrated_age' in str(x_col) or 'pool_raw_age' in str(x_col)
            age_group_labels = []
            if is_age_x and bool(RESPONSE_SCATTER_SHOW_AGE_GROUP_REGRESSIONS):
                age_group_labels = _add_age_group_regression_lines(
                    ax,
                    pulse_df,
                    x_col,
                    y_col,
                    group_col=RESPONSE_SCATTER_GROUP_COL,
                    cmap=cmap,
                    norm=norm,
                )
            elif bool(RESPONSE_SCATTER_SHOW_GROUP_MEANS):
                _add_group_mean_markers(
                    ax,
                    pulse_df,
                    x_col,
                    y_col,
                    group_col=RESPONSE_SCATTER_GROUP_COL,
                    cmap=cmap,
                    norm=norm,
                )

            text_lines = []
            if np.isfinite(global_r2):
                text_lines.append(f'all: R2={global_r2:.2f}')
            if age_group_labels:
                text_lines.extend(age_group_labels[:5])
                if len(age_group_labels) > 5:
                    text_lines.append(f'+{len(age_group_labels) - 5} groups')
            if text_lines:
                ax.text(
                    0.03,
                    0.97,
                    '\n'.join(text_lines),
                    transform=ax.transAxes,
                    ha='right', # 'left',
                    va='top',
                    fontsize=6,
                    bbox={'facecolor': 'white', 'edgecolor': '0.78', 'alpha': 0.6},
                    zorder=8,
                )

            if row_idx == 0:
                ax.set_title(title, fontsize=9)
            if col_idx == 0:
                ax.text(
                    -0.22,
                    0.5,
                    f'pulse {pulse_value:g} Hz',
                    transform=ax.transAxes,
                    rotation=90,
                    ha='center',
                    va='center',
                    fontsize=10,
                    fontweight='bold',
                )
            ax.set_xlabel(_response_pretty_label(x_col))
            ax.set_ylabel(_response_pretty_label(y_col))
            if x_col in x_limits:
                ax.set_xlim(*x_limits[x_col])
            if y_col in y_limits:
                ax.set_ylim(*y_limits[y_col])
            ax.grid(alpha=0.18)

    cbar = fig.colorbar(scalar_mappable, ax=axes.ravel().tolist(), shrink=0.88, pad=0.01)
    cbar.set_label(RESPONSE_FIGURE_COLOR_BY)
    if output_path is not None:
        _save_figure_formats(fig, Path(output_path) / 'trace_batch_response_global_scatters_by_pulse', TRACE_BATCH_SAVE_FIGURE_FORMATS)
    return fig, axes


if BUILD_TRACE_BATCH_RESPONSE_METRICS:
    trace_batch_output_path = make_trace_batch_output_path(TRACE_BATCH_OUTPUT_DIR, TRACE_BATCH_RUN_NAME)
    if 'trace_batch_matched_zero_baseline_df' not in globals() or trace_batch_matched_zero_baseline_df is None or len(trace_batch_matched_zero_baseline_df) == 0:
        matched_path = trace_batch_output_path / 'trace_batch_matched_zero_baseline_table.pkl'
        if matched_path.exists():
            trace_batch_matched_zero_baseline_df = pd.read_pickle(matched_path)
        else:
            if 'trace_batch_realization_df' not in globals() or trace_batch_realization_df is None or len(trace_batch_realization_df) == 0:
                loaded_trace_batch_output = load_trace_batch_outputs(trace_batch_output_path)
                trace_batch_realization_df = loaded_trace_batch_output['realization_df']
                trace_batch_parameter_set_df = loaded_trace_batch_output['parameter_set_df']
                trace_batch_config_snapshot = loaded_trace_batch_output['config_snapshot']
            trace_batch_matched_zero_baseline_df = build_trace_batch_matched_zero_baseline_table(
                trace_batch_realization_df,
                output_path=trace_batch_output_path,
                zero_amplitude_hz=TRACE_BATCH_MATCHED_ZERO_AMPLITUDE_HZ,
                require_all_matches=TRACE_BATCH_MATCHED_ZERO_REQUIRE_ALL_MATCHES,
            )
    trace_batch_response_metrics_df, trace_batch_response_trace_payloads, trace_batch_response_slope_candidates_df = build_trace_batch_response_metrics(
        trace_batch_matched_zero_baseline_df,
        output_path=trace_batch_output_path,
        epsilon=RESPONSE_EPSILON_EXTRA_SPIKES_PER_MN,
        target_fraction_of_input_rate=RESPONSE_TARGET_FRACTION_OF_INPUT_RATE,
        target_rate_source=RESPONSE_TARGET_RATE_SOURCE,
        baseline_window_rel_s=RESPONSE_BASELINE_CORRECTION_WINDOW_REL_S,
        max_slope_source=RESPONSE_MAX_SLOPE_SOURCE,
        max_slope_window_s=RESPONSE_MAX_SLOPE_WINDOW_S,
        max_slope_horizon_s=RESPONSE_MAX_SLOPE_SEARCH_HORIZON_S,
        max_slope_step_s=RESPONSE_MAX_SLOPE_STEP_S,
        capture_horizons_s=RESPONSE_CAPTURE_HORIZONS_S,
    )
    trace_batch_response_trace_arrays = _stack_response_trace_arrays(trace_batch_response_trace_payloads)
    if TRACE_BATCH_RESPONSE_SAVE_OUTPUTS:
        metrics_pkl = trace_batch_output_path / 'trace_batch_response_metrics.pkl'
        metrics_csv = trace_batch_output_path / 'trace_batch_response_metrics.csv'
        slopes_pkl = trace_batch_output_path / 'trace_batch_response_slope_candidates.pkl'
        slopes_csv = trace_batch_output_path / 'trace_batch_response_slope_candidates.csv'
        arrays_path = trace_batch_output_path / 'trace_batch_response_trace_arrays.npz'
        trace_batch_response_metrics_df.to_pickle(metrics_pkl)
        trace_batch_response_metrics_df.to_csv(metrics_csv, index=False)
        trace_batch_response_slope_candidates_df.to_pickle(slopes_pkl)
        trace_batch_response_slope_candidates_df.to_csv(slopes_csv, index=False)
        np.savez_compressed(arrays_path, **trace_batch_response_trace_arrays)
    if PLOT_TRACE_BATCH_RESPONSE_FIGURES:
        response_figure_paths = []
        age_fig, age_ax = plot_baseline_synchrony_vs_pool_age(trace_batch_matched_zero_baseline_df, output_path=trace_batch_output_path if TRACE_BATCH_RESPONSE_SAVE_OUTPUTS else None)
        diagnostic_fig, diagnostic_axes = plot_response_diagnostic(
            trace_batch_response_metrics_df,
            trace_batch_response_trace_arrays,
            trace_batch_response_slope_candidates_df,
            response_row_index=RESPONSE_DIAGNOSTIC_ROW_INDEX,
            xlim_rel_s=RESPONSE_DIAGNOSTIC_XLIM_REL_S,
            output_path=trace_batch_output_path if TRACE_BATCH_RESPONSE_SAVE_OUTPUTS else None,
        )
        scatter_fig, scatter_axes = plot_response_global_scatters(trace_batch_response_metrics_df, output_path=trace_batch_output_path if TRACE_BATCH_RESPONSE_SAVE_OUTPUTS else None)
        plt.show()
    if TRACE_BATCH_RESPONSE_SAVE_OUTPUTS:
        save_trace_batch_outputs(
            trace_batch_output_path,
            realization_df=trace_batch_realization_df if 'trace_batch_realization_df' in globals() else None,
            parameter_set_df=trace_batch_parameter_set_df if 'trace_batch_parameter_set_df' in globals() else None,
            config_snapshot=trace_batch_config_snapshot if 'trace_batch_config_snapshot' in globals() else collect_trace_batch_config_snapshot(),
            manifest_updates={
                'stage': 'trace_batch_response_metrics_saved',
                'response_metrics_pkl': str(metrics_pkl),
                'response_metrics_csv': str(metrics_csv),
                'response_trace_arrays_npz': str(arrays_path),
                'response_slope_candidates_pkl': str(slopes_pkl),
                'response_slope_candidates_csv': str(slopes_csv),
                'response_rows': int(len(trace_batch_response_metrics_df)),
                'reaction_time_status_counts': trace_batch_response_metrics_df['reaction_time_status'].value_counts(dropna=False).to_dict(),
            },
        )
    display(Markdown(
        f'Response metrics table: {len(trace_batch_response_metrics_df)} nonzero pulse rows; '
        f"statuses: {trace_batch_response_metrics_df['reaction_time_status'].value_counts(dropna=False).to_dict()}"
    ))
    display(trace_batch_response_metrics_df.head())
else:
    display(Markdown('Trace-batch response metrics are disabled. Set `BUILD_TRACE_BATCH_RESPONSE_METRICS=True` to generate them.'))


## Paper Synchrony-Response Scatter

This figure is a paper-oriented extraction of the trace-batch response analysis: baseline rate-normalized synchrony on the x-axis and maximum cumulative CST slope on the y-axis. It reloads cached response metrics when needed and has its own visual style controls in the editable configuration cell.


In [ ]:

def _paper_sync_slope_load_response_metrics():
    if 'trace_batch_response_metrics_df' in globals() and trace_batch_response_metrics_df is not None and len(trace_batch_response_metrics_df):
        return trace_batch_response_metrics_df.copy()
    trace_batch_output_path = make_trace_batch_output_path(TRACE_BATCH_OUTPUT_DIR, TRACE_BATCH_RUN_NAME)
    metrics_path = trace_batch_output_path / 'trace_batch_response_metrics.pkl'
    if not metrics_path.exists():
        raise FileNotFoundError(
            f'Could not find cached response metrics at {metrics_path}. '
            'Run the pulse-response metrics cell first, or enable BUILD_TRACE_BATCH_RESPONSE_METRICS.'
        )
    return pd.read_pickle(metrics_path)


def _paper_sync_slope_colormap(style):
    colors = style.get('group_colors', ['#f4e9f7', '#d78bca', '#ad4da6', '#852680', '#4d145f'])
    return LinearSegmentedColormap.from_list('paper_sync_pcopy', list(colors))


def _paper_sync_slope_finite_limits(values, pad_fraction=0.08):
    values = pd.to_numeric(pd.Series(values), errors='coerce').to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return None
    lo = float(np.min(values))
    hi = float(np.max(values))
    if np.isclose(lo, hi):
        pad = max(1e-9, abs(lo) * float(pad_fraction), 1e-6)
    else:
        pad = (hi - lo) * float(pad_fraction)
    return lo - pad, hi + pad


def _paper_sync_slope_regression(x, y):
    x = pd.to_numeric(pd.Series(x), errors='coerce').to_numpy(dtype=float)
    y = pd.to_numeric(pd.Series(y), errors='coerce').to_numpy(dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    if int(np.sum(finite)) < 2:
        return np.nan, np.nan, np.nan, np.nan
    x = x[finite]
    y = y[finite]
    if np.isclose(np.nanstd(x), 0) or np.isclose(np.nanstd(y), 0):
        return np.nan, np.nan, np.nan, np.nan
    slope, intercept = np.polyfit(x, y, 1)
    r = float(np.corrcoef(x, y)[0, 1])
    return float(slope), float(intercept), r, float(r * r)


def plot_paper_sync_vs_max_slope_scatter(metrics_df=None, *, output_path=None, style=None):
    style = dict(PAPER_SYNC_SLOPE_SCATTER_STYLE if style is None else style)
    df = _paper_sync_slope_load_response_metrics() if metrics_df is None else metrics_df.copy()
    x_col = str(PAPER_SYNC_SLOPE_SCATTER_X_COL)
    y_col = str(PAPER_SYNC_SLOPE_SCATTER_Y_COL)
    group_col = str(PAPER_SYNC_SLOPE_SCATTER_GROUP_COL)
    required = {'pulse_input_amplitude_hz', x_col, y_col, group_col}
    missing = sorted(required.difference(df.columns))
    if missing:
        raise KeyError(f'Response metrics table is missing required column(s): {missing}')

    df = df.copy()
    df['pulse_input_amplitude_hz'] = pd.to_numeric(df['pulse_input_amplitude_hz'], errors='coerce')
    df[x_col] = pd.to_numeric(df[x_col], errors='coerce')
    df[y_col] = pd.to_numeric(df[y_col], errors='coerce')
    df[group_col] = pd.to_numeric(df[group_col], errors='coerce')
    df = df[np.isfinite(df['pulse_input_amplitude_hz']) & (np.abs(df['pulse_input_amplitude_hz']) > 1e-12)].copy()

    requested_pulses = PAPER_SYNC_SLOPE_SCATTER_PULSE_VALUES_HZ
    if requested_pulses is not None:
        requested_pulses = [float(v) for v in requested_pulses]
        keep = np.zeros(len(df), dtype=bool)
        for value in requested_pulses:
            keep |= np.isclose(df['pulse_input_amplitude_hz'].to_numpy(dtype=float), value, atol=1e-12, rtol=0.0)
        missing_pulses = [value for value in requested_pulses if not np.any(np.isclose(df['pulse_input_amplitude_hz'].to_numpy(dtype=float), value, atol=1e-12, rtol=0.0))]
        if missing_pulses:
            raise ValueError(f'Requested pulse amplitudes are absent from response metrics: {missing_pulses}')
        df = df.loc[keep].copy()

    finite_df = df[np.isfinite(df[x_col]) & np.isfinite(df[y_col]) & np.isfinite(df[group_col])].copy()
    if finite_df.empty:
        raise ValueError('No finite rows are available for the paper synchrony-response scatter.')

    pulse_values = sorted(float(v) for v in pd.unique(finite_df['pulse_input_amplitude_hz']) if np.isfinite(v))
    n_rows = len(pulse_values)
    figsize_per_pulse = tuple(style.get('figsize_per_pulse', (3.2, 2.8)))
    fig, axes = plt.subplots(
        n_rows,
        1,
        figsize=(float(figsize_per_pulse[0]), float(figsize_per_pulse[1]) * n_rows),
        squeeze=False,
        constrained_layout=True,
    )

    cmap = _paper_sync_slope_colormap(style)
    group_values = finite_df[group_col].to_numpy(dtype=float)
    norm = Normalize(vmin=float(np.nanmin(group_values)), vmax=float(np.nanmax(group_values)))
    scalar_mappable = ScalarMappable(norm=norm, cmap=cmap)
    scalar_mappable.set_array([])

    xlim = style.get('xlim')
    ylim = style.get('ylim')
    if xlim is None and bool(style.get('shared_xlim', True)):
        xlim = _paper_sync_slope_finite_limits(finite_df[x_col], style.get('pad_fraction', 0.08))
    if ylim is None and bool(style.get('shared_ylim', True)):
        ylim = _paper_sync_slope_finite_limits(finite_df[y_col], style.get('pad_fraction', 0.08))

    summary_rows = []
    for row_idx, pulse_value in enumerate(pulse_values):
        ax = axes[row_idx, 0]
        sub = finite_df[np.isclose(finite_df['pulse_input_amplitude_hz'], pulse_value, atol=1e-12, rtol=0.0)].copy()
        x = sub[x_col].to_numpy(dtype=float)
        y = sub[y_col].to_numpy(dtype=float)
        c = sub[group_col].to_numpy(dtype=float)

        ax.scatter(
            x,
            y,
            c=c,
            cmap=cmap,
            norm=norm,
            s=float(style.get('point_size', 20)),
            alpha=float(style.get('point_alpha', 0.55)),
            edgecolors=style.get('point_edgecolor', 'none'),
            zorder=2,
        )

        slope, intercept, r, r2 = _paper_sync_slope_regression(x, y)
        if bool(style.get('show_regression', True)) and np.isfinite(slope):
            line_xlim = xlim if xlim is not None else _paper_sync_slope_finite_limits(x, style.get('pad_fraction', 0.08))
            if line_xlim is not None:
                x_line = np.linspace(float(line_xlim[0]), float(line_xlim[1]), 200)
                ax.plot(
                    x_line,
                    intercept + slope * x_line,
                    color=style.get('regression_color', 'black'),
                    lw=float(style.get('regression_lw', 1.4)),
                    ls=style.get('regression_ls', '-'),
                    alpha=float(style.get('regression_alpha', 0.85)),
                    zorder=4,
                )

        if bool(style.get('show_group_markers', True)):
            for group_value, group_df in sub.groupby(group_col, dropna=True):
                gx = group_df[x_col].to_numpy(dtype=float)
                gy = group_df[y_col].to_numpy(dtype=float)
                gx = gx[np.isfinite(gx)]
                gy = gy[np.isfinite(gy)]
                if gx.size == 0 or gy.size == 0:
                    continue
                x_mean = float(np.mean(gx))
                y_mean = float(np.mean(gy))
                x_sd = float(np.std(gx, ddof=0)) if gx.size > 1 else 0.0
                y_sd = float(np.std(gy, ddof=0)) if gy.size > 1 else 0.0
                face_color = cmap(norm(float(group_value)))
                ax.errorbar(
                    x_mean,
                    y_mean,
                    xerr=x_sd,
                    yerr=y_sd,
                    fmt='none',
                    ecolor=style.get('group_errorbar_color', 'black'),
                    elinewidth=float(style.get('group_errorbar_lw', 1.0)),
                    capsize=float(style.get('group_errorbar_capsize', 2.6)),
                    alpha=float(style.get('group_errorbar_alpha', 0.8)),
                    zorder=5,
                )
                ax.scatter(
                    [x_mean],
                    [y_mean],
                    s=float(style.get('group_marker_size', 95)),
                    marker='o',
                    facecolor=face_color,
                    edgecolor=style.get('group_marker_edgecolor', 'black'),
                    linewidth=float(style.get('group_marker_edgewidth', 0.9)),
                    zorder=6,
                )
                summary_rows.append({
                    'pulse_input_amplitude_hz': float(pulse_value),
                    group_col: float(group_value),
                    'n': int(len(group_df)),
                    f'{x_col}_mean': x_mean,
                    f'{x_col}_sd': x_sd,
                    f'{y_col}_mean': y_mean,
                    f'{y_col}_sd': y_sd,
                })

        annotation = []
        if np.isfinite(r):
            annotation.append(f'r={r:.2f}')
        if np.isfinite(r2):
            annotation.append(f'R2={r2:.2f}')
        if annotation:
            ax.text(
                *style.get('annotation_xy', (0.04, 0.96)),
                '\n'.join(annotation),
                transform=ax.transAxes,
                ha=style.get('annotation_ha', 'left'),
                va=style.get('annotation_va', 'top'),
                fontsize=float(style.get('annotation_fontsize', 8)),
                bbox=style.get('annotation_bbox', {'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.72, 'pad': 2.0}),
                zorder=8,
            )

        title = style.get('title')
        if title is None:
            title = f'pulse {pulse_value:g} Hz' if n_rows > 1 else ''
        ax.set_title(title, fontsize=float(style.get('title_fontsize', 9)))
        ax.set_xlabel(style.get('x_label', x_col), fontsize=float(style.get('axis_label_fontsize', 9)))
        ax.set_ylabel(style.get('y_label', y_col), fontsize=float(style.get('axis_label_fontsize', 9)))
        ax.tick_params(labelsize=float(style.get('tick_labelsize', 8)))
        if xlim is not None:
            ax.set_xlim(*xlim)
        if ylim is not None:
            ax.set_ylim(*ylim)
        ax.grid(alpha=float(style.get('grid_alpha', 0.18)))

    if bool(style.get('show_colorbar', True)):
        cbar = fig.colorbar(scalar_mappable, ax=axes.ravel().tolist(), shrink=0.88, pad=0.025)
        cbar.set_label(style.get('colorbar_label', group_col), fontsize=float(style.get('colorbar_label_fontsize', 8)))
        cbar.ax.tick_params(labelsize=float(style.get('colorbar_tick_labelsize', 7)))

    summary_df = pd.DataFrame(summary_rows)
    saved_paths = []
    if output_path is not None:
        output_path = Path(output_path)
        output_path.mkdir(parents=True, exist_ok=True)
        base = output_path / str(PAPER_SYNC_SLOPE_SCATTER_OUTPUT_BASENAME)
        formats = style.get('save_formats', ('pdf', 'png'))
        if isinstance(formats, str):
            formats = [formats]
        for fmt in formats:
            out = base.with_suffix(f'.{fmt}')
            fig.savefig(
                out,
                dpi=int(style.get('dpi', 300)),
                bbox_inches=style.get('bbox_inches', 'tight'),
                pad_inches=float(style.get('pad_inches', 0.02)),
            )
            saved_paths.append(str(out))
        summary_df.to_csv(base.with_name(base.name + '_group_summary.csv'), index=False)
        summary_df.to_pickle(base.with_name(base.name + '_group_summary.pkl'))
    return fig, axes, summary_df, saved_paths


if PLOT_PAPER_SYNC_SLOPE_SCATTER:
    trace_batch_output_path = make_trace_batch_output_path(TRACE_BATCH_OUTPUT_DIR, TRACE_BATCH_RUN_NAME)
    paper_sync_slope_fig, paper_sync_slope_axes, paper_sync_slope_group_summary_df, paper_sync_slope_saved_paths = plot_paper_sync_vs_max_slope_scatter(
        output_path=trace_batch_output_path if TRACE_BATCH_RESPONSE_SAVE_OUTPUTS else None,
    )
    display(paper_sync_slope_group_summary_df)
    if paper_sync_slope_saved_paths:
        display(Markdown('Saved paper synchrony-response scatter:\n' + '\n'.join(f'- `{path}`' for path in paper_sync_slope_saved_paths)))
    plt.show()
else:
    display(Markdown('Paper synchrony-response scatter is disabled. Set `PLOT_PAPER_SYNC_SLOPE_SCATTER=True` to generate it.'))


In [ ]:
# %% Baseline-removed cumulative CST summary figure
# Requires:
# - trace_batch_response_metrics_df
# - trace_batch_response_trace_arrays
# - helper functions already defined earlier in the notebook:
#   _sorted_unique_numeric
#   _format_parameter_value_for_title
#   _format_fixed_parameter_values_for_title   (optional; fallback included below)
#   _merge_minmax
#   _pad_minmax
#   _select_trace_rows_for_plot
#
# This plots the matched, baseline-removed cumulative CST effect:
#   cumulative_output_effect_spikes_per_mn
# together with cumulative instructed input:
#   cumulative_input_instruction_spikes_per_mn

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if '_format_fixed_parameter_values_for_title' not in globals():
    def _format_fixed_parameter_values_for_title(fixed_parameters):
        if not fixed_parameters:
            return ''
        return ', '.join(
            _format_parameter_value_for_title(name, value)
            for name, value in fixed_parameters.items()
        )

if '_select_trace_rows_for_plot' not in globals():
    def _select_trace_rows_for_plot(trace_matrix, max_traces=None):
        trace_matrix = np.asarray(trace_matrix, dtype=float)
        if trace_matrix.ndim != 2 or trace_matrix.shape[0] == 0:
            return trace_matrix
        if max_traces is None or trace_matrix.shape[0] <= int(max_traces):
            return trace_matrix
        max_traces = int(max_traces)
        if max_traces <= 0:
            return trace_matrix[:0]
        indices = np.linspace(0, trace_matrix.shape[0] - 1, max_traces)
        indices = np.unique(np.round(indices).astype(int))
        return trace_matrix[indices]


def _is_positive_numeric(value, tol=1e-12):
    try:
        return float(value) > tol
    except Exception:
        return False


def _is_zero_numeric(value, tol=1e-12):
    try:
        return abs(float(value)) <= tol
    except Exception:
        return False


def _choose_zero_if_available_else_first(values):
    values = list(values)
    if not values:
        return None
    for value in values:
        if _is_zero_numeric(value):
            return value
    return values[0]


def _parameter_value_mask(values, target_value, tol=1e-12):
    values_series = pd.Series(values)

    numeric_values = pd.to_numeric(values_series, errors='coerce')
    try:
        numeric_target = float(target_value)
        numeric_mask = np.isfinite(numeric_values.to_numpy(dtype=float)) & np.isclose(
            numeric_values.to_numpy(dtype=float),
            numeric_target,
            atol=tol,
            rtol=0.0,
        )
        if np.any(numeric_mask):
            return numeric_mask
    except Exception:
        pass

    return values_series.astype(str).to_numpy() == str(target_value)


def _filter_table_by_parameter_values(table, parameter_values):
    mask = np.ones(len(table), dtype=bool)
    for parameter_name, parameter_value in parameter_values.items():
        if parameter_name not in table.columns:
            raise KeyError(f'Cannot filter by missing parameter column: {parameter_name!r}')
        mask &= _parameter_value_mask(table[parameter_name], parameter_value)
    return table.loc[mask].copy()


def _representative_trace(trace_matrix, mode='first'):
    trace_matrix = np.asarray(trace_matrix, dtype=float)

    if trace_matrix.ndim == 1:
        return trace_matrix
    if trace_matrix.ndim != 2 or trace_matrix.shape[0] == 0:
        return np.asarray([], dtype=float)

    if mode == 'mean':
        with np.errstate(invalid='ignore'):
            return np.nanmean(trace_matrix, axis=0)

    for trace in trace_matrix:
        if np.any(np.isfinite(trace)):
            return trace

    return np.full(trace_matrix.shape[1], np.nan, dtype=float)


def _response_trace_matrix_for_rows(metrics_subset_df, trace_arrays, *, trace_key):
    if metrics_subset_df is None or len(metrics_subset_df) == 0:
        return (
            np.asarray(trace_arrays['response_time_rel_s'], dtype=float),
            np.asarray([], dtype=float),
            np.asarray([], dtype=float),
        )

    response_ids = np.asarray(trace_arrays['response_row_index'], dtype=int)
    id_to_idx = {int(rid): i for i, rid in enumerate(response_ids)}

    row_indices = metrics_subset_df['response_row_index'].to_numpy(dtype=int)
    missing = [int(rid) for rid in row_indices if int(rid) not in id_to_idx]
    if missing:
        raise KeyError(f'Missing {len(missing)} response trace(s) in trace_arrays; first missing response_row_index={missing[0]}')

    selected_indices = np.asarray([id_to_idx[int(rid)] for rid in row_indices], dtype=int)
    t = np.asarray(trace_arrays['response_time_rel_s'], dtype=float)
    values = np.asarray(trace_arrays[trace_key], dtype=float)[selected_indices]

    if values.ndim == 1:
        values = values[None, :]

    with np.errstate(invalid='ignore'):
        mean_values = np.nanmean(values, axis=0)

    return t, values, mean_values


def build_response_cumulative_plot_table(
    metrics_df,
    *,
    column_parameter='common_fraction',
    row_parameter='pulse_input_amplitude_hz',
    include_zero_row=False,
    extra_parameters=None,
):
    extra_parameters = [] if extra_parameters is None else list(extra_parameters)

    required = {'response_row_index', column_parameter, row_parameter}
    required.update(parameter for parameter in extra_parameters if parameter is not None)

    missing = sorted(required.difference(metrics_df.columns))
    if missing:
        raise KeyError(f'response metrics df is missing required column(s): {missing}')

    table = metrics_df[list(required)].drop_duplicates().copy()

    if not include_zero_row and row_parameter == 'pulse_input_amplitude_hz':
        table = table[np.abs(pd.to_numeric(table[row_parameter], errors='coerce')) > 1e-12].copy()

    table['response_row_index'] = table['response_row_index'].astype(int)

    sort_columns = [
        col for col in [row_parameter, column_parameter, *extra_parameters, 'response_row_index']
        if col in table.columns
    ]
    return table.sort_values(sort_columns).reset_index(drop=True)


def plot_response_cumulative_effect_grid(
    metrics_df,
    trace_arrays,
    *,
    column_parameter='common_fraction',
    row_parameter='pulse_input_amplitude_hz',
    include_zero_row=False,
    main_xlim_rel_s=None,
    share_y=True,
    figsize_per_cell=(3.0, 3.8),

    # detail rows
    plot_individual_zoom=True,
    detail_step_parameter='pulse_input_amplitude_hz',
    detail_step_values=None,
    detail_zoom_xlim=(0.0, 0.25),
    detail_fixed_parameters=None,
    detail_default_fix_row_parameter=True,
    detail_share_y=True,
    detail_ylim=None,
    detail_figsize_per_row=2.8,
    detail_individual_trace_alpha=0.18,
    detail_individual_trace_lw=0.7,
    detail_max_individual_traces=None,
    detail_plot_mean_trace=True,
    detail_input_trace_mode='first',
):
    detail_fixed_parameters = dict(detail_fixed_parameters or {})

    extra_parameters = []
    if plot_individual_zoom:
        extra_parameters.append(detail_step_parameter)
        extra_parameters.extend(detail_fixed_parameters.keys())

    plot_table = build_response_cumulative_plot_table(
        metrics_df,
        column_parameter=column_parameter,
        row_parameter=row_parameter,
        include_zero_row=include_zero_row,
        extra_parameters=extra_parameters,
    )

    if plot_table.empty:
        raise ValueError('No response rows available for cumulative-effect plotting')

    column_values = _sorted_unique_numeric(plot_table[column_parameter])
    row_values = _sorted_unique_numeric(plot_table[row_parameter])

    if plot_individual_zoom:
        if detail_step_parameter not in plot_table.columns:
            raise KeyError(
                f'Cannot add individual zoom rows because {detail_step_parameter!r} '
                f'is not present in the response metrics df.'
            )

        if (
            detail_default_fix_row_parameter
            and row_parameter != detail_step_parameter
            and row_parameter not in detail_fixed_parameters
        ):
            row_parameter_values = _sorted_unique_numeric(plot_table[row_parameter])
            default_row_value = _choose_zero_if_available_else_first(row_parameter_values)
            if default_row_value is not None:
                detail_fixed_parameters[row_parameter] = default_row_value

        if detail_step_values is None:
            detail_step_values = [
                value
                for value in _sorted_unique_numeric(plot_table[detail_step_parameter])
                if _is_positive_numeric(value)
            ]
        else:
            detail_step_values = list(detail_step_values)
    else:
        detail_step_values = []

    n_rows = len(row_values)
    n_cols = len(column_values)
    n_detail_rows = len(detail_step_values)
    n_total_rows = n_rows + n_detail_rows

    figure_width = max(3.0, figsize_per_cell[0] * n_cols)
    figure_height = max(
        2.5,
        figsize_per_cell[1] * n_rows + detail_figsize_per_row * n_detail_rows,
    )

    fig, axes = plt.subplots(
        n_total_rows,
        n_cols,
        figsize=(figure_width, figure_height),
        squeeze=False,
        sharex=False,
        sharey=False,
        constrained_layout=True,
    )

    # -------------------------
    # Main panels
    # -------------------------
    main_global_ylim = None
    main_panel_cache = {}

    for row_value in row_values:
        for column_value in column_values:
            subset = _filter_table_by_parameter_values(
                plot_table,
                {
                    row_parameter: row_value,
                    column_parameter: column_value,
                },
            )

            if subset.empty:
                continue

            metrics_subset = metrics_df[
                metrics_df['response_row_index'].isin(subset['response_row_index'].to_numpy(dtype=int))
            ].copy()

            t, cumulative_output_traces, cumulative_output_mean = _response_trace_matrix_for_rows(
                metrics_subset,
                trace_arrays,
                trace_key='cumulative_output_effect_spikes_per_mn',
            )
            _, cumulative_input_traces, cumulative_input_mean = _response_trace_matrix_for_rows(
                metrics_subset,
                trace_arrays,
                trace_key='cumulative_input_instruction_spikes_per_mn',
            )

            if main_xlim_rel_s is None:
                xlim = (float(np.nanmin(t)), float(np.nanmax(t))) if t.size else (0.0, 1.0)
            else:
                xlim = (float(main_xlim_rel_s[0]), float(main_xlim_rel_s[1]))

            zoom_mask = (t >= xlim[0]) & (t <= xlim[1]) if t.size else np.asarray([], dtype=bool)
            if np.any(zoom_mask):
                main_global_ylim = _merge_minmax(main_global_ylim, cumulative_output_mean[zoom_mask])
                main_global_ylim = _merge_minmax(main_global_ylim, cumulative_input_mean[zoom_mask])

            main_panel_cache[(row_value, column_value)] = {
                'response_row_indices': subset['response_row_index'].to_numpy(dtype=int),
                'time_rel_s': t,
                'cumulative_output_mean': cumulative_output_mean,
                'cumulative_input_mean': cumulative_input_mean,
                'xlim': xlim,
            }

    if bool(share_y):
        main_global_ylim = _pad_minmax(main_global_ylim, pad_fraction=0.08)

    # -------------------------
    # Detail panels
    # -------------------------
    detail_global_ylim = None
    detail_panel_cache = {}

    for step_value in detail_step_values:
        for column_value in column_values:
            filters = {
                detail_step_parameter: step_value,
                column_parameter: column_value,
                **detail_fixed_parameters,
            }

            subset = _filter_table_by_parameter_values(plot_table, filters)
            if subset.empty:
                continue

            metrics_subset = metrics_df[
                metrics_df['response_row_index'].isin(subset['response_row_index'].to_numpy(dtype=int))
            ].copy()

            t, cumulative_output_traces, cumulative_output_mean = _response_trace_matrix_for_rows(
                metrics_subset,
                trace_arrays,
                trace_key='cumulative_output_effect_spikes_per_mn',
            )
            _, cumulative_input_traces, cumulative_input_mean = _response_trace_matrix_for_rows(
                metrics_subset,
                trace_arrays,
                trace_key='cumulative_input_instruction_spikes_per_mn',
            )

            input_trace = _representative_trace(cumulative_input_traces, mode=detail_input_trace_mode)

            zoom_mask = (
                (t >= detail_zoom_xlim[0]) & (t <= detail_zoom_xlim[1])
                if t.size
                else np.asarray([], dtype=bool)
            )

            if np.any(zoom_mask):
                detail_global_ylim = _merge_minmax(detail_global_ylim, cumulative_output_traces[:, zoom_mask])
                detail_global_ylim = _merge_minmax(detail_global_ylim, input_trace[zoom_mask])
                if detail_plot_mean_trace:
                    detail_global_ylim = _merge_minmax(detail_global_ylim, cumulative_output_mean[zoom_mask])

            detail_panel_cache[(step_value, column_value)] = {
                'response_row_indices': subset['response_row_index'].to_numpy(dtype=int),
                'time_rel_s': t,
                'cumulative_output_traces': _select_trace_rows_for_plot(
                    cumulative_output_traces,
                    detail_max_individual_traces,
                ),
                'n_output_traces': int(cumulative_output_traces.shape[0]),
                'cumulative_output_mean': cumulative_output_mean,
                'cumulative_input_trace': input_trace,
                'xlim': tuple(detail_zoom_xlim),
            }

    if bool(detail_share_y) and detail_ylim is None:
        detail_global_ylim = _pad_minmax(detail_global_ylim, pad_fraction=0.08)

    # -------------------------
    # Draw main panels
    # -------------------------
    for r, row_value in enumerate(row_values):
        for c, column_value in enumerate(column_values):
            ax = axes[r, c]
            panel = main_panel_cache.get((row_value, column_value))

            if panel is None:
                ax.text(0.5, 0.5, 'missing', ha='center', va='center', transform=ax.transAxes)
                ax.set_axis_off()
                continue

            t = panel['time_rel_s']

            ax.plot(
                t,
                panel['cumulative_output_mean'],
                color=TRACE_BATCH_PULSE_STEP_CST_COLOR,
                lw=2.0,
                label='mean cumulative matched CST effect',
                zorder=3,
            )

            ax.plot(
                t,
                panel['cumulative_input_mean'],
                color=TRACE_BATCH_PULSE_STEP_INPUT_COLOR,
                lw=1.5,
                ls='--',
                label='mean cumulative instructed input',
                zorder=4,
            )

            ax.axhline(0, color='0.5', lw=0.8, ls='--', alpha=0.8)
            ax.axvline(0, color='0.5', lw=0.8, ls=':', alpha=0.8)

            ax.set_xlim(*panel['xlim'])

            if bool(share_y) and main_global_ylim is not None:
                ax.set_ylim(*main_global_ylim)

            ax.grid(alpha=0.18)

            if r == 0:
                ax.set_title(_format_parameter_value_for_title(column_parameter, column_value), fontsize=9)

            if c == 0:
                ax.set_ylabel(
                    _format_parameter_value_for_title(row_parameter, row_value)
                    + '\ncumulative spikes / MN'
                )

            if r == n_rows - 1:
                ax.set_xlabel('time after pulse onset (s)')

            if r == 0 and c == 0:
                ax.legend(fontsize=8, loc='best')

            n_rows_here = len(panel['response_row_indices'])
            ax.text(
                0.98,
                0.04,
                f'n traces={n_rows_here}',
                transform=ax.transAxes,
                ha='right',
                va='bottom',
                fontsize=7,
                bbox={'facecolor': 'white', 'edgecolor': '0.75', 'alpha': 0.75},
            )

    # -------------------------
    # Draw detail panels
    # -------------------------
    for detail_r, step_value in enumerate(detail_step_values):
        absolute_r = n_rows + detail_r

        for c, column_value in enumerate(column_values):
            ax = axes[absolute_r, c]
            panel = detail_panel_cache.get((step_value, column_value))

            if panel is None:
                ax.text(0.5, 0.5, 'missing', ha='center', va='center', transform=ax.transAxes)
                ax.set_axis_off()
                continue

            t = panel['time_rel_s']

            input_trace = panel['cumulative_input_trace']
            if input_trace.size:
                ax.plot(
                    t,
                    input_trace,
                    color=TRACE_BATCH_PULSE_STEP_INPUT_COLOR,
                    lw=1.6,
                    ls='--',
                    alpha=0.65,
                    label='cumulative instructed input',
                    zorder=1,
                )

            for trace_i, individual_trace in enumerate(panel['cumulative_output_traces']):
                ax.plot(
                    t,
                    individual_trace,
                    color=TRACE_BATCH_PULSE_STEP_CST_COLOR,
                    lw=detail_individual_trace_lw,
                    alpha=detail_individual_trace_alpha,
                    label='individual cumulative matched CST effect' if trace_i == 0 else None,
                    zorder=2,
                )

            if detail_plot_mean_trace:
                ax.plot(
                    t,
                    panel['cumulative_output_mean'],
                    color=TRACE_BATCH_PULSE_STEP_CST_COLOR,
                    lw=2.0,
                    alpha=0.95,
                    label='mean cumulative matched CST effect',
                    zorder=3,
                )

            ax.axhline(0, color='0.5', lw=0.8, ls='--', alpha=0.8)
            ax.axvline(0, color='0.5', lw=0.8, ls=':', alpha=0.8)

            ax.set_xlim(*panel['xlim'])

            if detail_ylim is not None:
                ax.set_ylim(*detail_ylim)
            elif bool(detail_share_y) and detail_global_ylim is not None:
                ax.set_ylim(*detail_global_ylim)

            ax.grid(alpha=0.18)

            if detail_r == 0:
                ax.set_title(
                    _format_parameter_value_for_title(column_parameter, column_value),
                    fontsize=9,
                )

            if c == 0:
                ax.set_ylabel(
                    _format_parameter_value_for_title(detail_step_parameter, step_value)
                    + '\nindividual cumulative traces'
                )

            if absolute_r == n_total_rows - 1:
                ax.set_xlabel('time after pulse onset (s)')

            if detail_r == 0 and c == 0:
                ax.legend(fontsize=8, loc='best')

            n_traces = panel['n_output_traces']
            ax.text(
                0.98,
                0.04,
                f'n traces={n_traces}',
                transform=ax.transAxes,
                ha='right',
                va='bottom',
                fontsize=7,
                bbox={'facecolor': 'white', 'edgecolor': '0.75', 'alpha': 0.75},
            )

    title = (
        'Matched baseline-removed cumulative CST effect '
        f'| columns={column_parameter}, rows={row_parameter}'
    )

    if n_detail_rows > 0:
        fixed_label = _format_fixed_parameter_values_for_title(detail_fixed_parameters)
        title += (
            f'\nBottom rows: individual cumulative traces for {detail_step_parameter}>0, '
            f'xlim={tuple(detail_zoom_xlim)}'
        )
        if fixed_label:
            title += f' | fixed: {fixed_label}'

    fig.suptitle(title, fontsize=11)

    return fig, axes, plot_table


# -------------------------
# Example call
# -------------------------

trace_batch_output_path = make_trace_batch_output_path(TRACE_BATCH_OUTPUT_DIR, TRACE_BATCH_RUN_NAME)

fig, axes, response_cumulative_plot_table = plot_response_cumulative_effect_grid(
    trace_batch_response_metrics_df,
    trace_batch_response_trace_arrays,
    column_parameter='common_copy_probability', # 'common_fraction',
    row_parameter='pulse_input_amplitude_hz',
    include_zero_row=False,          # <- removes pulse/step = 0 from top panels
    main_xlim_rel_s=(-0.1, 0.2), # None,            # or e.g. (0.0, 0.8)
    share_y=True,
    figsize_per_cell=(5.0, 5.0),

    plot_individual_zoom=True,
    detail_step_parameter='pulse_input_amplitude_hz',
    detail_step_values=None,         # defaults to positive pulse values
    detail_zoom_xlim=(-0.1, 0.2),    # adjust if you want
    detail_fixed_parameters=None,
    detail_default_fix_row_parameter=True,
    detail_share_y=True,
    detail_ylim=None,                # or e.g. (-0.2, 3.0)
    detail_figsize_per_row=5.0,
    detail_individual_trace_alpha=0.1,
    detail_individual_trace_lw=0.5,
    detail_max_individual_traces=None,
    detail_plot_mean_trace=True,
    detail_input_trace_mode='first',
)

# Optional save
if globals().get('TRACE_BATCH_RESPONSE_SAVE_OUTPUTS', False):
    response_cumulative_plot_table.to_pickle(
        trace_batch_output_path / 'trace_batch_response_cumulative_effect_plot_table.pkl'
    )
    response_cumulative_plot_table.to_csv(
        trace_batch_output_path / 'trace_batch_response_cumulative_effect_plot_table.csv',
        index=False,
    )
    _save_figure_formats(
        fig,
        trace_batch_output_path / 'trace_batch_response_cumulative_effect_summary',
        TRACE_BATCH_SAVE_FIGURE_FORMATS,
    )

plt.show()

In [ ]:
# %% Baseline-removed non-cumulative CST effect-rate summary figure
# Requires:
# - trace_batch_response_metrics_df
# - trace_batch_response_trace_arrays
#
# Default output trace:
#   sign_corrected_baseline_corrected_cst_effect_spikes_per_s_mu
#
# Default input trace:
#   input_instruction_rate_hz
#
# Optional individual-trace coloring:
#   color_individual_by_age=True
# using:
#   zero_ref_pool_integrated_age_mean
# if available.

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable


# ---------------------------------------------------------------------
# Small fallback helpers, only defined if not already present upstream.
# ---------------------------------------------------------------------

if '_format_fixed_parameter_values_for_title' not in globals():
    def _format_fixed_parameter_values_for_title(fixed_parameters):
        if not fixed_parameters:
            return ''
        return ', '.join(
            _format_parameter_value_for_title(name, value)
            for name, value in fixed_parameters.items()
        )


if '_select_trace_rows_for_plot' not in globals():
    def _select_trace_rows_for_plot(trace_matrix, max_traces=None):
        trace_matrix = np.asarray(trace_matrix, dtype=float)

        if trace_matrix.ndim != 2 or trace_matrix.shape[0] == 0:
            return trace_matrix

        if max_traces is None or trace_matrix.shape[0] <= int(max_traces):
            return trace_matrix

        max_traces = int(max_traces)
        if max_traces <= 0:
            return trace_matrix[:0]

        indices = np.linspace(0, trace_matrix.shape[0] - 1, max_traces)
        indices = np.unique(np.round(indices).astype(int))
        return trace_matrix[indices]


def _response_rate_is_positive_numeric(value, tol=1e-12):
    try:
        return float(value) > tol
    except Exception:
        return False


def _response_rate_is_zero_numeric(value, tol=1e-12):
    try:
        return abs(float(value)) <= tol
    except Exception:
        return False


def _response_rate_choose_zero_if_available_else_first(values):
    values = list(values)
    if not values:
        return None
    for value in values:
        if _response_rate_is_zero_numeric(value):
            return value
    return values[0]


def _response_rate_parameter_value_mask(values, target_value, tol=1e-12):
    values_series = pd.Series(values)

    numeric_values = pd.to_numeric(values_series, errors='coerce')
    try:
        numeric_target = float(target_value)
        numeric_mask = np.isfinite(numeric_values.to_numpy(dtype=float)) & np.isclose(
            numeric_values.to_numpy(dtype=float),
            numeric_target,
            atol=tol,
            rtol=0.0,
        )
        if np.any(numeric_mask):
            return numeric_mask
    except Exception:
        pass

    return values_series.astype(str).to_numpy() == str(target_value)


def _response_rate_filter_table_by_parameter_values(table, parameter_values):
    mask = np.ones(len(table), dtype=bool)

    for parameter_name, parameter_value in parameter_values.items():
        if parameter_name not in table.columns:
            raise KeyError(f'Cannot filter by missing parameter column: {parameter_name!r}')
        mask &= _response_rate_parameter_value_mask(table[parameter_name], parameter_value)

    return table.loc[mask].copy()


def _response_rate_representative_trace(trace_matrix, mode='first'):
    trace_matrix = np.asarray(trace_matrix, dtype=float)

    if trace_matrix.ndim == 1:
        return trace_matrix

    if trace_matrix.ndim != 2 or trace_matrix.shape[0] == 0:
        return np.asarray([], dtype=float)

    if mode == 'mean':
        with np.errstate(invalid='ignore'):
            return np.nanmean(trace_matrix, axis=0)

    for trace in trace_matrix:
        if np.any(np.isfinite(trace)):
            return trace

    return np.full(trace_matrix.shape[1], np.nan, dtype=float)


def _response_rate_resolve_age_column(metrics_df, preferred_age_column=None):
    if preferred_age_column is not None:
        if preferred_age_column not in metrics_df.columns:
            raise KeyError(f'Requested age column is missing: {preferred_age_column!r}')
        return preferred_age_column

    candidates = [
        'zero_ref_pool_integrated_age_mean',
        'pool_integrated_age_mean',
        'zero_ref_pool_age_mean',
        'pool_age_mean',
    ]

    for col in candidates:
        if col in metrics_df.columns and pd.to_numeric(metrics_df[col], errors='coerce').notna().sum() > 0:
            return col

    return None


def _response_rate_build_age_norm(values, *, vmin=None, vmax=None):
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]

    if finite.size == 0:
        return Normalize(vmin=0.0, vmax=1.0)

    if vmin is None:
        vmin = float(np.nanmin(finite))
    else:
        vmin = float(vmin)

    if vmax is None:
        vmax = float(np.nanmax(finite))
    else:
        vmax = float(vmax)

    if np.isclose(vmin, vmax):
        pad = max(0.05, abs(vmin) * 0.05)
        vmin -= pad
        vmax += pad

    return Normalize(vmin=vmin, vmax=vmax)


def _response_rate_trace_matrix_for_rows(metrics_subset_df, trace_arrays, *, trace_key):
    if trace_key not in trace_arrays:
        raise KeyError(
            f'{trace_key!r} is not present in trace_arrays. '
            f'Available keys: {sorted(trace_arrays.keys())}'
        )

    if metrics_subset_df is None or len(metrics_subset_df) == 0:
        return (
            np.asarray(trace_arrays['response_time_rel_s'], dtype=float),
            np.asarray([], dtype=float),
            np.asarray([], dtype=float),
            metrics_subset_df.copy() if metrics_subset_df is not None else pd.DataFrame(),
        )

    response_ids = np.asarray(trace_arrays['response_row_index'], dtype=int)
    id_to_idx = {int(rid): i for i, rid in enumerate(response_ids)}

    row_indices = metrics_subset_df['response_row_index'].to_numpy(dtype=int)
    missing = [int(rid) for rid in row_indices if int(rid) not in id_to_idx]

    if missing:
        raise KeyError(
            f'Missing {len(missing)} response trace(s) in trace_arrays; '
            f'first missing response_row_index={missing[0]}'
        )

    selected_indices = np.asarray([id_to_idx[int(rid)] for rid in row_indices], dtype=int)

    t = np.asarray(trace_arrays['response_time_rel_s'], dtype=float)
    values = np.asarray(trace_arrays[trace_key], dtype=float)[selected_indices]

    if values.ndim == 1:
        values = values[None, :]

    with np.errstate(invalid='ignore'):
        mean_values = np.nanmean(values, axis=0)

    aligned_metrics = metrics_subset_df.copy().reset_index(drop=True)

    return t, values, mean_values, aligned_metrics


def _response_rate_select_trace_rows_and_metadata(trace_matrix, aligned_metrics, max_traces=None):
    trace_matrix = np.asarray(trace_matrix, dtype=float)
    aligned_metrics = aligned_metrics.copy().reset_index(drop=True)

    if trace_matrix.ndim != 2 or trace_matrix.shape[0] == 0:
        return trace_matrix, aligned_metrics.iloc[:0].copy()

    if max_traces is None or trace_matrix.shape[0] <= int(max_traces):
        return trace_matrix, aligned_metrics

    max_traces = int(max_traces)
    if max_traces <= 0:
        return trace_matrix[:0], aligned_metrics.iloc[:0].copy()

    indices = np.linspace(0, trace_matrix.shape[0] - 1, max_traces)
    indices = np.unique(np.round(indices).astype(int))

    return trace_matrix[indices], aligned_metrics.iloc[indices].reset_index(drop=True)


def build_response_effect_rate_plot_table(
    metrics_df,
    *,
    column_parameter='common_fraction',
    row_parameter='pulse_input_amplitude_hz',
    include_zero_row=False,
    extra_parameters=None,
):
    extra_parameters = [] if extra_parameters is None else list(extra_parameters)

    required = {'response_row_index', column_parameter, row_parameter}
    required.update(parameter for parameter in extra_parameters if parameter is not None)

    missing = sorted(required.difference(metrics_df.columns))
    if missing:
        raise KeyError(f'response metrics df is missing required column(s): {missing}')

    table = metrics_df[list(required)].drop_duplicates().copy()

    if not include_zero_row and row_parameter == 'pulse_input_amplitude_hz':
        table = table[np.abs(pd.to_numeric(table[row_parameter], errors='coerce')) > 1e-12].copy()

    table['response_row_index'] = table['response_row_index'].astype(int)

    sort_columns = [
        col for col in [row_parameter, column_parameter, *extra_parameters, 'response_row_index']
        if col in table.columns
    ]

    return table.sort_values(sort_columns).reset_index(drop=True)


In [ ]:
def plot_response_effect_rate_grid(
    metrics_df,
    trace_arrays,
    *,
    column_parameter='common_fraction',
    row_parameter='pulse_input_amplitude_hz',
    include_zero_row=False,

    # Trace keys.
    output_trace_key='sign_corrected_baseline_corrected_cst_effect_spikes_per_s_mu',
    input_trace_key='input_instruction_rate_hz',

    # Main grid.
    main_xlim_rel_s=None,
    main_ylim=None,
    share_y=True,
    figsize_per_cell=(3.0, 3.8),

    # Detail rows: non-cumulative effect rate.
    plot_individual_zoom=True,
    detail_step_parameter='pulse_input_amplitude_hz',
    detail_step_values=None,
    detail_zoom_xlim=(0.0, 0.25),
    detail_fixed_parameters=None,
    detail_default_fix_row_parameter=True,
    detail_share_y=True,
    detail_ylim=None,
    detail_figsize_per_row=2.8,
    detail_max_individual_traces=None,
    detail_plot_mean_trace=True,
    detail_input_trace_mode='first',

    # Optional extra cumulative/detail rows.
    plot_cumulative_zoom=True,
    cumulative_output_trace_key='cumulative_output_effect_spikes_per_mn',
    cumulative_input_trace_key='cumulative_input_instruction_spikes_per_mn',
    cumulative_subtract_input_from_output=False,
    cumulative_zoom_xlim=None,
    cumulative_share_y=True,
    cumulative_ylim=None,
    cumulative_figsize_per_row=None,
    cumulative_detail_max_individual_traces=None,
    cumulative_plot_mean_trace=True,
    cumulative_input_trace_mode='first',

    # Colors and line styles: effect-rate rows.
    output_mean_color=None,
    output_individual_color=None,
    input_mean_color=None,
    input_individual_color=None,
    zero_line_color='0.5',
    output_mean_lw=2.2,
    output_mean_alpha=1.0,
    output_individual_lw=0.7,
    output_individual_alpha=0.20,
    input_mean_lw=1.5,
    input_mean_alpha=0.85,
    input_individual_lw=1.6,
    input_individual_alpha=0.65,
    input_ls='--',

    # Colors and line styles: cumulative rows.
    cumulative_mean_color=None,
    cumulative_individual_color=None,
    cumulative_input_color=None,
    cumulative_mean_lw=None,
    cumulative_mean_alpha=1.0,
    cumulative_individual_lw=None,
    cumulative_individual_alpha=None,
    cumulative_input_lw=None,
    cumulative_input_alpha=None,
    cumulative_input_ls='--',

    # Optional age coloring for individual traces.
    color_individual_by_age=False,
    age_column=None,
    age_cmap='viridis',
    age_vmin=None,
    age_vmax=None,
    age_nan_color='0.65',
    show_age_colorbar=True,
    age_colorbar_label=None,

    # Highlight min/max age traces in individual panels.
    highlight_minmax_age_traces=True,
    highlight_age_column=None,
    highlight_reaction_time_column='reaction_time_to_epsilon_s',
    highlight_age_lw=3.0,
    highlight_age_alpha=1.0,
    highlight_age_zorder=30,
    highlight_age_use_cmap_color=True,
    highlight_min_age_color='tab:blue',
    highlight_max_age_color='tab:orange',
    highlight_age_text=True,
    highlight_age_text_fontsize=8,
    highlight_age_text_xy=(0.02, 0.98),
    highlight_age_text_bbox=None,

    # Epsilon threshold display on cumulative panels.
    show_reaction_epsilon_in_cumulative=True,
    reaction_epsilon_column='reaction_epsilon_extra_spikes_per_mn',
    reaction_epsilon_line_color='0.25',
    reaction_epsilon_line_lw=1.2,
    reaction_epsilon_line_alpha=0.95,
    reaction_epsilon_line_ls=':',
    highlight_threshold_markers=True,
    highlight_threshold_marker='o',
    highlight_threshold_marker_size=48,
    highlight_threshold_marker_edgecolor='black',
    highlight_threshold_marker_linewidth=0.6,
    highlight_threshold_marker_zorder=40,

    # Labels.
    output_mean_label='mean matched CST effect rate',
    output_individual_label='individual matched CST effect rate',
    input_mean_label='mean input instruction rate',
    input_individual_label='input instruction target',
    cumulative_mean_label=None,
    cumulative_individual_label=None,
    cumulative_input_label='cumulative input instruction',
    y_label_main='effect rate\n(sp/s/MN)',
    y_label_detail='individual effect-rate traces',
    y_label_cumulative='individual cumulative traces',
    title_prefix='Matched baseline-removed CST effect rate',
):
    def _format_rt_value(value):
        try:
            value = float(value)
        except Exception:
            return 'nan'
        if not np.isfinite(value):
            return 'nan'
        return f'{value:.3g}s'

    def _get_trace_color_from_age(age_value, fallback_color):
        if color_individual_by_age and age_cmap_obj is not None and age_norm is not None:
            if np.isfinite(age_value):
                return age_cmap_obj(age_norm(age_value))
            return age_nan_color
        return fallback_color

    def _interp_trace_value_at_time(t, y, x_query):
        t = np.asarray(t, dtype=float).ravel()
        y = np.asarray(y, dtype=float).ravel()

        if t.size == 0 or y.size == 0 or t.size != y.size:
            return np.nan

        try:
            x_query = float(x_query)
        except Exception:
            return np.nan

        finite = np.isfinite(t) & np.isfinite(y)
        if int(np.sum(finite)) < 2:
            return np.nan

        t_finite = t[finite]
        y_finite = y[finite]

        if x_query < float(np.nanmin(t_finite)) or x_query > float(np.nanmax(t_finite)):
            return np.nan

        return float(np.interp(x_query, t_finite, y_finite))

    def _get_highlight_indices_by_minmax_age(aligned_metrics, age_col):
        if age_col is None or age_col not in aligned_metrics.columns or len(aligned_metrics) == 0:
            return []

        ages = pd.to_numeric(aligned_metrics[age_col], errors='coerce').to_numpy(dtype=float)
        finite = np.flatnonzero(np.isfinite(ages))

        if finite.size == 0:
            return []

        min_idx = int(finite[np.argmin(ages[finite])])
        max_idx = int(finite[np.argmax(ages[finite])])

        if min_idx == max_idx:
            return [('min/max age', min_idx)]

        return [('min age', min_idx), ('max age', max_idx)]

    def _draw_minmax_age_highlights(
        ax,
        *,
        t,
        trace_matrix,
        aligned_metrics,
        age_col,
        base_color,
        label_prefix,
        zorder_base,
    ):
        if not highlight_minmax_age_traces:
            return []

        if age_col is None or age_col not in aligned_metrics.columns:
            return []

        trace_matrix = np.asarray(trace_matrix, dtype=float)

        if trace_matrix.ndim != 2 or trace_matrix.shape[0] == 0:
            return []

        highlight_specs = _get_highlight_indices_by_minmax_age(aligned_metrics, age_col)

        if not highlight_specs:
            return []

        text_lines = []
        highlight_infos = []

        for highlight_i, (age_label, trace_idx) in enumerate(highlight_specs):
            if trace_idx >= trace_matrix.shape[0]:
                continue

            age_value = pd.to_numeric(
                pd.Series([aligned_metrics.iloc[trace_idx].get(age_col, np.nan)]),
                errors='coerce',
            ).iloc[0]

            rt_value = aligned_metrics.iloc[trace_idx].get(highlight_reaction_time_column, np.nan)

            if highlight_age_use_cmap_color:
                trace_color = _get_trace_color_from_age(age_value, base_color)
            elif 'min' in age_label and 'max' not in age_label:
                trace_color = highlight_min_age_color
            elif 'max' in age_label:
                trace_color = highlight_max_age_color
            else:
                trace_color = base_color

            ax.plot(
                t,
                trace_matrix[trace_idx],
                color=trace_color,
                lw=highlight_age_lw,
                alpha=highlight_age_alpha,
                zorder=zorder_base + highlight_i,
                label=f'{label_prefix}: {age_label}',
            )

            try:
                age_text = f'{float(age_value):.3g}'
            except Exception:
                age_text = 'nan'

            text_lines.append(
                f'{age_label}: age={age_text}, RTε={_format_rt_value(rt_value)}'
            )

            highlight_infos.append({
                'age_label': age_label,
                'trace_idx': int(trace_idx),
                'age_value': float(age_value) if np.isfinite(age_value) else np.nan,
                'rt_value': float(rt_value) if pd.notna(rt_value) and np.isfinite(float(rt_value)) else np.nan,
                'trace_color': trace_color,
            })

        if highlight_age_text and text_lines:
            bbox = highlight_age_text_bbox
            if bbox is None:
                bbox = {
                    'facecolor': 'white',
                    'edgecolor': '0.75',
                    'alpha': 0.85,
                }

            ax.text(
                float(highlight_age_text_xy[0]),
                float(highlight_age_text_xy[1]),
                '\n'.join(text_lines),
                transform=ax.transAxes,
                ha='left',
                va='top',
                fontsize=highlight_age_text_fontsize,
                bbox=bbox,
                zorder=zorder_base + 10,
            )

        return highlight_infos

    detail_fixed_parameters = dict(detail_fixed_parameters or {})

    if cumulative_zoom_xlim is None:
        cumulative_zoom_xlim = detail_zoom_xlim

    if cumulative_figsize_per_row is None:
        cumulative_figsize_per_row = detail_figsize_per_row

    if cumulative_detail_max_individual_traces is None:
        cumulative_detail_max_individual_traces = detail_max_individual_traces

    if cumulative_mean_lw is None:
        cumulative_mean_lw = output_mean_lw

    if cumulative_individual_lw is None:
        cumulative_individual_lw = output_individual_lw

    if cumulative_individual_alpha is None:
        cumulative_individual_alpha = output_individual_alpha

    if cumulative_input_lw is None:
        cumulative_input_lw = input_individual_lw

    if cumulative_input_alpha is None:
        cumulative_input_alpha = input_individual_alpha

    if output_mean_color is None:
        output_mean_color = globals().get('TRACE_BATCH_PULSE_STEP_CST_COLOR', '#ff3300')

    if output_individual_color is None:
        output_individual_color = globals().get('TRACE_BATCH_PULSE_STEP_CST_COLOR', '#ff3300')

    if input_mean_color is None:
        input_mean_color = globals().get('TRACE_BATCH_PULSE_STEP_INPUT_COLOR', 'black')

    if input_individual_color is None:
        input_individual_color = globals().get('TRACE_BATCH_PULSE_STEP_INPUT_COLOR', 'black')

    if cumulative_mean_color is None:
        cumulative_mean_color = output_mean_color

    if cumulative_individual_color is None:
        cumulative_individual_color = output_individual_color

    if cumulative_input_color is None:
        cumulative_input_color = input_individual_color

    if cumulative_subtract_input_from_output:
        if cumulative_mean_label is None:
            cumulative_mean_label = 'mean cumulative CST − input'
        if cumulative_individual_label is None:
            cumulative_individual_label = 'individual cumulative CST − input'
    else:
        if cumulative_mean_label is None:
            cumulative_mean_label = 'mean cumulative matched CST effect'
        if cumulative_individual_label is None:
            cumulative_individual_label = 'individual cumulative matched CST effect'

    age_column_resolved = None
    age_norm = None
    age_cmap_obj = None

    needs_age = bool(color_individual_by_age or highlight_minmax_age_traces)

    if needs_age:
        preferred_age_col = age_column if highlight_age_column is None else highlight_age_column
        age_column_resolved = _response_rate_resolve_age_column(metrics_df, preferred_age_column=preferred_age_col)

        if age_column_resolved is None:
            warnings.warn(
                'Age coloring/highlighting was requested, but no finite pool-age column was found. '
                'Age coloring and min/max-age highlighting are disabled.'
            )
            color_individual_by_age = False
            highlight_minmax_age_traces = False
        else:
            all_age_values = pd.to_numeric(metrics_df[age_column_resolved], errors='coerce').to_numpy(dtype=float)
            age_norm = _response_rate_build_age_norm(all_age_values, vmin=age_vmin, vmax=age_vmax)
            age_cmap_obj = plt.get_cmap(age_cmap)

    extra_parameters = []
    if plot_individual_zoom or plot_cumulative_zoom:
        extra_parameters.append(detail_step_parameter)
        extra_parameters.extend(detail_fixed_parameters.keys())

    plot_table = build_response_effect_rate_plot_table(
        metrics_df,
        column_parameter=column_parameter,
        row_parameter=row_parameter,
        include_zero_row=include_zero_row,
        extra_parameters=extra_parameters,
    )

    if plot_table.empty:
        raise ValueError('No response rows available for effect-rate plotting')

    column_values = _sorted_unique_numeric(plot_table[column_parameter])
    row_values = _sorted_unique_numeric(plot_table[row_parameter])

    if plot_individual_zoom or plot_cumulative_zoom:
        if detail_step_parameter not in plot_table.columns:
            raise KeyError(
                f'Cannot add individual zoom rows because {detail_step_parameter!r} '
                f'is not present in the response metrics df.'
            )

        if (
            detail_default_fix_row_parameter
            and row_parameter != detail_step_parameter
            and row_parameter not in detail_fixed_parameters
        ):
            row_parameter_values = _sorted_unique_numeric(plot_table[row_parameter])
            default_row_value = _response_rate_choose_zero_if_available_else_first(row_parameter_values)

            if default_row_value is not None:
                detail_fixed_parameters[row_parameter] = default_row_value

        if detail_step_values is None:
            detail_step_values = [
                value
                for value in _sorted_unique_numeric(plot_table[detail_step_parameter])
                if _response_rate_is_positive_numeric(value)
            ]
        else:
            detail_step_values = list(detail_step_values)
    else:
        detail_step_values = []

    n_rows = len(row_values)
    n_cols = len(column_values)
    n_detail_rows = len(detail_step_values) if plot_individual_zoom else 0
    n_cumulative_rows = len(detail_step_values) if plot_cumulative_zoom else 0
    n_total_rows = n_rows + n_detail_rows + n_cumulative_rows

    figure_width = max(3.0, figsize_per_cell[0] * n_cols)
    figure_height = max(
        2.5,
        figsize_per_cell[1] * n_rows
        + detail_figsize_per_row * n_detail_rows
        + cumulative_figsize_per_row * n_cumulative_rows,
    )

    fig, axes = plt.subplots(
        n_total_rows,
        n_cols,
        figsize=(figure_width, figure_height),
        squeeze=False,
        sharex=False,
        sharey=False,
        constrained_layout=True,
    )

    # -----------------------------------------------------------------
    # Cache main panels.
    # -----------------------------------------------------------------

    main_global_ylim = None
    main_panel_cache = {}

    for row_value in row_values:
        for column_value in column_values:
            subset = _response_rate_filter_table_by_parameter_values(
                plot_table,
                {
                    row_parameter: row_value,
                    column_parameter: column_value,
                },
            )

            if subset.empty:
                continue

            metrics_subset = metrics_df[
                metrics_df['response_row_index'].isin(subset['response_row_index'].to_numpy(dtype=int))
            ].copy()

            t, output_traces, output_mean, aligned_metrics = _response_rate_trace_matrix_for_rows(
                metrics_subset,
                trace_arrays,
                trace_key=output_trace_key,
            )

            _, input_traces, input_mean, _ = _response_rate_trace_matrix_for_rows(
                metrics_subset,
                trace_arrays,
                trace_key=input_trace_key,
            )

            if main_xlim_rel_s is None:
                xlim = (float(np.nanmin(t)), float(np.nanmax(t))) if t.size else (0.0, 1.0)
            else:
                xlim = (float(main_xlim_rel_s[0]), float(main_xlim_rel_s[1]))

            zoom_mask = (t >= xlim[0]) & (t <= xlim[1]) if t.size else np.asarray([], dtype=bool)

            if np.any(zoom_mask):
                main_global_ylim = _merge_minmax(main_global_ylim, output_mean[zoom_mask])
                main_global_ylim = _merge_minmax(main_global_ylim, input_mean[zoom_mask])

            main_panel_cache[(row_value, column_value)] = {
                'response_row_indices': subset['response_row_index'].to_numpy(dtype=int),
                'time_rel_s': t,
                'output_mean': output_mean,
                'input_mean': input_mean,
                'xlim': xlim,
            }

    if bool(share_y) and main_ylim is None:
        main_global_ylim = _pad_minmax(main_global_ylim, pad_fraction=0.08)

    # -----------------------------------------------------------------
    # Cache detail effect-rate panels.
    # -----------------------------------------------------------------

    detail_global_ylim = None
    detail_panel_cache = {}

    if plot_individual_zoom:
        for step_value in detail_step_values:
            for column_value in column_values:
                filters = {
                    detail_step_parameter: step_value,
                    column_parameter: column_value,
                    **detail_fixed_parameters,
                }

                subset = _response_rate_filter_table_by_parameter_values(plot_table, filters)

                if subset.empty:
                    continue

                metrics_subset = metrics_df[
                    metrics_df['response_row_index'].isin(subset['response_row_index'].to_numpy(dtype=int))
                ].copy()

                t, output_traces, output_mean, aligned_metrics = _response_rate_trace_matrix_for_rows(
                    metrics_subset,
                    trace_arrays,
                    trace_key=output_trace_key,
                )

                _, input_traces, input_mean, _ = _response_rate_trace_matrix_for_rows(
                    metrics_subset,
                    trace_arrays,
                    trace_key=input_trace_key,
                )

                input_trace = _response_rate_representative_trace(input_traces, mode=detail_input_trace_mode)

                selected_output_traces, selected_aligned_metrics = _response_rate_select_trace_rows_and_metadata(
                    output_traces,
                    aligned_metrics,
                    max_traces=detail_max_individual_traces,
                )

                if color_individual_by_age and age_column_resolved is not None:
                    selected_age_values = pd.to_numeric(
                        selected_aligned_metrics[age_column_resolved],
                        errors='coerce',
                    ).to_numpy(dtype=float)
                else:
                    selected_age_values = np.full(selected_output_traces.shape[0], np.nan, dtype=float)

                zoom_mask = (
                    (t >= detail_zoom_xlim[0]) & (t <= detail_zoom_xlim[1])
                    if t.size
                    else np.asarray([], dtype=bool)
                )

                if np.any(zoom_mask):
                    detail_global_ylim = _merge_minmax(detail_global_ylim, output_traces[:, zoom_mask])
                    detail_global_ylim = _merge_minmax(detail_global_ylim, input_trace[zoom_mask])

                    if detail_plot_mean_trace:
                        detail_global_ylim = _merge_minmax(detail_global_ylim, output_mean[zoom_mask])

                detail_panel_cache[(step_value, column_value)] = {
                    'response_row_indices': subset['response_row_index'].to_numpy(dtype=int),
                    'time_rel_s': t,
                    'output_traces': selected_output_traces,
                    'output_trace_age_values': selected_age_values,
                    'full_output_traces': output_traces,
                    'full_aligned_metrics': aligned_metrics,
                    'n_output_traces': int(output_traces.shape[0]),
                    'output_mean': output_mean,
                    'input_trace': input_trace,
                    'xlim': tuple(detail_zoom_xlim),
                }

        if bool(detail_share_y) and detail_ylim is None:
            detail_global_ylim = _pad_minmax(detail_global_ylim, pad_fraction=0.08)

    # -----------------------------------------------------------------
    # Cache cumulative panels.
    # -----------------------------------------------------------------

    cumulative_global_ylim = None
    cumulative_panel_cache = {}

    if plot_cumulative_zoom:
        for step_value in detail_step_values:
            for column_value in column_values:
                filters = {
                    detail_step_parameter: step_value,
                    column_parameter: column_value,
                    **detail_fixed_parameters,
                }

                subset = _response_rate_filter_table_by_parameter_values(plot_table, filters)

                if subset.empty:
                    continue

                metrics_subset = metrics_df[
                    metrics_df['response_row_index'].isin(subset['response_row_index'].to_numpy(dtype=int))
                ].copy()

                t, cumulative_output_traces, cumulative_output_mean_raw, aligned_metrics = _response_rate_trace_matrix_for_rows(
                    metrics_subset,
                    trace_arrays,
                    trace_key=cumulative_output_trace_key,
                )

                _, cumulative_input_traces, cumulative_input_mean, _ = _response_rate_trace_matrix_for_rows(
                    metrics_subset,
                    trace_arrays,
                    trace_key=cumulative_input_trace_key,
                )

                if cumulative_subtract_input_from_output:
                    cumulative_plot_traces = cumulative_output_traces - cumulative_input_traces
                    with np.errstate(invalid='ignore'):
                        cumulative_plot_mean = np.nanmean(cumulative_plot_traces, axis=0)
                else:
                    cumulative_plot_traces = cumulative_output_traces
                    cumulative_plot_mean = cumulative_output_mean_raw

                cumulative_input_trace = _response_rate_representative_trace(
                    cumulative_input_traces,
                    mode=cumulative_input_trace_mode,
                )

                selected_cumulative_traces, selected_cumulative_metrics = _response_rate_select_trace_rows_and_metadata(
                    cumulative_plot_traces,
                    aligned_metrics,
                    max_traces=cumulative_detail_max_individual_traces,
                )

                if color_individual_by_age and age_column_resolved is not None:
                    selected_age_values = pd.to_numeric(
                        selected_cumulative_metrics[age_column_resolved],
                        errors='coerce',
                    ).to_numpy(dtype=float)
                else:
                    selected_age_values = np.full(selected_cumulative_traces.shape[0], np.nan, dtype=float)

                zoom_mask = (
                    (t >= cumulative_zoom_xlim[0]) & (t <= cumulative_zoom_xlim[1])
                    if t.size
                    else np.asarray([], dtype=bool)
                )

                if np.any(zoom_mask):
                    cumulative_global_ylim = _merge_minmax(cumulative_global_ylim, cumulative_plot_traces[:, zoom_mask])
                    cumulative_global_ylim = _merge_minmax(cumulative_global_ylim, cumulative_input_trace[zoom_mask])

                    if cumulative_plot_mean_trace:
                        cumulative_global_ylim = _merge_minmax(cumulative_global_ylim, cumulative_plot_mean[zoom_mask])

                cumulative_panel_cache[(step_value, column_value)] = {
                    'response_row_indices': subset['response_row_index'].to_numpy(dtype=int),
                    'time_rel_s': t,
                    'cumulative_traces': selected_cumulative_traces,
                    'cumulative_trace_age_values': selected_age_values,
                    'full_cumulative_traces': cumulative_plot_traces,
                    'full_aligned_metrics': aligned_metrics,
                    'n_cumulative_traces': int(cumulative_plot_traces.shape[0]),
                    'cumulative_mean': cumulative_plot_mean,
                    'cumulative_input_trace': cumulative_input_trace,
                    'xlim': tuple(cumulative_zoom_xlim),
                }

        if bool(cumulative_share_y) and cumulative_ylim is None:
            cumulative_global_ylim = _pad_minmax(cumulative_global_ylim, pad_fraction=0.08)

    # -----------------------------------------------------------------
    # Draw main panels.
    # -----------------------------------------------------------------

    for r, row_value in enumerate(row_values):
        for c, column_value in enumerate(column_values):
            ax = axes[r, c]
            panel = main_panel_cache.get((row_value, column_value))

            if panel is None:
                ax.text(0.5, 0.5, 'missing', ha='center', va='center', transform=ax.transAxes)
                ax.set_axis_off()
                continue

            t = panel['time_rel_s']

            ax.plot(
                t,
                panel['output_mean'],
                color=output_mean_color,
                lw=output_mean_lw,
                alpha=output_mean_alpha,
                label=output_mean_label,
                zorder=3,
            )

            ax.plot(
                t,
                panel['input_mean'],
                color=input_mean_color,
                lw=input_mean_lw,
                alpha=input_mean_alpha,
                ls=input_ls,
                label=input_mean_label,
                zorder=4,
            )

            ax.axhline(0, color=zero_line_color, lw=0.8, ls='--', alpha=0.8)
            ax.axvline(0, color=zero_line_color, lw=0.8, ls=':', alpha=0.8)

            ax.set_xlim(*panel['xlim'])

            if main_ylim is not None:
                ax.set_ylim(*main_ylim)
            elif bool(share_y) and main_global_ylim is not None:
                ax.set_ylim(*main_global_ylim)

            ax.grid(alpha=0.18)

            if r == 0:
                ax.set_title(_format_parameter_value_for_title(column_parameter, column_value), fontsize=9)

            if c == 0:
                ax.set_ylabel(
                    _format_parameter_value_for_title(row_parameter, row_value)
                    + f'\n{y_label_main}'
                )

            if r == n_rows - 1 and n_detail_rows == 0 and n_cumulative_rows == 0:
                ax.set_xlabel('time after pulse onset (s)')

            if r == 0 and c == 0:
                ax.legend(fontsize=8, loc='best')

            n_rows_here = len(panel['response_row_indices'])
            ax.text(
                0.98,
                0.04,
                f'n traces={n_rows_here}',
                transform=ax.transAxes,
                ha='right',
                va='bottom',
                fontsize=7,
                bbox={'facecolor': 'white', 'edgecolor': '0.75', 'alpha': 0.75},
            )

    # -----------------------------------------------------------------
    # Draw detail effect-rate panels.
    # -----------------------------------------------------------------

    if plot_individual_zoom:
        for detail_r, step_value in enumerate(detail_step_values):
            absolute_r = n_rows + detail_r

            for c, column_value in enumerate(column_values):
                ax = axes[absolute_r, c]
                panel = detail_panel_cache.get((step_value, column_value))

                if panel is None:
                    ax.text(0.5, 0.5, 'missing', ha='center', va='center', transform=ax.transAxes)
                    ax.set_axis_off()
                    continue

                t = panel['time_rel_s']

                input_trace = panel['input_trace']
                if input_trace.size:
                    ax.plot(
                        t,
                        input_trace,
                        color=input_individual_color,
                        lw=input_individual_lw,
                        ls=input_ls,
                        alpha=input_individual_alpha,
                        label=input_individual_label,
                        zorder=1,
                    )

                for trace_i, individual_trace in enumerate(panel['output_traces']):
                    age_value = panel['output_trace_age_values'][trace_i]
                    trace_color = _get_trace_color_from_age(age_value, output_individual_color)

                    ax.plot(
                        t,
                        individual_trace,
                        color=trace_color,
                        lw=output_individual_lw,
                        alpha=output_individual_alpha,
                        label=output_individual_label if trace_i == 0 else None,
                        zorder=2,
                    )

                _ = _draw_minmax_age_highlights(
                    ax,
                    t=t,
                    trace_matrix=panel['full_output_traces'],
                    aligned_metrics=panel['full_aligned_metrics'],
                    age_col=age_column_resolved,
                    base_color=output_individual_color,
                    label_prefix='effect-rate highlight',
                    zorder_base=highlight_age_zorder,
                )

                if detail_plot_mean_trace:
                    ax.plot(
                        t,
                        panel['output_mean'],
                        color=output_mean_color,
                        lw=output_mean_lw,
                        alpha=output_mean_alpha,
                        label=output_mean_label,
                        zorder=3,
                    )

                ax.axhline(0, color=zero_line_color, lw=0.8, ls='--', alpha=0.8)
                ax.axvline(0, color=zero_line_color, lw=0.8, ls=':', alpha=0.8)

                ax.set_xlim(*panel['xlim'])

                if detail_ylim is not None:
                    ax.set_ylim(*detail_ylim)
                elif bool(detail_share_y) and detail_global_ylim is not None:
                    ax.set_ylim(*detail_global_ylim)

                ax.grid(alpha=0.18)

                if detail_r == 0:
                    ax.set_title(
                        _format_parameter_value_for_title(column_parameter, column_value),
                        fontsize=9,
                    )

                if c == 0:
                    ax.set_ylabel(
                        _format_parameter_value_for_title(detail_step_parameter, step_value)
                        + f'\n{y_label_detail}'
                    )

                if absolute_r == n_total_rows - 1:
                    ax.set_xlabel('time after pulse onset (s)')

                if detail_r == 0 and c == 0:
                    ax.legend(fontsize=8, loc='best')

                n_traces = panel['n_output_traces']
                ax.text(
                    0.98,
                    0.04,
                    f'n traces={n_traces}',
                    transform=ax.transAxes,
                    ha='right',
                    va='bottom',
                    fontsize=7,
                    bbox={'facecolor': 'white', 'edgecolor': '0.75', 'alpha': 0.75},
                )

    # -----------------------------------------------------------------
    # Draw cumulative panels.
    # -----------------------------------------------------------------

    if plot_cumulative_zoom:
        for cumulative_r, step_value in enumerate(detail_step_values):
            absolute_r = n_rows + n_detail_rows + cumulative_r

            for c, column_value in enumerate(column_values):
                ax = axes[absolute_r, c]
                panel = cumulative_panel_cache.get((step_value, column_value))

                if panel is None:
                    ax.text(0.5, 0.5, 'missing', ha='center', va='center', transform=ax.transAxes)
                    ax.set_axis_off()
                    continue

                t = panel['time_rel_s']

                cumulative_input_trace = panel['cumulative_input_trace']
                if cumulative_input_trace.size:
                    ax.plot(
                        t,
                        cumulative_input_trace,
                        color=cumulative_input_color,
                        lw=cumulative_input_lw,
                        ls=cumulative_input_ls,
                        alpha=cumulative_input_alpha,
                        label=cumulative_input_label,
                        zorder=1,
                    )

                for trace_i, individual_trace in enumerate(panel['cumulative_traces']):
                    age_value = panel['cumulative_trace_age_values'][trace_i]
                    trace_color = _get_trace_color_from_age(age_value, cumulative_individual_color)

                    ax.plot(
                        t,
                        individual_trace,
                        color=trace_color,
                        lw=cumulative_individual_lw,
                        alpha=cumulative_individual_alpha,
                        label=cumulative_individual_label if trace_i == 0 else None,
                        zorder=2,
                    )

                highlight_infos = _draw_minmax_age_highlights(
                    ax,
                    t=t,
                    trace_matrix=panel['full_cumulative_traces'],
                    aligned_metrics=panel['full_aligned_metrics'],
                    age_col=age_column_resolved,
                    base_color=cumulative_individual_color,
                    label_prefix='cumulative highlight',
                    zorder_base=highlight_age_zorder,
                )

                if show_reaction_epsilon_in_cumulative and reaction_epsilon_column in panel['full_aligned_metrics'].columns:
                    epsilon_values = pd.to_numeric(
                        panel['full_aligned_metrics'][reaction_epsilon_column],
                        errors='coerce',
                    ).to_numpy(dtype=float)
                    finite_eps = epsilon_values[np.isfinite(epsilon_values)]

                    if finite_eps.size > 0:
                        epsilon_value = float(np.median(finite_eps))

                        ax.axhline(
                            epsilon_value,
                            color=reaction_epsilon_line_color,
                            lw=reaction_epsilon_line_lw,
                            alpha=reaction_epsilon_line_alpha,
                            ls=reaction_epsilon_line_ls,
                            label=f'epsilon={epsilon_value:g}',
                            zorder=5,
                        )

                        if highlight_threshold_markers:
                            for marker_i, info in enumerate(highlight_infos):
                                rt_value = info.get('rt_value', np.nan)
                                trace_idx = info.get('trace_idx', None)
                                marker_color = info.get('trace_color', cumulative_individual_color)

                                if trace_idx is None or not np.isfinite(rt_value):
                                    continue

                                if trace_idx >= panel['full_cumulative_traces'].shape[0]:
                                    continue

                                y_marker = _interp_trace_value_at_time(
                                    t,
                                    panel['full_cumulative_traces'][trace_idx],
                                    rt_value,
                                )

                                if not np.isfinite(y_marker):
                                    continue

                                ax.scatter(
                                    [rt_value],
                                    [y_marker],
                                    s=highlight_threshold_marker_size,
                                    marker=highlight_threshold_marker,
                                    facecolors=[marker_color],
                                    edgecolors=highlight_threshold_marker_edgecolor,
                                    linewidths=highlight_threshold_marker_linewidth,
                                    zorder=highlight_threshold_marker_zorder + marker_i,
                                )

                if cumulative_plot_mean_trace:
                    ax.plot(
                        t,
                        panel['cumulative_mean'],
                        color=cumulative_mean_color,
                        lw=cumulative_mean_lw,
                        alpha=cumulative_mean_alpha,
                        label=cumulative_mean_label,
                        zorder=3,
                    )

                ax.axhline(0, color=zero_line_color, lw=0.8, ls='--', alpha=0.8)
                ax.axvline(0, color=zero_line_color, lw=0.8, ls=':', alpha=0.8)

                ax.set_xlim(*panel['xlim'])

                if cumulative_ylim is not None:
                    ax.set_ylim(*cumulative_ylim)
                elif bool(cumulative_share_y) and cumulative_global_ylim is not None:
                    ax.set_ylim(*cumulative_global_ylim)

                ax.grid(alpha=0.18)

                if cumulative_r == 0:
                    ax.set_title(
                        _format_parameter_value_for_title(column_parameter, column_value),
                        fontsize=9,
                    )

                if c == 0:
                    ax.set_ylabel(
                        _format_parameter_value_for_title(detail_step_parameter, step_value)
                        + f'\n{y_label_cumulative}'
                    )

                if absolute_r == n_total_rows - 1:
                    ax.set_xlabel('time after pulse onset (s)')

                if cumulative_r == 0 and c == 0:
                    ax.legend(fontsize=8, loc='best')

                n_traces = panel['n_cumulative_traces']
                ax.text(
                    0.98,
                    0.04,
                    f'n traces={n_traces}',
                    transform=ax.transAxes,
                    ha='right',
                    va='bottom',
                    fontsize=7,
                    bbox={'facecolor': 'white', 'edgecolor': '0.75', 'alpha': 0.75},
                )

    if (
        color_individual_by_age
        and show_age_colorbar
        and age_cmap_obj is not None
        and age_norm is not None
    ):
        scalar_mappable = ScalarMappable(norm=age_norm, cmap=age_cmap_obj)
        scalar_mappable.set_array([])

        colorbar_axes = axes[n_rows:, :].ravel().tolist() if (n_detail_rows + n_cumulative_rows) > 0 else axes.ravel().tolist()

        cbar = fig.colorbar(
            scalar_mappable,
            ax=colorbar_axes,
            shrink=0.88,
            pad=0.01,
        )

        if age_colorbar_label is None:
            age_colorbar_label = age_column_resolved.replace('_', ' ')

        cbar.set_label(age_colorbar_label)

    title = (
        f'{title_prefix} | columns={column_parameter}, rows={row_parameter}'
    )

    if n_detail_rows > 0:
        fixed_label = _format_fixed_parameter_values_for_title(detail_fixed_parameters)
        title += (
            f'\nMiddle rows: individual effect-rate traces for {detail_step_parameter}>0, '
            f'xlim={tuple(detail_zoom_xlim)}'
        )
        if fixed_label:
            title += f' | fixed: {fixed_label}'

    if n_cumulative_rows > 0:
        cumulative_label = 'cumulative CST − cumulative input' if cumulative_subtract_input_from_output else 'cumulative matched CST'
        title += (
            f'\nBottom rows: {cumulative_label}, '
            f'xlim={tuple(cumulative_zoom_xlim)}'
        )

    if color_individual_by_age and age_column_resolved is not None:
        title += f'\nIndividual detail traces colored by {age_column_resolved}'

    fig.suptitle(title, fontsize=11)

    return fig, axes, plot_table


# ---------------------------------------------------------------------
# Example call with exposed parameters.
# ---------------------------------------------------------------------

trace_batch_output_path = make_trace_batch_output_path(TRACE_BATCH_OUTPUT_DIR, TRACE_BATCH_RUN_NAME)

fig, axes, response_effect_rate_plot_table = plot_response_effect_rate_grid(
    trace_batch_response_metrics_df,
    trace_batch_response_trace_arrays,

    # Grid organization.
    column_parameter='common_copy_probability',
    row_parameter='pulse_input_amplitude_hz',
    include_zero_row=False,

    # Trace keys.
    output_trace_key='sign_corrected_baseline_corrected_cst_effect_spikes_per_s_mu',
    input_trace_key='input_instruction_rate_hz',

    # Main panels.
    main_xlim_rel_s=None,
    main_ylim=None,
    share_y=True,
    figsize_per_cell=(12.0, 4.0),

    # Detail panels.
    plot_individual_zoom=True,
    detail_step_parameter='pulse_input_amplitude_hz',
    detail_step_values=None,
    detail_zoom_xlim=(-0.05, 0.25),
    detail_fixed_parameters=None,
    detail_default_fix_row_parameter=True,
    detail_share_y=True,
    detail_ylim=None,
    detail_figsize_per_row=4.0,
    detail_max_individual_traces=None,
    detail_plot_mean_trace=True,
    detail_input_trace_mode='first',

    # Cumulative panels.
    plot_cumulative_zoom=True,
    cumulative_output_trace_key='cumulative_output_effect_spikes_per_mn',
    cumulative_input_trace_key='cumulative_input_instruction_spikes_per_mn',
    cumulative_subtract_input_from_output=False,
    cumulative_zoom_xlim=(-0.05, 0.25),
    cumulative_share_y=True,
    cumulative_ylim=None,
    cumulative_figsize_per_row=4.0,
    cumulative_detail_max_individual_traces=None,
    cumulative_plot_mean_trace=True,
    cumulative_input_trace_mode='first',

    # Colors.
    output_mean_color='#0072b2',
    output_individual_color='#0072b2',
    input_mean_color='black',
    input_individual_color='black',
    zero_line_color='0.5',

    # Cumulative colors.
    cumulative_mean_color='#d55e00',
    cumulative_individual_color='#d55e00',
    cumulative_input_color='black',

    # Line styles.
    output_mean_lw=1.2,
    output_mean_alpha=1.0,
    output_individual_lw=1.0,
    output_individual_alpha=0.1,
    input_mean_lw=1.5,
    input_mean_alpha=0.85,
    input_individual_lw=1.6,
    input_individual_alpha=0.65,
    input_ls='--',

    # Cumulative line styles.
    cumulative_mean_lw=1.2,
    cumulative_mean_alpha=1.0,
    cumulative_individual_lw=1.0,
    cumulative_individual_alpha=0.5,
    cumulative_input_lw=1.6,
    cumulative_input_alpha=0.65,
    cumulative_input_ls='--',

    # Optional coloring of individual traces by pool age.
    color_individual_by_age=True,
    age_column='zero_ref_pool_integrated_age_mean',
    age_cmap='viridis',
    age_vmin=None,
    age_vmax=None,
    age_nan_color='0.65',
    show_age_colorbar=True,
    age_colorbar_label='pool integrated age at pulse time',

    # Highlight min/max age traces.
    highlight_minmax_age_traces=True,
    highlight_age_column='zero_ref_pool_integrated_age_mean',
    highlight_reaction_time_column='reaction_time_to_epsilon_s',
    highlight_age_lw=1.5,   # <- highlighted curve linewidth
    highlight_age_alpha=1.0,
    highlight_age_zorder=30,
    highlight_age_use_cmap_color=True,
    highlight_min_age_color='tab:blue',
    highlight_max_age_color='tab:orange',
    highlight_age_text=True,
    highlight_age_text_fontsize=8,
    highlight_age_text_xy=(0.02, 0.98),
    highlight_age_text_bbox=None,

    # Epsilon threshold / marker display on cumulative panels.
    show_reaction_epsilon_in_cumulative=True,
    reaction_epsilon_column='reaction_epsilon_extra_spikes_per_mn',
    reaction_epsilon_line_color='0.2',
    reaction_epsilon_line_lw=1.2,
    reaction_epsilon_line_alpha=0.95,
    reaction_epsilon_line_ls=':',
    highlight_threshold_markers=True,
    highlight_threshold_marker='o',
    highlight_threshold_marker_size=80,
    highlight_threshold_marker_edgecolor='black',
    highlight_threshold_marker_linewidth=0.7,
    highlight_threshold_marker_zorder=45,

    # Labels.
    output_mean_label='mean matched CST effect rate',
    output_individual_label='individual matched CST effect rate',
    input_mean_label='mean input instruction rate',
    input_individual_label='input instruction target',
    cumulative_mean_label='mean cumulative matched CST effect',
    cumulative_individual_label='individual cumulative matched CST effect',
    cumulative_input_label='cumulative input instruction',
    y_label_main='effect rate\n(sp/s/MN)',
    y_label_detail='individual effect-rate traces',
    y_label_cumulative='individual cumulative traces',
    title_prefix='Matched baseline-removed CST effect rate',
)

# Optional save.
if globals().get('TRACE_BATCH_RESPONSE_SAVE_OUTPUTS', False):
    response_effect_rate_plot_table.to_pickle(
        trace_batch_output_path / 'trace_batch_response_effect_rate_plot_table.pkl'
    )
    response_effect_rate_plot_table.to_csv(
        trace_batch_output_path / 'trace_batch_response_effect_rate_plot_table.csv',
        index=False,
    )
    _save_figure_formats(
        fig,
        trace_batch_output_path / 'trace_batch_response_effect_rate_summary',
        TRACE_BATCH_SAVE_FIGURE_FORMATS,
    )

plt.show()